# 04 — Stage D: survival modelling (Sections 3.7–3.9, 4.5–4.6)

In [ ]:
import os
from pathlib import Path

# Folder that holds the downloaded public data; all outputs are written into it.
# Set the environment variable GBM_DATA_DIR, or edit the default below.
BASE = Path(os.environ.get("GBM_DATA_DIR", Path.home() / "Downloads"))
print("Data folder:", BASE)

### STAGE D — PART 1

In [ ]:
"""
STAGE D — PART 1
LOCKED SURVIVAL MODELING COHORT + TRAIN/TEST SPLIT
==================================================

Purpose
-------
Create ONE reproducible internal development/test split for the final
144-patient TCGA-GBM cohort.

IMPORTANT:
- Stage C is already frozen.
- No Stage-C network parameter is changed here.
- No feature selection is performed here.
- No model is trained here.
- No test-set information will later be used for preprocessing,
  feature selection, hyperparameter tuning, or model selection.

Split:
    80% development / 20% locked internal test
    stratified ONLY on survival event status.

The split is saved explicitly to disk and must be reused unchanged
throughout Stage D.

Expected inputs
---------------
1. Final 144-patient inventory/clinical table
2. Final frozen Stage-C eNESH matrix
3. Healthy-reference node summary, used only to attach reference_degree
   metadata to the eNESH feature dictionary.

Outputs
-------
STAGE_D_LOCKED_SPLIT.csv
STAGE_D_DEVELOPMENT_PATIENTS.csv
STAGE_D_LOCKED_TEST_PATIENTS.csv
STAGE_D_SURVIVAL_MODELING_COHORT.csv
STAGE_D_ENESH_FEATURE_METADATA.csv
STAGE_D_PART1_SPLIT_SUMMARY.csv
"""

import json
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
from scipy import stats


# ======================================================================
# CONFIGURATION
# ======================================================================

DOWNLOADS = BASE

STAGE_A_DIR = DOWNLOADS / "TCGA_GBM_STAGE_A_DISCOVERY"
STAGE_C_DIR = DOWNLOADS / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
STAGE_D_DIR = DOWNLOADS / "TCGA_GBM_STAGE_D_SURVIVAL"

STAGE_D_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------
# IMPORTANT:
# This script searches for the final 144-patient inventory because the
# exact filename may differ between previous stages.
# ----------------------------------------------------------------------

PREFERRED_INVENTORY_FILES = [

    STAGE_A_DIR / "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.csv",

    STAGE_A_DIR / "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.tsv",

    DOWNLOADS / "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.csv",

    DOWNLOADS / "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.tsv",
]


ENESH_FILE = (
    STAGE_C_DIR /
    "STAGE_C_FINAL_REWIRING_MATRIX_144xANCHORS_20PCT.csv"
)


NODE_SUMMARY_FILE = (
    STAGE_C_DIR /
    "STAGE_C_HEALTHY_REFERENCE_NODE_SUMMARY.csv"
)


TEST_SIZE = 0.20

RANDOM_SEED = 42


# ======================================================================
# UTILITIES
# ======================================================================

def section(title):

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def checkpoint(condition, message):

    if condition:

        print(f"[CHECKPOINT PASS] {message}")

    else:

        print(f"[CHECKPOINT FAIL] {message}")

        raise AssertionError(message)


def normalize_patient_id(x):

    """
    Normalize TCGA patient identifiers.

    Examples
    --------
    TCGA-06-0125-01A -> TCGA-06-0125
    TCGA-06-0125     -> TCGA-06-0125
    """

    x = str(x).strip()

    if x.startswith("TCGA-"):

        parts = x.split("-")

        if len(parts) >= 3:

            return "-".join(parts[:3])

    return x


def find_inventory():

    for path in PREFERRED_INVENTORY_FILES:

        if path.exists():

            return path

    # fallback recursive search

    candidates = list(
        DOWNLOADS.rglob("*144*INVENTORY*.csv")
    )

    candidates += list(
        DOWNLOADS.rglob("*144*INVENTORY*.tsv")
    )

    candidates = [
        p for p in candidates
        if "STAGE_D" not in str(p)
    ]

    if len(candidates) == 0:

        raise FileNotFoundError(
            "\nCould not locate the final 144-patient inventory.\n"
            "Expected something similar to:\n"
            "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.csv/.tsv\n"
        )

    print("\nCandidate inventory files found:")

    for p in candidates:

        print(" ", p)

    # Prefer filename containing FINAL_DISCOVERY_INVENTORY

    preferred = [
        p for p in candidates
        if "FINAL_DISCOVERY_INVENTORY" in p.name.upper()
    ]

    if len(preferred) == 1:

        return preferred[0]

    if len(candidates) == 1:

        return candidates[0]

    raise RuntimeError(
        "\nMultiple possible inventory files were found.\n"
        "Set the correct file manually in PREFERRED_INVENTORY_FILES."
    )


def load_table(path):

    if path.suffix.lower() == ".tsv":

        return pd.read_csv(
            path,
            sep="\t",
            low_memory=False
        )

    return pd.read_csv(
        path,
        low_memory=False
    )


def find_column(df, candidates):

    """
    Find a column using exact match first,
    then case-insensitive exact match.
    """

    for candidate in candidates:

        if candidate in df.columns:

            return candidate

    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[candidate.lower()]

    return None


# ======================================================================
# STEP 1 — LOCATE FINAL 144-PATIENT INVENTORY
# ======================================================================

section(
    "STEP 1 — LOAD FINAL 144-PATIENT SURVIVAL COHORT"
)

INVENTORY_FILE = find_inventory()

print("Inventory file:")
print(INVENTORY_FILE)

inventory = load_table(INVENTORY_FILE)

print("\nInventory shape:", inventory.shape)

checkpoint(
    len(inventory) == 144,
    f"inventory contains exactly 144 rows ({len(inventory)})"
)

print("\nColumns:")

for c in inventory.columns:

    print(" ", c)


# ======================================================================
# STEP 2 — IDENTIFY PATIENT / SURVIVAL COLUMNS
# ======================================================================

section(
    "STEP 2 — RESOLVE PATIENT AND SURVIVAL COLUMNS"
)


PATIENT_CANDIDATES = [

    "patient_id",
    "Patient ID",
    "PATIENT_ID",
    "case_submitter_id",
    "submitter_id",
    "bcr_patient_barcode",
]


TIME_CANDIDATES = [

    "Overall Survival (Months)",
    "overall_survival_months",
    "OS_MONTHS",
    "os_months",
    "OS.time",
    "OS_time",
    "survival_months",
]


STATUS_CANDIDATES = [

    "Overall Survival Status",
    "overall_survival_status",
    "OS_STATUS",
    "os_status",
    "OS",
    "event",
    "death_event",
]


patient_col = find_column(
    inventory,
    PATIENT_CANDIDATES
)

time_col = find_column(
    inventory,
    TIME_CANDIDATES
)

status_col = find_column(
    inventory,
    STATUS_CANDIDATES
)


if patient_col is None:

    raise RuntimeError(
        "Could not identify the patient-ID column."
    )


if time_col is None:

    raise RuntimeError(
        "\nCould not identify the survival-time column.\n"
        "Available columns were printed above.\n"
        "Set TIME_CANDIDATES to the correct column."
    )


if status_col is None:

    raise RuntimeError(
        "\nCould not identify the survival-status column.\n"
        "Available columns were printed above.\n"
        "Set STATUS_CANDIDATES to the correct column."
    )


print("Patient column:", patient_col)
print("Survival time column:", time_col)
print("Survival status column:", status_col)


# ======================================================================
# STEP 3 — STANDARDIZE PATIENT IDs
# ======================================================================

section(
    "STEP 3 — STANDARDIZE PATIENT IDs"
)


inventory["patient_id_stageD"] = (
    inventory[patient_col]
    .apply(normalize_patient_id)
)


checkpoint(
    inventory["patient_id_stageD"].nunique() == 144,
    "144 unique patient IDs after normalization"
)


# ======================================================================
# STEP 4 — STANDARDIZE SURVIVAL TIME
# ======================================================================

section(
    "STEP 4 — STANDARDIZE SURVIVAL TIME"
)


inventory["OS_months_stageD"] = pd.to_numeric(
    inventory[time_col],
    errors="coerce"
)


print(
    inventory["OS_months_stageD"].describe()
)


checkpoint(
    inventory["OS_months_stageD"].notna().all(),
    "all 144 patients have survival time"
)


checkpoint(
    (inventory["OS_months_stageD"] > 0).all(),
    "all survival times are > 0"
)


# ======================================================================
# STEP 5 — STANDARDIZE EVENT INDICATOR
# ======================================================================

section(
    "STEP 5 — STANDARDIZE SURVIVAL EVENT"
)


print("Raw survival-status values:")

print(
    inventory[status_col]
    .value_counts(dropna=False)
)


def convert_event(x):

    """
    Convert common TCGA/cBioPortal survival status formats:

        0:LIVING
        1:DECEASED
        LIVING
        DECEASED
        ALIVE
        DEAD
        0
        1

    into:
        0 = censored/alive
        1 = death event
    """

    if pd.isna(x):

        return np.nan

    s = str(x).strip().upper()

    death_values = {
        "1",
        "1.0",
        "1:DECEASED",
        "DECEASED",
        "DEAD",
        "DIED",
        "EVENT",
        "TRUE"
    }

    alive_values = {
        "0",
        "0.0",
        "0:LIVING",
        "LIVING",
        "ALIVE",
        "CENSORED",
        "FALSE"
    }

    if s in death_values:

        return 1

    if s in alive_values:

        return 0

    # handle strings such as "1:DEAD"

    if s.startswith("1:"):

        return 1

    if s.startswith("0:"):

        return 0

    return np.nan


inventory["OS_event_stageD"] = (
    inventory[status_col]
    .apply(convert_event)
)


unresolved_status = inventory.loc[
    inventory["OS_event_stageD"].isna(),
    status_col
].unique()


if len(unresolved_status) > 0:

    print(
        "\nUnrecognized survival status values:"
    )

    print(unresolved_status)

    raise RuntimeError(
        "Some survival-event values could not be converted."
    )


inventory["OS_event_stageD"] = (
    inventory["OS_event_stageD"]
    .astype(int)
)


checkpoint(
    set(inventory["OS_event_stageD"].unique())
    <= {0, 1},
    "survival event encoded as 0/1"
)


print("\nEvent distribution:")

print(
    inventory["OS_event_stageD"]
    .value_counts()
    .sort_index()
)


print("\nEvent percentages:")

print(
    inventory["OS_event_stageD"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)


# ======================================================================
# STEP 6 — LOAD FINAL FROZEN eNESH MATRIX
# ======================================================================

section(
    "STEP 6 — LOAD FINAL STAGE-C eNESH MATRIX"
)


checkpoint(
    ENESH_FILE.exists(),
    "final Stage-C rewiring matrix exists"
)


enesh = pd.read_csv(
    ENESH_FILE,
    index_col=0
)


enesh.index = [
    normalize_patient_id(x)
    for x in enesh.index
]


enesh.index.name = "patient_id"


print(
    "eNESH matrix:",
    enesh.shape
)


checkpoint(
    enesh.shape[0] == 144,
    "eNESH matrix contains exactly 144 patients"
)


checkpoint(
    enesh.index.nunique() == 144,
    "144 unique eNESH patient IDs"
)


checkpoint(
    enesh.shape[1] == 3992,
    f"expected 3992 frozen DEG-anchor features ({enesh.shape[1]})"
)


checkpoint(
    not enesh.isna().any().any(),
    "eNESH matrix contains no missing values"
)


checkpoint(
    np.isfinite(enesh.to_numpy()).all(),
    "all eNESH values are finite"
)


# ======================================================================
# STEP 7 — EXACT COHORT ALIGNMENT
# ======================================================================

section(
    "STEP 7 — VERIFY SURVIVAL ↔ eNESH ALIGNMENT"
)


clinical_patients = set(
    inventory["patient_id_stageD"]
)

enesh_patients = set(
    enesh.index
)


missing_enesh = (
    clinical_patients -
    enesh_patients
)

extra_enesh = (
    enesh_patients -
    clinical_patients
)


print(
    "Clinical patients missing from eNESH:",
    len(missing_enesh)
)

print(
    "Extra eNESH patients:",
    len(extra_enesh)
)


if missing_enesh:

    print(
        "Missing:",
        sorted(missing_enesh)
    )


if extra_enesh:

    print(
        "Extra:",
        sorted(extra_enesh)
    )


checkpoint(
    clinical_patients == enesh_patients,
    "exact 144-patient alignment between survival and eNESH"
)


# ======================================================================
# STEP 8 — BUILD MASTER MODELING COHORT
# ======================================================================

section(
    "STEP 8 — BUILD MASTER MODELING COHORT"
)


modeling_cohort = pd.DataFrame({

    "patient_id":
        inventory["patient_id_stageD"],

    "OS_months":
        inventory["OS_months_stageD"],

    "OS_event":
        inventory["OS_event_stageD"]
})


modeling_cohort = (
    modeling_cohort
    .set_index("patient_id")
    .loc[enesh.index]
    .reset_index()
)


checkpoint(
    len(modeling_cohort) == 144,
    "master modeling cohort contains 144 patients"
)


checkpoint(
    modeling_cohort["patient_id"].tolist()
    == list(enesh.index),
    "modeling cohort follows exact Stage-C patient order"
)


# ======================================================================
# STEP 9 — CREATE ONE LOCKED 80/20 SPLIT
# ======================================================================

section(
    "STEP 9 — CREATE LOCKED 80/20 TRAIN/TEST SPLIT"
)


train_ids, test_ids = train_test_split(

    modeling_cohort["patient_id"],

    test_size=TEST_SIZE,

    random_state=RANDOM_SEED,

    stratify=modeling_cohort["OS_event"]
)


train_ids = list(train_ids)

test_ids = list(test_ids)


print(
    "Development patients:",
    len(train_ids)
)

print(
    "Locked test patients:",
    len(test_ids)
)


checkpoint(
    len(train_ids) + len(test_ids) == 144,
    "all patients assigned exactly once"
)


checkpoint(
    len(set(train_ids) & set(test_ids)) == 0,
    "development and test sets are disjoint"
)


checkpoint(
    set(train_ids) | set(test_ids)
    == set(modeling_cohort["patient_id"]),
    "development + test equals full cohort"
)


# ======================================================================
# STEP 10 — SAVE SPLIT ASSIGNMENT
# ======================================================================

section(
    "STEP 10 — SAVE LOCKED SPLIT"
)


split_map = {
    pid: "development"
    for pid in train_ids
}


split_map.update({
    pid: "locked_test"
    for pid in test_ids
})


modeling_cohort["split"] = (
    modeling_cohort["patient_id"]
    .map(split_map)
)


checkpoint(
    modeling_cohort["split"].notna().all(),
    "every patient has a split assignment"
)


print(
    modeling_cohort["split"]
    .value_counts()
)


# ======================================================================
# STEP 11 — AUDIT EVENT BALANCE
# ======================================================================

section(
    "STEP 11 — EVENT BALANCE AUDIT"
)


dev = modeling_cohort[
    modeling_cohort["split"]
    == "development"
].copy()


test = modeling_cohort[
    modeling_cohort["split"]
    == "locked_test"
].copy()


def cohort_summary(df, label):

    return {

        "cohort": label,

        "n": len(df),

        "events":
            int(df["OS_event"].sum()),

        "censored":
            int((df["OS_event"] == 0).sum()),

        "event_pct":
            df["OS_event"].mean() * 100,

        "median_OS_months":
            df["OS_months"].median(),

        "mean_OS_months":
            df["OS_months"].mean(),

        "min_OS_months":
            df["OS_months"].min(),

        "max_OS_months":
            df["OS_months"].max(),
    }


summary_df = pd.DataFrame([

    cohort_summary(
        modeling_cohort,
        "all_144"
    ),

    cohort_summary(
        dev,
        "development"
    ),

    cohort_summary(
        test,
        "locked_test"
    ),
])


print(
    summary_df.to_string(
        index=False
    )
)


# ======================================================================
# STEP 12 — SURVIVAL-TIME BALANCE AUDIT
# ======================================================================

section(
    "STEP 12 — SURVIVAL-TIME BALANCE AUDIT"
)


print(
    "Development OS months:"
)

print(
    dev["OS_months"].describe()
)


print(
    "\nLocked-test OS months:"
)

print(
    test["OS_months"].describe()
)


# Descriptive comparison only.
# This does NOT change the split.

u_stat, u_p = stats.mannwhitneyu(

    dev["OS_months"],

    test["OS_months"],

    alternative="two-sided"
)


print(
    f"\nMann-Whitney U = {u_stat:.2f}"
)

print(
    f"Descriptive p = {u_p:.6g}"
)

print(
    "\nIMPORTANT: this test is an audit only."
)

print(
    "The split will NOT be repeatedly regenerated based on this p-value."
)


# ======================================================================
# STEP 13 — BUILD eNESH FEATURE METADATA
# ======================================================================

section(
    "STEP 13 — BUILD eNESH FEATURE METADATA"
)


feature_metadata = pd.DataFrame({

    "ensembl_id":
        enesh.columns
})


feature_metadata[
    "is_final_enesh_feature"
] = True


# Add reference degree if available.

if NODE_SUMMARY_FILE.exists():

    print(
        "Loading reference node summary:"
    )

    print(
        NODE_SUMMARY_FILE
    )

    node_summary = pd.read_csv(
        NODE_SUMMARY_FILE
    )

    print(
        "Node-summary columns:"
    )

    print(
        list(node_summary.columns)
    )

    gene_col = find_column(
        node_summary,
        [
            "ensembl_id",
            "gene_id"
        ]
    )

    degree_col = find_column(
        node_summary,
        [
            "reference_degree",
            "degree",
            "Reference_Degree"
        ]
    )


    if (
        gene_col is not None
        and
        degree_col is not None
    ):

        degree_map = (
            node_summary
            .drop_duplicates(gene_col)
            .set_index(gene_col)[degree_col]
        )


        feature_metadata[
            "reference_degree"
        ] = (
            feature_metadata[
                "ensembl_id"
            ]
            .map(degree_map)
        )


        print(
            "Reference degree attached."
        )

        print(
            feature_metadata[
                "reference_degree"
            ].describe()
        )

    else:

        print(
            "Could not automatically identify "
            "reference-degree columns."
        )

        feature_metadata[
            "reference_degree"
        ] = np.nan

else:

    print(
        "Healthy-reference node summary not found."
    )

    feature_metadata[
        "reference_degree"
    ] = np.nan


# ----------------------------------------------------------------------
# IMPORTANT:
# Compute variance ONLY on DEVELOPMENT patients.
#
# This isn't feature selection yet.
# It is an audit field for later Stage-D preprocessing.
# ----------------------------------------------------------------------

dev_enesh = enesh.loc[
    train_ids
]


feature_metadata[
    "development_variance"
] = (
    dev_enesh.var(
        axis=0,
        ddof=1
    )
    .reindex(
        feature_metadata[
            "ensembl_id"
        ]
    )
    .values
)


feature_metadata[
    "development_sd"
] = np.sqrt(
    feature_metadata[
        "development_variance"
    ]
)


feature_metadata[
    "development_zero_fraction"
] = (
    (dev_enesh == 0)
    .mean(axis=0)
    .reindex(
        feature_metadata[
            "ensembl_id"
        ]
    )
    .values
)


print(
    "\nFeature metadata:"
)

print(
    feature_metadata.head()
)


print(
    "\nDevelopment SD:"
)

print(
    feature_metadata[
        "development_sd"
    ].describe()
)


# ======================================================================
# STEP 14 — DEGREE / VARIANCE DIAGNOSTIC
# ======================================================================

section(
    "STEP 14 — LOW-DEGREE / VARIANCE DIAGNOSTIC"
)


if (
    feature_metadata[
        "reference_degree"
    ].notna().any()
):

    top_variance = (
        feature_metadata
        .sort_values(
            "development_sd",
            ascending=False
        )
        .head(30)
    )


    print(
        top_variance[
            [
                "ensembl_id",
                "reference_degree",
                "development_sd",
                "development_zero_fraction"
            ]
        ].to_string(
            index=False
        )
    )


    degree1_fraction = (
        top_variance[
            "reference_degree"
        ] == 1
    ).mean()


    print(
        "\nFraction of top-30 variance features "
        f"with reference_degree=1: "
        f"{degree1_fraction*100:.1f}%"
    )


    print(
        "\nThis is DIAGNOSTIC ONLY."
    )

    print(
        "No feature is removed here."
    )

else:

    print(
        "Reference degree unavailable — "
        "diagnostic skipped."
    )


# ======================================================================
# STEP 15 — SAVE MASTER MODELING COHORT
# ======================================================================

section(
    "STEP 15 — SAVE STAGE-D PART 1 OUTPUTS"
)


MASTER_OUT = (
    STAGE_D_DIR /
    "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)


SPLIT_OUT = (
    STAGE_D_DIR /
    "STAGE_D_LOCKED_SPLIT.csv"
)


DEV_OUT = (
    STAGE_D_DIR /
    "STAGE_D_DEVELOPMENT_PATIENTS.csv"
)


TEST_OUT = (
    STAGE_D_DIR /
    "STAGE_D_LOCKED_TEST_PATIENTS.csv"
)


FEATURE_META_OUT = (
    STAGE_D_DIR /
    "STAGE_D_ENESH_FEATURE_METADATA.csv"
)


SUMMARY_OUT = (
    STAGE_D_DIR /
    "STAGE_D_PART1_SPLIT_SUMMARY.csv"
)


modeling_cohort.to_csv(
    MASTER_OUT,
    index=False
)


modeling_cohort[
    [
        "patient_id",
        "split"
    ]
].to_csv(
    SPLIT_OUT,
    index=False
)


dev.to_csv(
    DEV_OUT,
    index=False
)


test.to_csv(
    TEST_OUT,
    index=False
)


feature_metadata.to_csv(
    FEATURE_META_OUT,
    index=False
)


summary_df.to_csv(
    SUMMARY_OUT,
    index=False
)


# ======================================================================
# STEP 16 — SAVE LOCKED DESIGN JSON
# ======================================================================

section(
    "STEP 16 — SAVE LOCKED DESIGN METADATA"
)


design = {

    "stage":
        "Stage D Part 1",

    "total_patients":
        144,

    "development_n":
        len(dev),

    "locked_test_n":
        len(test),

    "test_fraction":
        TEST_SIZE,

    "random_seed":
        RANDOM_SEED,

    "stratification":
        "overall_survival_event",

    "network_rule":
        "Stage C frozen top-20pct reference / independent patient top-20pct",

    "n_enesh_features":
        int(enesh.shape[1]),

    "survival_used_for_stage_c_network_selection":
        False,

    "test_set_used_for_feature_selection":
        False,

    "test_set_used_for_preprocessing":
        False,

    "test_set_used_for_model_tuning":
        False,

    "split_locked":
        True,

    "split_must_not_be_regenerated":
        True,

    "notes": (
        "All future preprocessing, feature filtering, "
        "feature selection and hyperparameter tuning "
        "must be learned using development data only. "
        "The locked internal test set is evaluated only "
        "after the final model specification is frozen."
    )
}


DESIGN_OUT = (
    STAGE_D_DIR /
    "STAGE_D_LOCKED_DESIGN.json"
)


with open(
    DESIGN_OUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        design,
        f,
        indent=2
    )


# ======================================================================
# STEP 17 — FINAL CHECKPOINTS
# ======================================================================

section(
    "STEP 17 — FINAL CHECKPOINTS"
)


checkpoint(
    MASTER_OUT.exists(),
    "master modeling cohort saved"
)


checkpoint(
    SPLIT_OUT.exists(),
    "locked split saved"
)


checkpoint(
    DEV_OUT.exists(),
    "development cohort saved"
)


checkpoint(
    TEST_OUT.exists(),
    "locked test cohort saved"
)


checkpoint(
    FEATURE_META_OUT.exists(),
    "eNESH feature metadata saved"
)


checkpoint(
    DESIGN_OUT.exists(),
    "locked design metadata saved"
)


# Reload split to prove persistence

split_reload = pd.read_csv(
    SPLIT_OUT
)


checkpoint(
    len(split_reload) == 144,
    "saved split contains 144 patients"
)


checkpoint(
    split_reload[
        "patient_id"
    ].nunique() == 144,
    "saved split contains 144 unique patients"
)


checkpoint(
    set(
        split_reload["split"]
    )
    == {
        "development",
        "locked_test"
    },
    "saved split contains only valid assignments"
)


# ======================================================================
# COMPLETE
# ======================================================================

section(
    "STAGE D PART 1 COMPLETE"
)


print(
    "\nThe 144-patient cohort is now LOCKED."
)


print(
    f"\nDevelopment cohort: {len(dev)} patients"
)


print(
    f"Locked internal test: {len(test)} patients"
)


print(
    "\nNO feature selection has been performed."
)


print(
    "NO survival model has been fitted."
)


print(
    "NO test-set information has been used for model development."
)


print(
    "\nFILES CREATED:"
)


for p in [

    MASTER_OUT,
    SPLIT_OUT,
    DEV_OUT,
    TEST_OUT,
    FEATURE_META_OUT,
    SUMMARY_OUT,
    DESIGN_OUT

]:

    print(p)


print(
    "\nNEXT:"
)


print(
    "Stage D Part 2 — leakage-safe feature preprocessing "
    "inside the DEVELOPMENT cohort only."
)


print(
    "\nIMPORTANT:"
)


print(
    "From this point onward, DO NOT regenerate the train/test split."
)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DOWNLOADS = BASE

STAGE_C_DIR = (
    DOWNLOADS /
    "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
)

STAGE_D_DIR = (
    DOWNLOADS /
    "TCGA_GBM_STAGE_D_SURVIVAL"
)

ENESH_FILE = (
    STAGE_C_DIR /
    "STAGE_C_FINAL_REWIRING_MATRIX_144xANCHORS_20PCT.csv"
)

NODE_SUMMARY_FILE = (
    STAGE_C_DIR /
    "STAGE_C_HEALTHY_REFERENCE_NODE_SUMMARY.csv"
)

SPLIT_FILE = (
    STAGE_D_DIR /
    "STAGE_D_LOCKED_SPLIT.csv"
)

FEATURE_META_OUT = (
    STAGE_D_DIR /
    "STAGE_D_ENESH_FEATURE_METADATA.csv"
)


print("=" * 80)
print("FIX — ATTACH REFERENCE DEGREE TO STAGE-D FEATURES")
print("=" * 80)


# ============================================================
# 1. LOAD EXISTING FROZEN DATA
# ============================================================

enesh = pd.read_csv(
    ENESH_FILE,
    index_col=0
)

split_df = pd.read_csv(
    SPLIT_FILE
)

nodes = pd.read_csv(
    NODE_SUMMARY_FILE
)

print("\neNESH:", enesh.shape)
print("Split:", split_df.shape)
print("Node summary:", nodes.shape)

print("\nNode columns:")
print(nodes.columns.tolist())


# ============================================================
# 2. CHECK EXPECTED COLUMNS
# ============================================================

assert "gene" in nodes.columns, \
    "Column 'gene' not found in reference node summary"

assert "reference_degree" in nodes.columns, \
    "Column 'reference_degree' not found"

assert "patient_id" in split_df.columns
assert "split" in split_df.columns


# ============================================================
# 3. BUILD DEGREE MAP
# ============================================================

nodes["gene"] = (
    nodes["gene"]
    .astype(str)
    .str.split(".")
    .str[0]
    .str.strip()
)

degree_map = (
    nodes
    .drop_duplicates("gene")
    .set_index("gene")["reference_degree"]
)


# ============================================================
# 4. GET DEVELOPMENT PATIENTS ONLY
# ============================================================

development_ids = (
    split_df.loc[
        split_df["split"] == "development",
        "patient_id"
    ]
    .astype(str)
    .tolist()
)

assert len(development_ids) == 115, \
    f"Expected 115 development patients, found {len(development_ids)}"

assert set(development_ids).issubset(set(enesh.index)), \
    "Some development patients missing from eNESH matrix"

dev_enesh = enesh.loc[development_ids]

print(
    f"\nDevelopment matrix: "
    f"{dev_enesh.shape[0]} patients x "
    f"{dev_enesh.shape[1]} features"
)


# ============================================================
# 5. REBUILD FEATURE METADATA
# ============================================================

feature_metadata = pd.DataFrame({
    "ensembl_id": enesh.columns.astype(str)
})

feature_metadata["ensembl_id"] = (
    feature_metadata["ensembl_id"]
    .str.split(".")
    .str[0]
    .str.strip()
)

feature_metadata["is_final_enesh_feature"] = True

feature_metadata["reference_degree"] = (
    feature_metadata["ensembl_id"]
    .map(degree_map)
)


# ============================================================
# 6. DEVELOPMENT-ONLY VARIANCE
# ============================================================

dev_variance = dev_enesh.var(
    axis=0,
    ddof=1
)

dev_sd = dev_enesh.std(
    axis=0,
    ddof=1
)

dev_zero_fraction = (
    dev_enesh == 0
).mean(axis=0)


feature_metadata["development_variance"] = (
    feature_metadata["ensembl_id"]
    .map(dev_variance)
)

feature_metadata["development_sd"] = (
    feature_metadata["ensembl_id"]
    .map(dev_sd)
)

feature_metadata["development_zero_fraction"] = (
    feature_metadata["ensembl_id"]
    .map(dev_zero_fraction)
)


# ============================================================
# 7. CHECK DEGREE RESOLUTION
# ============================================================

resolved = (
    feature_metadata["reference_degree"]
    .notna()
    .sum()
)

total = len(feature_metadata)

resolution = 100 * resolved / total

print("\nReference-degree resolution:")
print(f"{resolved}/{total} ({resolution:.2f}%)")

assert resolution >= 99, \
    "Unexpectedly low reference-degree resolution"


print("\nReference degree distribution:")

print(
    feature_metadata[
        "reference_degree"
    ].describe()
)


# ============================================================
# 8. CHECK LOW-DEGREE / HIGH-VARIANCE EFFECT
# ============================================================

print("\n" + "=" * 80)
print("TOP 30 FEATURES BY DEVELOPMENT SD")
print("=" * 80)

top30 = (
    feature_metadata
    .sort_values(
        "development_sd",
        ascending=False
    )
    .head(30)
)

print(
    top30[
        [
            "ensembl_id",
            "reference_degree",
            "development_sd",
            "development_zero_fraction"
        ]
    ].to_string(index=False)
)


degree1_fraction = (
    top30["reference_degree"] == 1
).mean()

degree_le2_fraction = (
    top30["reference_degree"] <= 2
).mean()


print(
    "\nTop-30 with reference_degree = 1: "
    f"{degree1_fraction * 100:.1f}%"
)

print(
    "Top-30 with reference_degree <= 2: "
    f"{degree_le2_fraction * 100:.1f}%"
)


# ============================================================
# 9. ZERO-VARIANCE FEATURES
# ============================================================

zero_variance = (
    feature_metadata[
        "development_variance"
    ] <= 1e-12
)

print(
    "\nZero-variance eNESH features "
    f"in DEVELOPMENT cohort: "
    f"{zero_variance.sum()}/{total}"
)


# ============================================================
# 10. SAVE CORRECTED METADATA
# ============================================================

feature_metadata.to_csv(
    FEATURE_META_OUT,
    index=False
)

print("\nCorrected metadata saved:")
print(FEATURE_META_OUT)


print("\n" + "=" * 80)
print("FIX COMPLETE")
print("=" * 80)

print(
    "\nIMPORTANT:"
)

print(
    "The locked 115/29 split was NOT changed."
)

print(
    "No test-set information was used."
)

print(
    "No feature selection was performed."
)

print(
    "\nNext: Stage D Part 2 — "
    "development-only leakage-safe preprocessing."
)

### STAGE D — PART 2

In [ ]:
"""
STAGE D — PART 2
DEVELOPMENT-ONLY LEAKAGE-SAFE FEATURE PREPARATION
==================================================

PURPOSE
-------
Prepare the feature spaces for survival modeling without touching the
locked test set for any learned preprocessing decision.

Models to prepare
-----------------
M1 = Clinical baseline

M2 = Expression baseline
     Uses calibrated expression for the same DEG-anchor gene universe.

M3 = Personalized rewiring-only
     Uses final eNESH features.

M4 = Clinical + personalized rewiring.

IMPORTANT
---------
- Locked 115/29 split is reused exactly.
- Test split is NEVER regenerated.
- Zero-variance filtering is learned from DEVELOPMENT only.
- NO survival-association feature selection is done here.
- NO model is trained here.
- NO C-index is calculated here.
- NO test-set information determines feature inclusion.

Only mathematically unusable eNESH features are removed here:
    development variance <= 1e-10

Degree is retained as metadata only.
We do NOT automatically remove degree-1 or degree-2 anchors.
"""

# ======================================================================
# IMPORTS
# ======================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd


# ======================================================================
# PATHS
# ======================================================================

DOWNLOADS = BASE

STAGE_D_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)

STAGE_C_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
)

CALIBRATED_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_C_144_CALIBRATED"
)


# ----------------------------------------------------------------------
# Locked Stage-D files
# ----------------------------------------------------------------------

SPLIT_FILE = (
    STAGE_D_DIR
    / "STAGE_D_LOCKED_SPLIT.csv"
)

COHORT_FILE = (
    STAGE_D_DIR
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)

FEATURE_METADATA_FILE = (
    STAGE_D_DIR
    / "STAGE_D_ENESH_FEATURE_METADATA.csv"
)


# ----------------------------------------------------------------------
# Stage-C final rewiring matrix
# ----------------------------------------------------------------------

ENESH_FILE = (
    STAGE_C_DIR
    / "STAGE_C_FINAL_REWIRING_MATRIX_144xANCHORS_20PCT.csv"
)


# ----------------------------------------------------------------------
# Calibrated expression baseline
#
# This was already created in Stage C0 Part 3.
# ----------------------------------------------------------------------

EXPRESSION_FILE = (
    CALIBRATED_DIR
    / "STAGE_C_144_CALIBRATED_DEG_ANCHORS.csv"
)


# ======================================================================
# OUTPUTS
# ======================================================================

USABLE_ENESH_FEATURES_OUT = (
    STAGE_D_DIR
    / "STAGE_D_USABLE_ENESH_FEATURES.csv"
)

M2_EXPRESSION_MATRIX_OUT = (
    STAGE_D_DIR
    / "STAGE_D_M2_EXPRESSION_RAW.csv"
)

M3_REWIRING_MATRIX_OUT = (
    STAGE_D_DIR
    / "STAGE_D_M3_REWIRING_RAW.csv"
)

CLINICAL_RAW_OUT = (
    STAGE_D_DIR
    / "STAGE_D_M1_CLINICAL_RAW.csv"
)

FINAL_FEATURE_AUDIT_OUT = (
    STAGE_D_DIR
    / "STAGE_D_PART2_FEATURE_AUDIT.csv"
)

PART2_DESIGN_OUT = (
    STAGE_D_DIR
    / "STAGE_D_PART2_PREPROCESSING_RULES.json"
)


# ======================================================================
# SETTINGS
# ======================================================================

ZERO_VARIANCE_THRESHOLD = 1e-10

EXPECTED_TOTAL_PATIENTS = 144
EXPECTED_DEVELOPMENT = 115
EXPECTED_TEST = 29
EXPECTED_ORIGINAL_ENESH = 3992


# ======================================================================
# HELPERS
# ======================================================================

def section(title):

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def checkpoint(condition, message):

    status = (
        "PASS"
        if condition
        else "FAIL"
    )

    print(
        f"[CHECKPOINT {status}] {message}"
    )

    if not condition:

        raise RuntimeError(
            f"Checkpoint failed: {message}"
        )


def normalize_patient_id(x):

    x = str(x).strip()

    if x.startswith("TCGA-"):

        parts = x.split("-")

        if len(parts) >= 3:

            return "-".join(
                parts[:3]
            )

    return x


def strip_version(x):

    return (
        str(x)
        .split(".")[0]
        .strip()
    )


# ======================================================================
# STEP 1 — LOAD LOCKED SPLIT
# ======================================================================

section(
    "STEP 1 — LOAD LOCKED 115/29 SPLIT"
)


checkpoint(
    SPLIT_FILE.exists(),
    "locked split file exists"
)


split_df = pd.read_csv(
    SPLIT_FILE
)


checkpoint(
    len(split_df)
    ==
    EXPECTED_TOTAL_PATIENTS,
    "locked split contains 144 patients"
)


split_df["patient_id"] = (
    split_df["patient_id"]
    .apply(
        normalize_patient_id
    )
)


development_ids = (
    split_df.loc[
        split_df["split"]
        ==
        "development",
        "patient_id"
    ]
    .tolist()
)


test_ids = (
    split_df.loc[
        split_df["split"]
        ==
        "locked_test",
        "patient_id"
    ]
    .tolist()
)


print(
    "Development:",
    len(development_ids)
)

print(
    "Locked test:",
    len(test_ids)
)


checkpoint(
    len(development_ids)
    ==
    EXPECTED_DEVELOPMENT,
    "development cohort remains 115 patients"
)


checkpoint(
    len(test_ids)
    ==
    EXPECTED_TEST,
    "locked test remains 29 patients"
)


checkpoint(
    len(
        set(development_ids)
        &
        set(test_ids)
    )
    ==
    0,
    "development/test remain disjoint"
)


# ======================================================================
# STEP 2 — LOAD SURVIVAL COHORT
# ======================================================================

section(
    "STEP 2 — LOAD LOCKED SURVIVAL COHORT"
)


checkpoint(
    COHORT_FILE.exists(),
    "survival modeling cohort exists"
)


cohort = pd.read_csv(
    COHORT_FILE
)


cohort["patient_id"] = (
    cohort["patient_id"]
    .apply(
        normalize_patient_id
    )
)


checkpoint(
    cohort["patient_id"].nunique()
    ==
    EXPECTED_TOTAL_PATIENTS,
    "144 unique survival patients"
)


checkpoint(
    cohort["OS_months"].notna().all(),
    "all survival times present"
)


checkpoint(
    cohort["OS_event"].notna().all(),
    "all event indicators present"
)


# ======================================================================
# STEP 3 — LOAD FINAL eNESH MATRIX
# ======================================================================

section(
    "STEP 3 — LOAD FINAL PERSONALIZED REWIRING MATRIX"
)


checkpoint(
    ENESH_FILE.exists(),
    "final frozen eNESH matrix exists"
)


enesh = pd.read_csv(
    ENESH_FILE,
    index_col=0
)


enesh.index = [
    normalize_patient_id(x)
    for x in enesh.index
]


enesh.columns = [
    strip_version(x)
    for x in enesh.columns
]


print(
    "eNESH:",
    enesh.shape
)


checkpoint(
    enesh.shape
    ==
    (
        144,
        EXPECTED_ORIGINAL_ENESH
    ),
    "eNESH remains 144 × 3992"
)


checkpoint(
    set(enesh.index)
    ==
    set(split_df["patient_id"]),
    "eNESH patient IDs match locked split"
)


# ======================================================================
# STEP 4 — LOAD CORRECTED FEATURE METADATA
# ======================================================================

section(
    "STEP 4 — LOAD DEVELOPMENT-ONLY FEATURE METADATA"
)


checkpoint(
    FEATURE_METADATA_FILE.exists(),
    "corrected eNESH feature metadata exists"
)


metadata = pd.read_csv(
    FEATURE_METADATA_FILE
)


metadata["ensembl_id"] = (
    metadata["ensembl_id"]
    .apply(
        strip_version
    )
)


print(
    "Feature metadata:",
    metadata.shape
)


checkpoint(
    len(metadata)
    ==
    EXPECTED_ORIGINAL_ENESH,
    "metadata contains all 3992 eNESH features"
)


checkpoint(
    metadata[
        "reference_degree"
    ]
    .notna()
    .all(),
    "reference degree resolved for all features"
)


# ======================================================================
# STEP 5 — DEVELOPMENT-ONLY ZERO-VARIANCE FILTER
# ======================================================================

section(
    "STEP 5 — REMOVE ONLY MATHEMATICALLY UNUSABLE eNESH FEATURES"
)


# ------------------------------------------------------------
# IMPORTANT:
#
# The filter is based ONLY on development variance.
# ------------------------------------------------------------

usable_mask = (
    metadata[
        "development_variance"
    ]
    >
    ZERO_VARIANCE_THRESHOLD
)


usable_metadata = (
    metadata[
        usable_mask
    ]
    .copy()
)


removed_metadata = (
    metadata[
        ~usable_mask
    ]
    .copy()
)


usable_enesh_genes = (
    usable_metadata[
        "ensembl_id"
    ]
    .tolist()
)


print(
    "Original eNESH features:",
    len(metadata)
)


print(
    "Zero-variance removed:",
    len(removed_metadata)
)


print(
    "Usable eNESH features:",
    len(usable_metadata)
)


checkpoint(
    len(removed_metadata)
    ==
    323,
    (
        "development-only zero-variance removal "
        "reproduces 323 features"
    )
)


checkpoint(
    len(usable_metadata)
    ==
    3669,
    "3669 usable eNESH features remain"
)


# ======================================================================
# STEP 6 — IMPORTANT DEGREE DIAGNOSTIC
# ======================================================================

section(
    "STEP 6 — DEGREE DISTRIBUTION AFTER OBJECTIVE FILTER"
)


print(
    usable_metadata[
        "reference_degree"
    ].describe()
)


degree1 = int(
    (
        usable_metadata[
            "reference_degree"
        ]
        ==
        1
    ).sum()
)


degree2_or_less = int(
    (
        usable_metadata[
            "reference_degree"
        ]
        <=
        2
    ).sum()
)


print(
    f"\nDegree = 1: "
    f"{degree1}/{len(usable_metadata)} "
    f"({100*degree1/len(usable_metadata):.1f}%)"
)


print(
    f"Degree <= 2: "
    f"{degree2_or_less}/{len(usable_metadata)} "
    f"({100*degree2_or_less/len(usable_metadata):.1f}%)"
)


print(
    "\nNo degree-based filtering is applied."
)


# ======================================================================
# STEP 7 — BUILD FINAL M3 REWIRING MATRIX
# ======================================================================

section(
    "STEP 7 — BUILD M3 REWIRING FEATURE MATRIX"
)


M3_rewiring = (
    enesh[
        usable_enesh_genes
    ]
    .copy()
)


print(
    "M3 rewiring matrix:",
    M3_rewiring.shape
)


checkpoint(
    M3_rewiring.shape
    ==
    (
        144,
        3669
    ),
    "M3 raw matrix = 144 × 3669"
)


checkpoint(
    np.isfinite(
        M3_rewiring.values
    ).all(),
    "M3 matrix contains finite values"
)


# ======================================================================
# STEP 8 — VERIFY TEST SET WAS NOT USED TO FILTER
# ======================================================================

section(
    "STEP 8 — PROVE FILTER WAS DEVELOPMENT-ONLY"
)


dev_rewiring = (
    M3_rewiring.loc[
        development_ids
    ]
)


test_rewiring = (
    M3_rewiring.loc[
        test_ids
    ]
)


checkpoint(
    dev_rewiring.shape[0]
    ==
    115,
    "M3 development matrix contains 115 patients"
)


checkpoint(
    test_rewiring.shape[0]
    ==
    29,
    "M3 locked-test matrix contains 29 patients"
)


# This is an audit only.
# We DO NOT remove a feature because of anything observed here.

test_only_variability = (
    test_rewiring.var(
        axis=0,
        ddof=1
    )
    >
    ZERO_VARIANCE_THRESHOLD
)


print(
    "Features retained based on development:",
    M3_rewiring.shape[1]
)


print(
    "Of those, variable in locked test:",
    int(
        test_only_variability.sum()
    )
)


print(
    "\nThis number is diagnostic only."
)

print(
    "It does NOT alter feature inclusion."
)


# ======================================================================
# STEP 9 — LOAD CALIBRATED EXPRESSION FOR M2
# ======================================================================

section(
    "STEP 9 — LOAD CALIBRATED EXPRESSION BASELINE"
)


checkpoint(
    EXPRESSION_FILE.exists(),
    "calibrated DEG-anchor expression matrix exists"
)


expression = pd.read_csv(
    EXPRESSION_FILE,
    index_col=0
)


expression.index = [
    strip_version(x)
    for x in expression.index
]


# Expected Stage-C expression format:
# genes × patients
#
# Detect orientation robustly.

patient_set = set(
    split_df[
        "patient_id"
    ]
)


column_patient_ids = set(
    normalize_patient_id(x)
    for x in expression.columns
)


index_patient_ids = set(
    normalize_patient_id(x)
    for x in expression.index
)


if len(
    column_patient_ids
    &
    patient_set
) == 144:

    print(
        "Expression orientation detected: "
        "genes × patients"
    )

    expression.columns = [
        normalize_patient_id(x)
        for x in expression.columns
    ]

    expression_patient_rows = (
        expression.T
    )


elif len(
    index_patient_ids
    &
    patient_set
) == 144:

    print(
        "Expression orientation detected: "
        "patients × genes"
    )

    expression.index = [
        normalize_patient_id(x)
        for x in expression.index
    ]

    expression_patient_rows = (
        expression.copy()
    )


else:

    raise RuntimeError(
        "Could not determine expression-matrix orientation."
    )


expression_patient_rows.columns = [
    strip_version(x)
    for x in expression_patient_rows.columns
]


checkpoint(
    set(
        expression_patient_rows.index
    )
    ==
    patient_set,
    "expression baseline contains exactly the locked 144 patients"
)


print(
    "Expression patient matrix:",
    expression_patient_rows.shape
)


# ======================================================================
# STEP 10 — MATCH M2 EXPRESSION TO SAME USABLE GENE UNIVERSE
# ======================================================================

section(
    "STEP 10 — CREATE MATCHED M2 EXPRESSION FEATURE SPACE"
)


expression_available = set(
    expression_patient_rows.columns
)


matched_expression_genes = [
    gene
    for gene
    in usable_enesh_genes
    if gene
    in expression_available
]


print(
    "Usable eNESH genes:",
    len(
        usable_enesh_genes
    )
)


print(
    "Same genes available in calibrated expression:",
    len(
        matched_expression_genes
    )
)


expression_resolution = (
    100
    *
    len(
        matched_expression_genes
    )
    /
    len(
        usable_enesh_genes
    )
)


print(
    f"Resolution: "
    f"{expression_resolution:.2f}%"
)


checkpoint(
    expression_resolution
    >=
    95,
    "at least 95% of usable rewiring genes have expression"
)


M2_expression = (
    expression_patient_rows[
        matched_expression_genes
    ]
    .copy()
)


checkpoint(
    M2_expression.shape[0]
    ==
    144,
    "M2 expression contains 144 patients"
)


checkpoint(
    np.isfinite(
        M2_expression.values
    ).all(),
    "M2 expression contains finite values"
)


print(
    "M2 raw expression matrix:",
    M2_expression.shape
)


# ======================================================================
# STEP 11 — CLINICAL BASELINE VARIABLES
# ======================================================================

section(
    "STEP 11 — BUILD RAW M1 CLINICAL TABLE"
)


# ----------------------------------------------------------------------
# We deliberately do NOT encode/impute/scale yet.
#
# This is the raw clinical table.
#
# Encoding/imputation will be fitted INSIDE CV using development only.
# ----------------------------------------------------------------------

INVENTORY_CANDIDATES = [

    DOWNLOADS
    / "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.tsv",

    DOWNLOADS
    / "TCGA_GBM_144_FINAL_DISCOVERY_INVENTORY.csv",
]


inventory_file = next(
    (
        p
        for p in INVENTORY_CANDIDATES
        if p.exists()
    ),
    None
)


if inventory_file is None:

    raise FileNotFoundError(
        "Could not find the final 144-patient inventory."
    )


if (
    inventory_file.suffix.lower()
    ==
    ".tsv"
):

    inventory = pd.read_csv(
        inventory_file,
        sep="\t",
        low_memory=False
    )

else:

    inventory = pd.read_csv(
        inventory_file,
        low_memory=False
    )


patient_col = (
    "Patient ID"
)


checkpoint(
    patient_col
    in inventory.columns,
    "inventory contains Patient ID"
)


inventory[
    "patient_id"
] = (
    inventory[
        patient_col
    ]
    .apply(
        normalize_patient_id
    )
)


# ----------------------------------------------------------------------
# Pre-specified clinically interpretable baseline.
#
# Missing columns are NOT silently invented.
# ----------------------------------------------------------------------

candidate_clinical_features = [

    "Diagnosis Age",
    "Sex",
    "MGMT Status",
    "Karnofsky Performance Score",
]


available_clinical_features = [
    feature
    for feature
    in candidate_clinical_features
    if feature
    in inventory.columns
]


print(
    "Requested clinical variables:"
)

print(
    candidate_clinical_features
)


print(
    "\nAvailable:"
)

print(
    available_clinical_features
)


checkpoint(
    "Diagnosis Age"
    in available_clinical_features,
    "age available for clinical baseline"
)


clinical_raw = (
    inventory[
        [
            "patient_id"
        ]
        +
        available_clinical_features
    ]
    .drop_duplicates(
        "patient_id"
    )
    .set_index(
        "patient_id"
    )
)


clinical_raw = (
    clinical_raw.loc[
        split_df[
            "patient_id"
        ]
    ]
)


checkpoint(
    clinical_raw.shape[0]
    ==
    144,
    "clinical raw matrix contains 144 patients"
)


print(
    "\nClinical missingness:"
)


print(
    clinical_raw
    .isna()
    .sum()
)


print(
    "\nIMPORTANT:"
)

print(
    "Missing clinical values are NOT imputed here."
)

print(
    "Imputation must be learned inside training folds."
)


# ======================================================================
# STEP 12 — SAVE RAW FEATURE MATRICES
# ======================================================================

section(
    "STEP 12 — SAVE RAW M1/M2/M3 FEATURE SPACES"
)


clinical_raw.to_csv(
    CLINICAL_RAW_OUT
)


M2_expression.to_csv(
    M2_EXPRESSION_MATRIX_OUT
)


M3_rewiring.to_csv(
    M3_REWIRING_MATRIX_OUT
)


usable_metadata.to_csv(
    USABLE_ENESH_FEATURES_OUT,
    index=False
)


print(
    "M1 clinical:"
)

print(
    CLINICAL_RAW_OUT
)


print(
    "\nM2 expression:"
)

print(
    M2_EXPRESSION_MATRIX_OUT
)


print(
    "\nM3 rewiring:"
)

print(
    M3_REWIRING_MATRIX_OUT
)


# ======================================================================
# STEP 13 — FEATURE AUDIT
# ======================================================================

section(
    "STEP 13 — FINAL FEATURE-SPACE AUDIT"
)


audit = pd.DataFrame(
    [
        {
            "model":
                "M1_clinical",

            "n_patients":
                clinical_raw.shape[0],

            "n_raw_features":
                clinical_raw.shape[1],

            "feature_definition":
                "Pre-specified clinical baseline"
        },

        {
            "model":
                "M2_expression",

            "n_patients":
                M2_expression.shape[0],

            "n_raw_features":
                M2_expression.shape[1],

            "feature_definition":
                (
                    "Calibrated expression on genes "
                    "surviving development-only eNESH "
                    "zero-variance filter"
                )
        },

        {
            "model":
                "M3_rewiring",

            "n_patients":
                M3_rewiring.shape[0],

            "n_raw_features":
                M3_rewiring.shape[1],

            "feature_definition":
                (
                    "Personalized eNESH; "
                    "development-only zero-variance filter"
                )
        },

        {
            "model":
                "M4_clinical_plus_rewiring",

            "n_patients":
                144,

            "n_raw_features":
                clinical_raw.shape[1]
                +
                M3_rewiring.shape[1],

            "feature_definition":
                (
                    "Clinical + personalized eNESH"
                )
        }
    ]
)


print(
    audit.to_string(
        index=False
    )
)


audit.to_csv(
    FINAL_FEATURE_AUDIT_OUT,
    index=False
)


# ======================================================================
# STEP 14 — SAVE PREPROCESSING RULES
# ======================================================================

section(
    "STEP 14 — SAVE LOCKED PREPROCESSING RULES"
)


rules = {

    "stage":
        "Stage D Part 2",

    "locked_split_reused":
        True,

    "development_n":
        115,

    "locked_test_n":
        29,

    "original_enesh_features":
        3992,

    "zero_variance_threshold":
        ZERO_VARIANCE_THRESHOLD,

    "zero_variance_features_removed":
        int(
            len(
                removed_metadata
            )
        ),

    "usable_enesh_features":
        int(
            len(
                usable_metadata
            )
        ),

    "degree_based_filtering":
        False,

    "reference_degree_role":
        (
            "audit/interpretation metadata only"
        ),

    "survival_based_feature_selection_performed":
        False,

    "test_set_used_for_feature_filtering":
        False,

    "test_set_used_for_imputation":
        False,

    "test_set_used_for_scaling":
        False,

    "test_set_used_for_model_selection":
        False,

    "clinical_preprocessing":
        (
            "deferred to inner development folds"
        ),

    "expression_feature_selection":
        (
            "deferred to inner development folds"
        ),

    "rewiring_feature_selection":
        (
            "deferred to inner development folds"
        )
}


with open(
    PART2_DESIGN_OUT,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        rules,
        handle,
        indent=4
    )


print(
    json.dumps(
        rules,
        indent=4
    )
)


# ======================================================================
# COMPLETE
# ======================================================================

section(
    "STAGE D PART 2 COMPLETE"
)


print(
    "Locked split preserved:"
)

print(
    "115 development / 29 locked test"
)


print(
    "\nObjective eNESH filtering:"
)

print(
    f"3992 -> {len(usable_metadata)} features"
)


print(
    "\nNo degree cutoff was applied."
)


print(
    "No survival-driven feature selection was performed."
)


print(
    "No test-set information influenced preprocessing."
)


print(
    "\nNEXT:"
)


print(
    "Stage D Part 3 — nested development-only survival modeling:"
)


print(
    "M1 = clinical"
)

print(
    "M2 = expression"
)

print(
    "M3 = personalized rewiring"
)

print(
    "M4 = clinical + personalized rewiring"
)


print(
    "\nFeature selection and hyperparameter tuning will occur "
    "inside training folds only."
)

### STAGE D — PART 3

In [ ]:
"""
STAGE D — PART 3
V4-OFF-CORRECTED CONTROLLED EXPERIMENT
======================================

PURPOSE
-------
This script preserves the original V4 modeling design as closely as possible:

- same 115 development / 29 internal-test split
- same RANDOM_SEED
- same repeated 3 x 5 outer CV
- same 3-fold inner CV
- same TOP_K_VALUES = [10, 25, 50]
- same RSF / CoxNet / GBSA learner grid
- same univariate train-fold molecular ranking
- same M1-M6 definitions
- same final RSF 5-seed ensemble

ONLY IMPORTANT SCIENTIFIC CORRECTION:
-------------------------------------
The original V4 stacking trained the meta-model using base-model risk
predictions generated on the SAME patients used to fit the base model.

This version replaces those in-sample training risks with internal
out-of-fold (OOF) risks.

For strict leakage control, molecular feature selection is ALSO repeated
inside each OOF meta-training fold.

The 29-patient internal test must NOT be used to choose between versions.
Primary comparison = repeated nested development outer-CV.
"""

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sksurv.ensemble import (
    RandomSurvivalForest,
    GradientBoostingSurvivalAnalysis,
)
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored


warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION
# =============================================================================

DOWNLOADS = BASE
STAGE_D_DIR = DOWNLOADS / "TCGA_GBM_STAGE_D_SURVIVAL"

SPLIT_FILE = STAGE_D_DIR / "STAGE_D_LOCKED_SPLIT.csv"
COHORT_FILE = STAGE_D_DIR / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
M1_FILE = STAGE_D_DIR / "STAGE_D_M1_CLINICAL_RAW.csv"
M2_FILE = STAGE_D_DIR / "STAGE_D_M2_EXPRESSION_RAW.csv"
M3_FILE = STAGE_D_DIR / "STAGE_D_M3_REWIRING_RAW.csv"

# IMPORTANT:
# Do NOT overwrite original V4 / V3 results.
RESULT_DIR = STAGE_D_DIR / "STAGE_D_PART3_V4_OOF_CORRECTED"
RESULT_DIR.mkdir(parents=True, exist_ok=True)


RANDOM_SEED = 42

OUTER_FOLDS = 5
INNER_FOLDS = 3
N_OUTER_REPEATS = 3

# internal folds used ONLY to generate OOF stacking risks
META_OOF_FOLDS = 3

N_FINAL_SEEDS = 5

# Preserve original V4 Top-K grid
TOP_K_VALUES = [10, 25, 50]


MODELS = ["M1", "M2", "M3", "M4", "M5", "M6"]

MODEL_DESCRIPTIONS = {
    "M1": "Clinical",
    "M2": "Expression",
    "M3": "Rewiring",
    "M4": "Clinical + Expression (OOF stacked)",
    "M5": "Clinical + Rewiring (OOF stacked)",
    "M6": "Clinical + Expression + Rewiring (OOF triple stacked)",
}


# Preserve original V4 learner grid exactly
LEARNER_GRID = [
    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },
    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
    },
    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 5,
        "max_features": 0.30,
    },
    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 10,
        "max_features": 0.30,
    },
    {
        "learner": "coxnet",
        "l1_ratio": 0.5,
        "alpha_min_ratio": 0.01,
    },
    {
        "learner": "coxnet",
        "l1_ratio": 0.9,
        "alpha_min_ratio": 0.01,
    },
    {
        "learner": "coxnet",
        "l1_ratio": 0.5,
        "alpha_min_ratio": 0.1,
    },
    {
        "learner": "gbsa",
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 2,
        "subsample": 0.8,
    },
    {
        "learner": "gbsa",
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 3,
        "subsample": 0.8,
    },
]


# =============================================================================
# UTILITIES
# =============================================================================

def section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def checkpoint(cond, msg):
    print(f"[CHECKPOINT {'PASS' if cond else 'FAIL'}] {msg}")
    if not cond:
        raise RuntimeError(f"Checkpoint failed: {msg}")


def make_survival_y(df):
    events = pd.to_numeric(df["OS_event"], errors="raise").astype(int)
    times = pd.to_numeric(df["OS_months"], errors="raise").astype(float)

    if not events.isin([0, 1]).all():
        raise ValueError("OS_event must contain only 0/1.")

    if (times <= 0).any():
        raise ValueError("OS_months must be > 0.")

    return np.array(
        [(bool(e), float(t)) for e, t in zip(events, times)],
        dtype=[("event", "?"), ("time", "<f8")],
    )


def cindex(y, risk):
    risk = np.asarray(risk, dtype=float)

    if not np.isfinite(risk).all():
        return np.nan

    return float(
        concordance_index_censored(
            y["event"],
            y["time"],
            risk,
        )[0]
    )


# =============================================================================
# BASE LEARNERS
# =============================================================================

def fit_predict(params, X_train, y_train, X_valid):

    if params["learner"] == "rsf":

        model = RandomSurvivalForest(
            n_estimators=params["n_estimators"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            min_samples_split=10,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)
        return model.predict(X_valid)

    elif params["learner"] == "coxnet":

        # Preserved exactly from V4 for controlled comparison.
        model = CoxnetSurvivalAnalysis(
            l1_ratio=params["l1_ratio"],
            alpha_min_ratio=params["alpha_min_ratio"],
        )

        model.fit(X_train, y_train)
        return model.predict(X_valid)

    elif params["learner"] == "gbsa":

        model = GradientBoostingSurvivalAnalysis(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=params["subsample"],
            random_state=RANDOM_SEED,
        )

        model.fit(X_train, y_train)
        return model.predict(X_valid)

    else:
        raise ValueError(params["learner"])


def fit_predict_seed_ensemble(
    params,
    X_train,
    y_train,
    X_valid,
    n_seeds=N_FINAL_SEEDS,
):

    if params["learner"] != "rsf":
        return fit_predict(
            params,
            X_train,
            y_train,
            X_valid,
        )

    preds = []

    for seed_offset in range(n_seeds):

        model = RandomSurvivalForest(
            n_estimators=params["n_estimators"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            min_samples_split=10,
            random_state=RANDOM_SEED + seed_offset,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)
        preds.append(model.predict(X_valid))

    return np.mean(preds, axis=0)


# =============================================================================
# CLINICAL PREPROCESSING
# =============================================================================

def make_onehot():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def build_clinical_preprocessor(clinical_df):

    numeric_cols = [
        c
        for c in [
            "Diagnosis Age",
            "Karnofsky Performance Score",
        ]
        if c in clinical_df.columns
    ]

    categorical_cols = [
        c
        for c in [
            "Sex",
            "MGMT Status",
        ]
        if c in clinical_df.columns
    ]

    transformers = []

    if numeric_cols:
        transformers.append(
            (
                "numeric",
                SimpleImputer(strategy="median"),
                numeric_cols,
            )
        )

    if categorical_cols:

        cat_pipe = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "onehot",
                    make_onehot(),
                ),
            ]
        )

        transformers.append(
            (
                "categorical",
                cat_pipe,
                categorical_cols,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


# =============================================================================
# MOLECULAR FEATURE RANKING
# =============================================================================

def rank_molecular_features(X_train, y_train):

    feature_names = list(X_train.columns)

    values = X_train.to_numpy(
        dtype=np.float64
    )

    scores = np.full(
        len(feature_names),
        0.5,
        dtype=float,
    )

    for j in range(values.shape[1]):

        x = values[:, j]

        if np.nanstd(x) <= 1e-12:
            continue

        try:

            c = cindex(
                y_train,
                x,
            )

            scores[j] = (
                0.5
                if np.isnan(c)
                else max(c, 1.0 - c)
            )

        except Exception:
            pass

    ranking = pd.DataFrame(
        {
            "feature": feature_names,
            "univariate_c": scores,
        }
    )

    return ranking.sort_values(
        ["univariate_c", "feature"],
        ascending=[False, True],
    ).reset_index(drop=True)


_RANKING_CACHE = {}


def rank_molecular_features_cached(
    molecular,
    train_ids,
    y_train,
):

    key = (
        id(molecular),
        tuple(sorted(train_ids)),
    )

    cached = _RANKING_CACHE.get(key)

    if cached is None:

        cached = rank_molecular_features(
            molecular.loc[train_ids],
            y_train,
        )

        _RANKING_CACHE[key] = cached

    return cached


def select_molecular(
    molecular,
    train_ids,
    y_train,
    top_k,
):

    ranking = rank_molecular_features_cached(
        molecular,
        train_ids,
        y_train,
    )

    return (
        ranking["feature"]
        .head(top_k)
        .tolist()
    )


# =============================================================================
# BASIC BLOCK PREPARATION
# =============================================================================

def prepare_clinical_block(
    clinical,
    train_ids,
    valid_ids,
):

    clinical_train = clinical.loc[train_ids]
    clinical_valid = clinical.loc[valid_ids]

    processor = build_clinical_preprocessor(
        clinical_train
    )

    X_train = processor.fit_transform(
        clinical_train
    )

    X_valid = processor.transform(
        clinical_valid
    )

    return X_train, X_valid


def prepare_molecular_block(
    molecular,
    selected_cols,
    train_ids,
    valid_ids,
):

    X_train = molecular.loc[
        train_ids,
        selected_cols,
    ].to_numpy(
        dtype=np.float64
    )

    X_valid = molecular.loc[
        valid_ids,
        selected_cols,
    ].to_numpy(
        dtype=np.float64
    )

    return X_train, X_valid


# =============================================================================
# OOF STACKING
# =============================================================================

def generate_oof_base_risks(
    block_type,
    clinical,
    molecular,
    train_ids,
    y_train,
    survival,
    top_k,
    learner_params,
):

    """
    Generate leakage-safe OOF base-model risk predictions.

    block_type:
        "clinical"
        "expression"
        "rewiring"

    IMPORTANT:
    For molecular blocks, feature selection is repeated INSIDE each
    meta-OOF training fold.
    """

    train_ids = list(train_ids)

    train_events = (
        survival.loc[
            train_ids,
            "OS_event",
        ]
        .astype(int)
        .to_numpy()
    )

    meta_cv = StratifiedKFold(
        n_splits=META_OOF_FOLDS,
        shuffle=True,
        random_state=RANDOM_SEED + 777,
    )

    oof_risk = np.full(
        len(train_ids),
        np.nan,
        dtype=float,
    )

    for (
        meta_train_idx,
        meta_valid_idx,
    ) in meta_cv.split(
        train_ids,
        train_events,
    ):

        meta_train_ids = [
            train_ids[i]
            for i in meta_train_idx
        ]

        meta_valid_ids = [
            train_ids[i]
            for i in meta_valid_idx
        ]

        y_meta_train = make_survival_y(
            survival.loc[
                meta_train_ids
            ]
        )

        if block_type == "clinical":

            X_meta_train, X_meta_valid = (
                prepare_clinical_block(
                    clinical,
                    meta_train_ids,
                    meta_valid_ids,
                )
            )

        else:

            if block_type == "expression":
                block_df = molecular

            elif block_type == "rewiring":
                block_df = molecular

            else:
                raise ValueError(block_type)

            selected = select_molecular(
                block_df,
                meta_train_ids,
                y_meta_train,
                top_k,
            )

            (
                X_meta_train,
                X_meta_valid,
            ) = prepare_molecular_block(
                block_df,
                selected,
                meta_train_ids,
                meta_valid_ids,
            )

        preds = fit_predict(
            learner_params,
            X_meta_train,
            y_meta_train,
            X_meta_valid,
        )

        oof_risk[
            meta_valid_idx
        ] = preds

    if not np.isfinite(
        oof_risk
    ).all():

        raise RuntimeError(
            "Non-finite OOF risk generated."
        )

    return oof_risk


def fit_full_block_predict(
    block_type,
    clinical,
    molecular,
    train_ids,
    valid_ids,
    y_train,
    top_k,
    learner_params,
    use_seed_ensemble=False,
):

    """
    Fit one base block on ALL current training patients
    and predict the held-out current validation patients.

    Feature selection is performed using current training patients only.
    """

    predict_fn = (
        fit_predict_seed_ensemble
        if use_seed_ensemble
        else fit_predict
    )

    selected = ["clinical"]

    if block_type == "clinical":

        X_train, X_valid = (
            prepare_clinical_block(
                clinical,
                train_ids,
                valid_ids,
            )
        )

    else:

        selected = select_molecular(
            molecular,
            train_ids,
            y_train,
            top_k,
        )

        X_train, X_valid = (
            prepare_molecular_block(
                molecular,
                selected,
                train_ids,
                valid_ids,
            )
        )

    valid_risk = predict_fn(
        learner_params,
        X_train,
        y_train,
        X_valid,
    )

    return valid_risk, selected


def fit_oof_stacked_and_predict(
    model_name,
    clinical,
    expression,
    rewiring,
    survival,
    train_ids,
    valid_ids,
    y_train,
    top_k,
    learner_params,
    use_seed_ensemble=False,
):

    """
    Leakage-safe stacked prediction.

    Meta-model training features:
        OOF risks generated inside current training cohort.

    Meta-model validation features:
        risks from base models fitted on the entire current training cohort.

    No current validation outcome enters:
        feature selection
        base learner training
        OOF risk generation
        meta-model fitting
    """

    if model_name == "M4":

        block_specs = [
            (
                "clinical",
                None,
            ),
            (
                "expression",
                expression,
            ),
        ]

    elif model_name == "M5":

        block_specs = [
            (
                "clinical",
                None,
            ),
            (
                "rewiring",
                rewiring,
            ),
        ]

    elif model_name == "M6":

        block_specs = [
            (
                "clinical",
                None,
            ),
            (
                "expression",
                expression,
            ),
            (
                "rewiring",
                rewiring,
            ),
        ]

    else:
        raise ValueError(
            f"Not a stacked model: {model_name}"
        )

    train_risks = []
    valid_risks = []
    all_selected = []

    for (
        block_type,
        molecular_df,
    ) in block_specs:

        # -------------------------------------------------------------
        # OOF training risks for meta-model
        # -------------------------------------------------------------

        oof_risk = generate_oof_base_risks(
            block_type=block_type,
            clinical=clinical,
            molecular=molecular_df,
            train_ids=train_ids,
            y_train=y_train,
            survival=survival,
            top_k=top_k,
            learner_params=learner_params,
        )

        train_risks.append(
            oof_risk
        )

        # -------------------------------------------------------------
        # Full-training fit -> current held-out validation risk
        # -------------------------------------------------------------

        valid_risk, selected = (
            fit_full_block_predict(
                block_type=block_type,
                clinical=clinical,
                molecular=molecular_df,
                train_ids=train_ids,
                valid_ids=valid_ids,
                y_train=y_train,
                top_k=top_k,
                learner_params=learner_params,
                use_seed_ensemble=use_seed_ensemble,
            )
        )

        valid_risks.append(
            valid_risk
        )

        if block_type != "clinical":

            prefix = (
                "EXPR::"
                if block_type == "expression"
                else "REWIRE::"
            )

            all_selected.extend(
                [
                    prefix + f
                    for f in selected
                ]
            )

    meta_X_train = np.column_stack(
        train_risks
    )

    meta_X_valid = np.column_stack(
        valid_risks
    )

    # Preserve original V4 meta-model
    meta_model = CoxnetSurvivalAnalysis(
        l1_ratio=0.01,
        alpha_min_ratio=0.5,
    )

    meta_model.fit(
        meta_X_train,
        y_train,
    )

    meta_risk = meta_model.predict(
        meta_X_valid
    )

    return meta_risk, all_selected


# =============================================================================
# MODEL EVALUATION
# =============================================================================

def evaluate_model(
    model_name,
    train_ids,
    valid_ids,
    clinical,
    expression,
    rewiring,
    survival,
    top_k,
    learner_params,
    use_seed_ensemble=False,
):

    predict_fn = (
        fit_predict_seed_ensemble
        if use_seed_ensemble
        else fit_predict
    )

    y_train = make_survival_y(
        survival.loc[
            train_ids
        ]
    )

    # -------------------------------------------------------------------------
    # M1 CLINICAL
    # -------------------------------------------------------------------------

    if model_name == "M1":

        X_train, X_valid = (
            prepare_clinical_block(
                clinical,
                train_ids,
                valid_ids,
            )
        )

        risk = predict_fn(
            learner_params,
            X_train,
            y_train,
            X_valid,
        )

        return risk, ["clinical"]

    # -------------------------------------------------------------------------
    # STACKED M4 / M5 / M6
    # -------------------------------------------------------------------------

    if model_name in (
        "M4",
        "M5",
        "M6",
    ):

        return fit_oof_stacked_and_predict(
            model_name=model_name,
            clinical=clinical,
            expression=expression,
            rewiring=rewiring,
            survival=survival,
            train_ids=train_ids,
            valid_ids=valid_ids,
            y_train=y_train,
            top_k=top_k,
            learner_params=learner_params,
            use_seed_ensemble=use_seed_ensemble,
        )

    # -------------------------------------------------------------------------
    # STANDALONE MOLECULAR M2 / M3
    # -------------------------------------------------------------------------

    molecular = {
        "M2": expression,
        "M3": rewiring,
    }[model_name]

    selected = select_molecular(
        molecular,
        train_ids,
        y_train,
        top_k,
    )

    X_train, X_valid = (
        prepare_molecular_block(
            molecular,
            selected,
            train_ids,
            valid_ids,
        )
    )

    risk = predict_fn(
        learner_params,
        X_train,
        y_train,
        X_valid,
    )

    return risk, selected


# =============================================================================
# STEP 1 — LOAD DATA
# =============================================================================

section(
    "STEP 1 -- LOAD STAGE-D LOCKED DATA"
)

for path in [
    SPLIT_FILE,
    COHORT_FILE,
    M1_FILE,
    M2_FILE,
    M3_FILE,
]:
    checkpoint(
        path.exists(),
        f"{path.name} exists",
    )


split_df = pd.read_csv(
    SPLIT_FILE
)

survival = pd.read_csv(
    COHORT_FILE
).set_index(
    "patient_id"
)

clinical = pd.read_csv(
    M1_FILE,
    index_col=0,
)

expression = pd.read_csv(
    M2_FILE,
    index_col=0,
)

rewiring = pd.read_csv(
    M3_FILE,
    index_col=0,
)


development_ids = (
    split_df.loc[
        split_df["split"]
        == "development",
        "patient_id",
    ]
    .tolist()
)

test_ids = (
    split_df.loc[
        split_df["split"]
        == "locked_test",
        "patient_id",
    ]
    .tolist()
)


checkpoint(
    len(development_ids) == 115,
    "115 development patients",
)

checkpoint(
    len(test_ids) == 29,
    "29 internal-test patients",
)

checkpoint(
    not (
        set(development_ids)
        & set(test_ids)
    ),
    "development/test remain disjoint",
)


for name, df in [
    ("survival", survival),
    ("clinical", clinical),
    ("expression", expression),
    ("rewiring", rewiring),
]:

    checkpoint(
        df.index.is_unique,
        f"{name} patient IDs unique",
    )


all_ids = set(
    development_ids
    + test_ids
)

checkpoint(
    all_ids.issubset(
        survival.index
    ),
    "all split IDs present in survival",
)

checkpoint(
    all_ids.issubset(
        clinical.index
    ),
    "all split IDs present in clinical",
)

checkpoint(
    all_ids.issubset(
        expression.index
    ),
    "all split IDs present in expression",
)

checkpoint(
    all_ids.issubset(
        rewiring.index
    ),
    "all split IDs present in rewiring",
)


print(
    "\nModels:",
    [
        (
            m,
            MODEL_DESCRIPTIONS[m],
        )
        for m in MODELS
    ],
)

print(
    f"Base learner candidates: "
    f"{len(LEARNER_GRID)}"
)

print(
    f"Top-K values: "
    f"{TOP_K_VALUES}"
)

print(
    f"Outer CV: "
    f"{N_OUTER_REPEATS} x "
    f"{OUTER_FOLDS}"
)

print(
    f"Inner CV: "
    f"{INNER_FOLDS}"
)

print(
    f"Meta OOF folds: "
    f"{META_OOF_FOLDS}"
)


# =============================================================================
# STEP 2 — REPEATED NESTED DEVELOPMENT CV
# =============================================================================

section(
    "STEP 2 -- REPEATED NESTED DEVELOPMENT-ONLY CV"
)


development_survival = (
    survival.loc[
        development_ids
    ]
)

development_events = (
    development_survival[
        "OS_event"
    ]
    .astype(int)
    .to_numpy()
)


outer_results = []
outer_selected_features = []

pipeline_start = time.time()


for repeat in range(
    1,
    N_OUTER_REPEATS + 1,
):

    outer_cv = StratifiedKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_SEED
            + repeat * 100
        ),
    )

    print(
        "\n"
        + "#" * 70
    )

    print(
        f"REPEAT "
        f"{repeat}/"
        f"{N_OUTER_REPEATS}"
    )

    print(
        "#" * 70
    )


    for (
        outer_fold,
        (
            outer_train_idx,
            outer_valid_idx,
        ),
    ) in enumerate(
        outer_cv.split(
            development_ids,
            development_events,
        ),
        start=1,
    ):

        fold_start = time.time()

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"REPEAT {repeat} "
            f"-- OUTER FOLD "
            f"{outer_fold}/"
            f"{OUTER_FOLDS}"
        )

        print(
            "=" * 70
        )


        outer_train_ids = [
            development_ids[i]
            for i
            in outer_train_idx
        ]

        outer_valid_ids = [
            development_ids[i]
            for i
            in outer_valid_idx
        ]

        y_outer_valid = make_survival_y(
            survival.loc[
                outer_valid_ids
            ]
        )

        inner_events = (
            survival.loc[
                outer_train_ids,
                "OS_event",
            ]
            .astype(int)
            .to_numpy()
        )


        for model_name in MODELS:

            print(
                f"\n  MODEL "
                f"{model_name} "
                f"("
                f"{MODEL_DESCRIPTIONS[model_name]}"
                f")"
            )


            candidate_k_values = (
                [None]
                if model_name == "M1"
                else TOP_K_VALUES
            )

            candidate_results = []


            for top_k in candidate_k_values:

                for params in LEARNER_GRID:

                    inner_cv = StratifiedKFold(
                        n_splits=INNER_FOLDS,
                        shuffle=True,
                        random_state=(
                            RANDOM_SEED
                            + repeat * 100
                            + outer_fold
                        ),
                    )

                    fold_scores = []


                    for (
                        inner_train_idx,
                        inner_valid_idx,
                    ) in inner_cv.split(
                        outer_train_ids,
                        inner_events,
                    ):

                        inner_train_ids = [
                            outer_train_ids[i]
                            for i
                            in inner_train_idx
                        ]

                        inner_valid_ids = [
                            outer_train_ids[i]
                            for i
                            in inner_valid_idx
                        ]

                        y_inner_valid = (
                            make_survival_y(
                                survival.loc[
                                    inner_valid_ids
                                ]
                            )
                        )

                        try:

                            risk, _ = evaluate_model(
                                model_name,
                                inner_train_ids,
                                inner_valid_ids,
                                clinical,
                                expression,
                                rewiring,
                                survival,
                                top_k,
                                params,
                                use_seed_ensemble=False,
                            )

                            score = cindex(
                                y_inner_valid,
                                risk,
                            )

                        except Exception as exc:

                            print(
                                "      "
                                "[candidate failure] "
                                f"{model_name}, "
                                f"k={top_k}, "
                                f"{params['learner']}: "
                                f"{type(exc).__name__}: "
                                f"{exc}"
                            )

                            score = np.nan


                        fold_scores.append(
                            score
                        )


                    valid_fold_scores = [
                        s
                        for s in fold_scores
                        if np.isfinite(s)
                    ]

                    mean_inner = (
                        float(
                            np.mean(
                                valid_fold_scores
                            )
                        )
                        if valid_fold_scores
                        else np.nan
                    )


                    candidate_results.append(
                        {
                            "top_k": top_k,
                            "params": params,
                            "mean_inner_c": mean_inner,
                        }
                    )


            valid_candidates = [
                x
                for x in candidate_results
                if np.isfinite(
                    x["mean_inner_c"]
                )
            ]

            if not valid_candidates:

                raise RuntimeError(
                    f"No valid candidates for "
                    f"{model_name} "
                    f"repeat={repeat} "
                    f"outer_fold={outer_fold}"
                )


            best = max(
                valid_candidates,
                key=lambda x:
                    x["mean_inner_c"],
            )


            print(
                f"    Best inner C-index: "
                f"{best['mean_inner_c']:.4f}"
            )

            print(
                f"    Learner: "
                f"{best['params']['learner']}"
            )

            print(
                f"    Top-K: "
                f"{best['top_k']}"
            )


            risk, selected = (
                evaluate_model(
                    model_name,
                    outer_train_ids,
                    outer_valid_ids,
                    clinical,
                    expression,
                    rewiring,
                    survival,
                    best["top_k"],
                    best["params"],
                    use_seed_ensemble=False,
                )
            )


            outer_c = cindex(
                y_outer_valid,
                risk,
            )


            print(
                f"    OUTER C-index: "
                f"{outer_c:.4f}"
            )


            outer_results.append(
                {
                    "repeat": repeat,
                    "outer_fold": outer_fold,
                    "model": model_name,
                    "description":
                        MODEL_DESCRIPTIONS[
                            model_name
                        ],
                    "top_k":
                        best["top_k"],
                    "learner":
                        best["params"][
                            "learner"
                        ],
                    "inner_cindex":
                        best[
                            "mean_inner_c"
                        ],
                    "outer_cindex":
                        outer_c,
                    "n_selected_molecular":
                        (
                            0
                            if model_name == "M1"
                            else len(selected)
                        ),
                }
            )


            if model_name != "M1":

                for (
                    rank,
                    feature,
                ) in enumerate(
                    selected,
                    start=1,
                ):

                    outer_selected_features.append(
                        {
                            "repeat":
                                repeat,
                            "outer_fold":
                                outer_fold,
                            "model":
                                model_name,
                            "rank":
                                rank,
                            "feature":
                                feature,
                        }
                    )


        pd.DataFrame(
            outer_results
        ).to_csv(
            RESULT_DIR
            / "STAGE_D_V4_OOF_OUTER_CV_RESULTS_partial.csv",
            index=False,
        )


        elapsed = (
            time.time()
            - fold_start
        )

        total_elapsed = (
            time.time()
            - pipeline_start
        )


        print(
            f"\n  [TIMING] "
            f"fold took "
            f"{elapsed / 60:.1f} min"
        )

        print(
            f"  Total elapsed: "
            f"{total_elapsed / 60:.1f} min"
        )

        print(
            f"  Ranking cache size: "
            f"{len(_RANKING_CACHE)}"
        )


# =============================================================================
# STEP 3 — DEVELOPMENT SUMMARY
# =============================================================================

section(
    "STEP 3 -- DEVELOPMENT CV SUMMARY"
)


outer_results_df = pd.DataFrame(
    outer_results
)


development_summary = (
    outer_results_df
    .groupby(
        "model"
    )
    .agg(
        mean_outer_cindex=(
            "outer_cindex",
            "mean",
        ),
        sd_outer_cindex=(
            "outer_cindex",
            "std",
        ),
        median_outer_cindex=(
            "outer_cindex",
            "median",
        ),
        min_outer_cindex=(
            "outer_cindex",
            "min",
        ),
        max_outer_cindex=(
            "outer_cindex",
            "max",
        ),
        n_folds=(
            "outer_cindex",
            "count",
        ),
    )
    .reset_index()
)


development_summary[
    "description"
] = development_summary[
    "model"
].map(
    MODEL_DESCRIPTIONS
)


print(
    "\nDEVELOPMENT PERFORMANCE:"
)

print(
    development_summary.to_string(
        index=False
    )
)


outer_results_df.to_csv(
    RESULT_DIR
    / "STAGE_D_V4_OOF_OUTER_CV_RESULTS.csv",
    index=False,
)

development_summary.to_csv(
    RESULT_DIR
    / "STAGE_D_V4_OOF_DEVELOPMENT_SUMMARY.csv",
    index=False,
)

pd.DataFrame(
    outer_selected_features
).to_csv(
    RESULT_DIR
    / "STAGE_D_V4_OOF_OUTER_SELECTED_FEATURES.csv",
    index=False,
)


# =============================================================================
# STEP 4 — FINAL TUNING ON ALL DEVELOPMENT PATIENTS
# =============================================================================

section(
    "STEP 4 -- FINAL TUNING ON ALL 115 DEVELOPMENT PATIENTS"
)


final_specs = {}


for model_name in MODELS:

    print(
        f"\nFinal tuning: "
        f"{model_name}"
    )


    candidate_k_values = (
        [None]
        if model_name == "M1"
        else TOP_K_VALUES
    )


    candidate_results = []


    final_inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_SEED
            + 1000
        ),
    )


    for top_k in candidate_k_values:

        for params in LEARNER_GRID:

            scores = []


            for (
                train_idx,
                valid_idx,
            ) in final_inner_cv.split(
                development_ids,
                development_events,
            ):

                train_ids = [
                    development_ids[i]
                    for i
                    in train_idx
                ]

                valid_ids = [
                    development_ids[i]
                    for i
                    in valid_idx
                ]

                y_valid = make_survival_y(
                    survival.loc[
                        valid_ids
                    ]
                )


                try:

                    risk, _ = evaluate_model(
                        model_name,
                        train_ids,
                        valid_ids,
                        clinical,
                        expression,
                        rewiring,
                        survival,
                        top_k,
                        params,
                        use_seed_ensemble=False,
                    )

                    scores.append(
                        cindex(
                            y_valid,
                            risk,
                        )
                    )

                except Exception as exc:

                    print(
                        "    "
                        "[final-tuning failure] "
                        f"{model_name}, "
                        f"k={top_k}, "
                        f"{params['learner']}: "
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    )

                    scores.append(
                        np.nan
                    )


            valid_scores = [
                s
                for s in scores
                if np.isfinite(s)
            ]

            mean_c = (
                float(
                    np.mean(
                        valid_scores
                    )
                )
                if valid_scores
                else np.nan
            )


            candidate_results.append(
                {
                    "top_k":
                        top_k,
                    "params":
                        params,
                    "mean_cindex":
                        mean_c,
                }
            )


    valid_candidates = [
        x
        for x in candidate_results
        if np.isfinite(
            x["mean_cindex"]
        )
    ]


    if not valid_candidates:

        raise RuntimeError(
            f"No valid final tuning "
            f"candidate for "
            f"{model_name}"
        )


    best = max(
        valid_candidates,
        key=lambda x:
            x["mean_cindex"],
    )


    final_specs[
        model_name
    ] = best


    print(
        "  Best specification:"
    )

    print(
        f"  {best}"
    )


# Save specification BEFORE test
with open(
    RESULT_DIR
    / "STAGE_D_V4_OOF_FINAL_SPECS.json",
    "w",
) as f:

    json.dump(
        final_specs,
        f,
        indent=2,
    )


print(
    "\nFinal specifications "
    "saved BEFORE internal-test evaluation."
)


# =============================================================================
# STEP 5 — INTERNAL TEST
# =============================================================================

section(
    "STEP 5 -- INTERNAL 29-PATIENT TEST"
)


locked_results = []
final_selected_features = []

y_test = make_survival_y(
    survival.loc[
        test_ids
    ]
)


for model_name in MODELS:

    spec = final_specs[
        model_name
    ]


    risk, selected = evaluate_model(
        model_name,
        development_ids,
        test_ids,
        clinical,
        expression,
        rewiring,
        survival,
        spec["top_k"],
        spec["params"],
        use_seed_ensemble=True,
    )


    test_c = cindex(
        y_test,
        risk,
    )


    print(
        f"\nFINAL "
        f"{model_name} "
        f"("
        f"{MODEL_DESCRIPTIONS[model_name]}"
        f")"
    )

    print(
        f"Internal-test C-index: "
        f"{test_c:.4f}"
    )

    print(
        f"Learner: "
        f"{spec['params']['learner']}"
    )

    print(
        f"Top-K: "
        f"{spec['top_k']}"
    )


    locked_results.append(
        {
            "model":
                model_name,
            "description":
                MODEL_DESCRIPTIONS[
                    model_name
                ],
            "development_inner_cindex":
                spec[
                    "mean_cindex"
                ],
            "internal_test_cindex":
                test_c,
            "top_k":
                spec[
                    "top_k"
                ],
            "learner":
                spec[
                    "params"
                ][
                    "learner"
                ],
            "n_selected_molecular_features":
                (
                    0
                    if model_name == "M1"
                    else len(selected)
                ),
        }
    )


    if model_name != "M1":

        for (
            rank,
            feature,
        ) in enumerate(
            selected,
            start=1,
        ):

            final_selected_features.append(
                {
                    "model":
                        model_name,
                    "rank":
                        rank,
                    "feature":
                        feature,
                }
            )


locked_results_df = pd.DataFrame(
    locked_results
)


# =============================================================================
# STEP 6 — RESULTS
# =============================================================================

section(
    "STEP 6 -- FINAL M1-M6 RESULTS"
)


print(
    locked_results_df.to_string(
        index=False
    )
)


locked_results_df.to_csv(
    RESULT_DIR
    / "STAGE_D_V4_OOF_INTERNAL_TEST_RESULTS.csv",
    index=False,
)


pd.DataFrame(
    final_selected_features
).to_csv(
    RESULT_DIR
    / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv",
    index=False,
)


# =============================================================================
# AUDIT
# =============================================================================

audit = {

    "version":
        "V4_OOF_CORRECTED",

    "outer_repeats":
        N_OUTER_REPEATS,

    "outer_folds":
        OUTER_FOLDS,

    "inner_folds":
        INNER_FOLDS,

    "meta_oof_folds":
        META_OOF_FOLDS,

    "top_k_values":
        TOP_K_VALUES,

    "final_seed_ensemble_size":
        N_FINAL_SEEDS,

    "learners":
        [
            "rsf",
            "coxnet",
            "gbsa",
        ],

    "models":
        MODEL_DESCRIPTIONS,

    "stacking_training_risks":
        "internal out-of-fold predictions",

    "molecular_selection_inside_meta_oof":
        True,

    "test_set_used_for_tuning":
        False,

    "test_set_used_for_feature_selection":
        False,

    "note":
        (
            "Internal 29-patient test has already "
            "been observed in previous project "
            "iterations. Use repeated development "
            "outer CV as the primary comparison "
            "between V4 and V4-OOF-corrected. "
            "External validation remains required."
        ),
}


with open(
    RESULT_DIR
    / "STAGE_D_V4_OOF_AUDIT.json",
    "w",
) as f:

    json.dump(
        audit,
        f,
        indent=2,
    )


section(
    "V4 OOF-CORRECTED RUN COMPLETE"
)


print(
    f"Total wall time: "
    f"{(time.time() - pipeline_start) / 60:.1f} min"
)


print(
    "\nResults saved in:"
)

print(
    RESULT_DIR
)


print(
    "\nPRIMARY FILE TO EXAMINE FIRST:"
)

print(
    RESULT_DIR
    / "STAGE_D_V4_OOF_DEVELOPMENT_SUMMARY.csv"
)


print(
    "\nIMPORTANT:"
)

print(
    "Compare original V4 versus this corrected "
    "version using DEVELOPMENT OUTER CV first."
)

print(
    "Do NOT choose the version based on the "
    "29-patient internal-test C-index."
)

### STAGE D — V4 OOF STACKING DIAGNOSTIC

In [ ]:
"""
STAGE D — V4 OOF STACKING DIAGNOSTIC
=====================================

PURPOSE
-------
Diagnose why corrected V4 models M4, M5 and M6 all produced
internal-test C-index = 0.5000.

THIS SCRIPT DOES NOT:
- rerun repeated nested CV
- retune Top-K
- retune the base learner
- change the 115/29 split
- use the internal test to choose a model

IT DOES:
1. Load the frozen final M4/M5/M6 specifications.
2. Reconstruct leakage-safe OOF training risks.
3. Reconstruct internal-test base risks.
4. Inspect variance / range / correlations.
5. Fit the original CoxNet meta-model.
6. Inspect CoxNet alphas and coefficient norms.
7. Check whether meta predictions are constant.
8. Fit a CoxPH sanity-check meta-model ONLY to diagnose whether
   the problem comes from the CoxNet meta layer.

Do NOT select a final model from the internal-test C-index produced here.
"""

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sksurv.ensemble import (
    RandomSurvivalForest,
    GradientBoostingSurvivalAnalysis,
)
from sksurv.linear_model import (
    CoxnetSurvivalAnalysis,
    CoxPHSurvivalAnalysis,
)
from sksurv.metrics import concordance_index_censored


warnings.filterwarnings("ignore")


# =============================================================================
# CONFIG
# =============================================================================

RANDOM_SEED = 42
META_OOF_FOLDS = 3
N_FINAL_SEEDS = 5

DOWNLOADS = BASE

STAGE_D_DIR = (
    DOWNLOADS
    / "TCGA_GBM_STAGE_D_SURVIVAL"
)

RESULT_DIR = (
    STAGE_D_DIR
    / "STAGE_D_PART3_V4_OOF_CORRECTED"
)

SPLIT_FILE = (
    STAGE_D_DIR
    / "STAGE_D_LOCKED_SPLIT.csv"
)

COHORT_FILE = (
    STAGE_D_DIR
    / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
)

M1_FILE = (
    STAGE_D_DIR
    / "STAGE_D_M1_CLINICAL_RAW.csv"
)

M2_FILE = (
    STAGE_D_DIR
    / "STAGE_D_M2_EXPRESSION_RAW.csv"
)

M3_FILE = (
    STAGE_D_DIR
    / "STAGE_D_M3_REWIRING_RAW.csv"
)

SPECS_FILE = (
    RESULT_DIR
    / "STAGE_D_V4_OOF_FINAL_SPECS.json"
)

DIAG_DIR = (
    RESULT_DIR
    / "STACKING_DIAGNOSTIC"
)

DIAG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# UTILITIES
# =============================================================================

def section(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def checkpoint(condition, message):
    status = "PASS" if condition else "FAIL"
    print(f"[CHECKPOINT {status}] {message}")

    if not condition:
        raise RuntimeError(message)


def make_survival_y(df):

    return np.array(
        [
            (bool(e), float(t))
            for e, t in zip(
                df["OS_event"],
                df["OS_months"],
            )
        ],
        dtype=[
            ("event", "?"),
            ("time", "<f8"),
        ],
    )


def cindex(y, risk):

    risk = np.asarray(
        risk,
        dtype=float,
    )

    if not np.isfinite(risk).all():
        return np.nan

    return float(
        concordance_index_censored(
            y["event"],
            y["time"],
            risk,
        )[0]
    )


def describe_vector(name, x):

    x = np.asarray(
        x,
        dtype=float,
    )

    finite = x[
        np.isfinite(x)
    ]

    if len(finite) == 0:
        print(
            f"{name}: NO FINITE VALUES"
        )
        return

    print(
        f"{name}: "
        f"n={len(finite)}, "
        f"mean={np.mean(finite):.8f}, "
        f"sd={np.std(finite):.8f}, "
        f"min={np.min(finite):.8f}, "
        f"max={np.max(finite):.8f}, "
        f"range={np.ptp(finite):.8f}, "
        f"unique≈{len(np.unique(np.round(finite, 10)))}"
    )


# =============================================================================
# LOAD DATA
# =============================================================================

section(
    "STEP 1 — LOAD DATA"
)

for path in [
    SPLIT_FILE,
    COHORT_FILE,
    M1_FILE,
    M2_FILE,
    M3_FILE,
    SPECS_FILE,
]:

    checkpoint(
        path.exists(),
        f"{path.name} exists",
    )


split_df = pd.read_csv(
    SPLIT_FILE
)

survival = pd.read_csv(
    COHORT_FILE
).set_index(
    "patient_id"
)

clinical = pd.read_csv(
    M1_FILE,
    index_col=0,
)

expression = pd.read_csv(
    M2_FILE,
    index_col=0,
)

rewiring = pd.read_csv(
    M3_FILE,
    index_col=0,
)


development_ids = (
    split_df.loc[
        split_df["split"]
        == "development",
        "patient_id",
    ]
    .tolist()
)

test_ids = (
    split_df.loc[
        split_df["split"]
        == "locked_test",
        "patient_id",
    ]
    .tolist()
)


checkpoint(
    len(development_ids) == 115,
    "115 development patients",
)

checkpoint(
    len(test_ids) == 29,
    "29 internal-test patients",
)


with open(
    SPECS_FILE,
    "r",
) as f:

    final_specs = json.load(f)


print(
    "\nFrozen final specifications:"
)

for m in [
    "M4",
    "M5",
    "M6",
]:

    print(
        m,
        final_specs[m],
    )


# =============================================================================
# PREPROCESSING
# =============================================================================

def make_onehot():

    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )

    except TypeError:

        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def build_clinical_preprocessor(df):

    numeric_cols = [
        c
        for c in [
            "Diagnosis Age",
            "Karnofsky Performance Score",
        ]
        if c in df.columns
    ]

    categorical_cols = [
        c
        for c in [
            "Sex",
            "MGMT Status",
        ]
        if c in df.columns
    ]

    transformers = []

    if numeric_cols:

        transformers.append(
            (
                "numeric",
                SimpleImputer(
                    strategy="median"
                ),
                numeric_cols,
            )
        )

    if categorical_cols:

        cat_pipe = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    ),
                ),
                (
                    "onehot",
                    make_onehot(),
                ),
            ]
        )

        transformers.append(
            (
                "categorical",
                cat_pipe,
                categorical_cols,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


# =============================================================================
# BASE LEARNERS
# =============================================================================

def fit_predict(
    params,
    X_train,
    y_train,
    X_valid,
    seed=RANDOM_SEED,
):

    learner = params[
        "learner"
    ]

    if learner == "rsf":

        model = RandomSurvivalForest(
            n_estimators=params[
                "n_estimators"
            ],
            min_samples_leaf=params[
                "min_samples_leaf"
            ],
            max_features=params[
                "max_features"
            ],
            min_samples_split=10,
            random_state=seed,
            n_jobs=-1,
        )

    elif learner == "coxnet":

        model = CoxnetSurvivalAnalysis(
            l1_ratio=params[
                "l1_ratio"
            ],
            alpha_min_ratio=params[
                "alpha_min_ratio"
            ],
        )

    elif learner == "gbsa":

        model = GradientBoostingSurvivalAnalysis(
            n_estimators=params[
                "n_estimators"
            ],
            learning_rate=params[
                "learning_rate"
            ],
            max_depth=params[
                "max_depth"
            ],
            subsample=params[
                "subsample"
            ],
            random_state=seed,
        )

    else:

        raise ValueError(
            learner
        )

    model.fit(
        X_train,
        y_train,
    )

    return model.predict(
        X_valid
    )


def fit_predict_final(
    params,
    X_train,
    y_train,
    X_valid,
):

    if params[
        "learner"
    ] != "rsf":

        return fit_predict(
            params,
            X_train,
            y_train,
            X_valid,
        )

    predictions = []

    for seed_offset in range(
        N_FINAL_SEEDS
    ):

        predictions.append(
            fit_predict(
                params,
                X_train,
                y_train,
                X_valid,
                seed=(
                    RANDOM_SEED
                    + seed_offset
                ),
            )
        )

    return np.mean(
        predictions,
        axis=0,
    )


# =============================================================================
# MOLECULAR FEATURE SELECTION
# =============================================================================

def rank_molecular_features(
    X_train,
    y_train,
):

    names = list(
        X_train.columns
    )

    values = X_train.to_numpy(
        dtype=float
    )

    scores = np.full(
        len(names),
        0.5,
    )

    for j in range(
        values.shape[1]
    ):

        x = values[
            :,
            j,
        ]

        if np.nanstd(
            x
        ) <= 1e-12:
            continue

        try:

            cc = cindex(
                y_train,
                x,
            )

            if np.isfinite(
                cc
            ):

                scores[j] = max(
                    cc,
                    1.0 - cc,
                )

        except Exception:
            pass

    return (
        pd.DataFrame(
            {
                "feature": names,
                "score": scores,
            }
        )
        .sort_values(
            [
                "score",
                "feature",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


def select_features(
    molecular,
    train_ids,
    y_train,
    top_k,
):

    ranking = (
        rank_molecular_features(
            molecular.loc[
                train_ids
            ],
            y_train,
        )
    )

    return (
        ranking[
            "feature"
        ]
        .head(
            int(top_k)
        )
        .tolist()
    )


# =============================================================================
# PREPARE BLOCK
# =============================================================================

def prepare_clinical(
    train_ids,
    valid_ids,
):

    processor = (
        build_clinical_preprocessor(
            clinical.loc[
                train_ids
            ]
        )
    )

    X_train = (
        processor.fit_transform(
            clinical.loc[
                train_ids
            ]
        )
    )

    X_valid = (
        processor.transform(
            clinical.loc[
                valid_ids
            ]
        )
    )

    return (
        X_train,
        X_valid,
    )


def prepare_molecular(
    molecular,
    selected,
    train_ids,
    valid_ids,
):

    X_train = (
        molecular.loc[
            train_ids,
            selected,
        ]
        .to_numpy(
            dtype=float
        )
    )

    X_valid = (
        molecular.loc[
            valid_ids,
            selected,
        ]
        .to_numpy(
            dtype=float
        )
    )

    return (
        X_train,
        X_valid,
    )


# =============================================================================
# OOF BASE RISKS
# =============================================================================

def generate_oof_risk(
    block_name,
    molecular,
    top_k,
    params,
):

    train_events = (
        survival.loc[
            development_ids,
            "OS_event",
        ]
        .astype(int)
        .to_numpy()
    )

    cv = StratifiedKFold(
        n_splits=META_OOF_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_SEED
            + 777
        ),
    )

    oof = np.full(
        len(development_ids),
        np.nan,
    )

    for (
        fold,
        (
            train_idx,
            valid_idx,
        ),
    ) in enumerate(
        cv.split(
            development_ids,
            train_events,
        ),
        start=1,
    ):

        train_ids = [
            development_ids[i]
            for i in train_idx
        ]

        valid_ids = [
            development_ids[i]
            for i in valid_idx
        ]

        y_train = make_survival_y(
            survival.loc[
                train_ids
            ]
        )

        if block_name == "clinical":

            X_train, X_valid = (
                prepare_clinical(
                    train_ids,
                    valid_ids,
                )
            )

        else:

            selected = select_features(
                molecular,
                train_ids,
                y_train,
                top_k,
            )

            X_train, X_valid = (
                prepare_molecular(
                    molecular,
                    selected,
                    train_ids,
                    valid_ids,
                )
            )

        pred = fit_predict(
            params,
            X_train,
            y_train,
            X_valid,
        )

        oof[
            valid_idx
        ] = pred

    checkpoint(
        np.isfinite(
            oof
        ).all(),
        f"{block_name} OOF risks finite",
    )

    return oof


# =============================================================================
# FINAL TEST BASE RISKS
# =============================================================================

def generate_test_risk(
    block_name,
    molecular,
    top_k,
    params,
):

    y_train = make_survival_y(
        survival.loc[
            development_ids
        ]
    )

    if block_name == "clinical":

        X_train, X_test = (
            prepare_clinical(
                development_ids,
                test_ids,
            )
        )

    else:

        selected = select_features(
            molecular,
            development_ids,
            y_train,
            top_k,
        )

        print(
            f"\n{block_name}: "
            f"{len(selected)} final selected features"
        )

        print(
            selected[:10]
        )

        X_train, X_test = (
            prepare_molecular(
                molecular,
                selected,
                development_ids,
                test_ids,
            )
        )

    risk = fit_predict_final(
        params,
        X_train,
        y_train,
        X_test,
    )

    return risk


# =============================================================================
# DIAGNOSE ONE STACK
# =============================================================================

def diagnose_stack(
    model_name,
):

    section(
        f"DIAGNOSTIC — {model_name}"
    )

    spec = final_specs[
        model_name
    ]

    params = spec[
        "params"
    ]

    top_k = spec[
        "top_k"
    ]

    print(
        f"Frozen params: {params}"
    )

    print(
        f"Frozen Top-K: {top_k}"
    )


    if model_name == "M4":

        blocks = [
            (
                "clinical",
                None,
            ),
            (
                "expression",
                expression,
            ),
        ]

    elif model_name == "M5":

        blocks = [
            (
                "clinical",
                None,
            ),
            (
                "rewiring",
                rewiring,
            ),
        ]

    elif model_name == "M6":

        blocks = [
            (
                "clinical",
                None,
            ),
            (
                "expression",
                expression,
            ),
            (
                "rewiring",
                rewiring,
            ),
        ]

    else:

        raise ValueError(
            model_name
        )


    train_columns = []
    test_columns = []
    names = []


    for (
        block_name,
        molecular,
    ) in blocks:

        print(
            f"\n--- {block_name.upper()} ---"
        )

        oof_risk = generate_oof_risk(
            block_name,
            molecular,
            top_k,
            params,
        )

        test_risk = generate_test_risk(
            block_name,
            molecular,
            top_k,
            params,
        )

        describe_vector(
            f"{block_name} OOF",
            oof_risk,
        )

        describe_vector(
            f"{block_name} TEST",
            test_risk,
        )

        # >>> STEP 1 CONFIRMATION: block-level discrimination
        _y_dev_chk = make_survival_y(survival.loc[development_ids])
        _y_test_chk = make_survival_y(survival.loc[test_ids])
        print(
            f"{block_name} block C-index: "
            f"OOF (development) = {cindex(_y_dev_chk, oof_risk):.4f} | "
            f"internal test = {cindex(_y_test_chk, test_risk):.4f}"
        )
        # <<<

        train_columns.append(
            oof_risk
        )

        test_columns.append(
            test_risk
        )

        names.append(
            block_name
        )


    meta_train = np.column_stack(
        train_columns
    )

    meta_test = np.column_stack(
        test_columns
    )


    print(
        "\nMETA TRAIN CORRELATION:"
    )

    print(
        pd.DataFrame(
            meta_train,
            columns=names,
        )
        .corr()
        .round(4)
    )


    print(
        "\nMETA TEST CORRELATION:"
    )

    print(
        pd.DataFrame(
            meta_test,
            columns=names,
        )
        .corr()
        .round(4)
    )


    y_dev = make_survival_y(
        survival.loc[
            development_ids
        ]
    )

    y_test = make_survival_y(
        survival.loc[
            test_ids
        ]
    )


    # =========================================================================
    # ORIGINAL V4 COXNET META-MODEL
    # =========================================================================

    print(
        "\n--- ORIGINAL COXNET META-MODEL ---"
    )

    original_meta = (
        CoxnetSurvivalAnalysis(
            l1_ratio=0.01,
            alpha_min_ratio=0.5,
        )
    )

    original_meta.fit(
        meta_train,
        y_dev,
    )


    print(
        "alphas_:"
    )

    print(
        original_meta.alphas_
    )


    print(
        "\ncoef_ shape:",
        original_meta.coef_.shape,
    )


    coef_norms = np.linalg.norm(
        original_meta.coef_,
        axis=0,
    )


    print(
        "\nCoefficient norms "
        "across CoxNet path:"
    )

    print(
        coef_norms
    )


    default_train = (
        original_meta.predict(
            meta_train
        )
    )

    default_test = (
        original_meta.predict(
            meta_test
        )
    )


    describe_vector(
        "CoxNet DEFAULT meta TRAIN prediction",
        default_train,
    )

    describe_vector(
        "CoxNet DEFAULT meta TEST prediction",
        default_test,
    )


    print(
        f"\nCoxNet default "
        f"TRAIN C-index = "
        f"{cindex(y_dev, default_train):.6f}"
    )

    print(
        f"CoxNet default "
        f"TEST C-index = "
        f"{cindex(y_test, default_test):.6f}"
    )

    # >>> STEP 1 CONFIRMATION: is the stacked prediction just the clinical block?
    from scipy.stats import spearmanr
    rho_test = spearmanr(default_test, meta_test[:, 0]).correlation
    print(
        f"Spearman(stacked TEST prediction, clinical block TEST risk) = "
        f"{rho_test:.6f}   (1.000000 => stack = rescaled clinical risk)"
    )
    print(
        f"clinical block TEST C = {cindex(y_test, meta_test[:, 0]):.6f} | "
        f"stacked TEST C = {cindex(y_test, default_test):.6f}"
    )
    # <<<



    # Examine prediction variation across path.
    # DO NOT use test C-index to choose alpha.
    path_rows = []

    for i, alpha in enumerate(
        original_meta.alphas_
    ):

        try:

            p_train = (
                original_meta.predict(
                    meta_train,
                    alpha=float(alpha),
                )
            )

            p_test = (
                original_meta.predict(
                    meta_test,
                    alpha=float(alpha),
                )
            )

            path_rows.append(
                {
                    "path_index": i,
                    "alpha": float(alpha),
                    "coef_norm":
                        float(
                            coef_norms[i]
                        ),
                    "train_prediction_sd":
                        float(
                            np.std(
                                p_train
                            )
                        ),
                    "test_prediction_sd":
                        float(
                            np.std(
                                p_test
                            )
                        ),
                    "train_prediction_range":
                        float(
                            np.ptp(
                                p_train
                            )
                        ),
                    "test_prediction_range":
                        float(
                            np.ptp(
                                p_test
                            )
                        ),
                }
            )

        except Exception as exc:

            print(
                f"Path alpha failure "
                f"{alpha}: {exc}"
            )


    path_df = pd.DataFrame(
        path_rows
    )

    print(
        "\nCOXNET PATH DIAGNOSTIC:"
    )

    print(
        path_df.to_string(
            index=False
        )
    )


    path_df.to_csv(
        DIAG_DIR
        / f"{model_name}_COXNET_PATH.csv",
        index=False,
    )


    # =========================================================================
    # COXPH SANITY CHECK
    # =========================================================================

    print(
        "\n--- COXPH SANITY CHECK ---"
    )

    print(
        "This is diagnostic only. "
        "Do NOT choose it because of test performance."
    )


    # Standardize meta-risk columns using DEVELOPMENT ONLY.
    means = np.mean(
        meta_train,
        axis=0,
    )

    sds = np.std(
        meta_train,
        axis=0,
    )

    sds[
        sds < 1e-12
    ] = 1.0


    meta_train_z = (
        meta_train
        - means
    ) / sds

    meta_test_z = (
        meta_test
        - means
    ) / sds


    coxph = (
        CoxPHSurvivalAnalysis(
            alpha=0.1
        )
    )

    coxph.fit(
        meta_train_z,
        y_dev,
    )


    ph_train = (
        coxph.predict(
            meta_train_z
        )
    )

    ph_test = (
        coxph.predict(
            meta_test_z
        )
    )


    print(
        "\nCoxPH coefficients:"
    )

    for name, coef in zip(
        names,
        coxph.coef_,
    ):

        print(
            f"  {name}: "
            f"{coef:.8f}"
        )


    describe_vector(
        "CoxPH meta TRAIN prediction",
        ph_train,
    )

    describe_vector(
        "CoxPH meta TEST prediction",
        ph_test,
    )


    print(
        f"\nCoxPH sanity TRAIN C-index = "
        f"{cindex(y_dev, ph_train):.6f}"
    )

    print(
        f"CoxPH sanity TEST C-index = "
        f"{cindex(y_test, ph_test):.6f}"
    )


    # =========================================================================
    # SAVE PATIENT-LEVEL DIAGNOSTICS
    # =========================================================================

    dev_output = pd.DataFrame(
        {
            "patient_id":
                development_ids,
            "OS_event":
                survival.loc[
                    development_ids,
                    "OS_event",
                ].values,
            "OS_months":
                survival.loc[
                    development_ids,
                    "OS_months",
                ].values,
        }
    )


    test_output = pd.DataFrame(
        {
            "patient_id":
                test_ids,
            "OS_event":
                survival.loc[
                    test_ids,
                    "OS_event",
                ].values,
            "OS_months":
                survival.loc[
                    test_ids,
                    "OS_months",
                ].values,
        }
    )


    for j, name in enumerate(
        names
    ):

        dev_output[
            f"{name}_base_risk"
        ] = meta_train[
            :,
            j,
        ]

        test_output[
            f"{name}_base_risk"
        ] = meta_test[
            :,
            j,
        ]


    dev_output[
        "coxnet_meta_risk"
    ] = default_train

    test_output[
        "coxnet_meta_risk"
    ] = default_test

    dev_output[
        "coxph_sanity_meta_risk"
    ] = ph_train

    test_output[
        "coxph_sanity_meta_risk"
    ] = ph_test


    dev_output.to_csv(
        DIAG_DIR
        / f"{model_name}_DEVELOPMENT_RISKS.csv",
        index=False,
    )

    test_output.to_csv(
        DIAG_DIR
        / f"{model_name}_TEST_RISKS.csv",
        index=False,
    )


    summary = {

        "model":
            model_name,

        "top_k":
            top_k,

        "base_learner":
            params[
                "learner"
            ],

        "coxnet_default_train_cindex":
            cindex(
                y_dev,
                default_train,
            ),

        "coxnet_default_test_cindex":
            cindex(
                y_test,
                default_test,
            ),

        "coxnet_test_prediction_sd":
            float(
                np.std(
                    default_test
                )
            ),

        "coxnet_test_prediction_range":
            float(
                np.ptp(
                    default_test
                )
            ),

        "coxph_sanity_train_cindex":
            cindex(
                y_dev,
                ph_train,
            ),

        "coxph_sanity_test_cindex":
            cindex(
                y_test,
                ph_test,
            ),

        "coxph_test_prediction_sd":
            float(
                np.std(
                    ph_test
                )
            ),

        "coxph_test_prediction_range":
            float(
                np.ptp(
                    ph_test
                )
            ),
    }


    return summary


# =============================================================================
# RUN M4-M6
# =============================================================================

all_summaries = []

for model_name in [
    "M4",
    "M5",
    "M6",
]:

    summary = diagnose_stack(
        model_name
    )

    all_summaries.append(
        summary
    )


# =============================================================================
# FINAL SUMMARY
# =============================================================================

section(
    "FINAL STACKING DIAGNOSTIC SUMMARY"
)


summary_df = pd.DataFrame(
    all_summaries
)


print(
    summary_df.to_string(
        index=False
    )
)


summary_df.to_csv(
    DIAG_DIR
    / "STACKING_DIAGNOSTIC_SUMMARY.csv",
    index=False,
)


print(
    "\nSaved to:"
)

print(
    DIAG_DIR
)


print(
    "\nCHECK THIS FILE:"
)

print(
    DIAG_DIR
    / "STACKING_DIAGNOSTIC_SUMMARY.csv"
)


print(
    "\nAlso copy/paste the sections showing:"
)

print(
    "1. CoxNet coefficient norms"
)

print(
    "2. CoxNet DEFAULT meta TEST prediction stats"
)

print(
    "3. CoxPH coefficients"
)

print(
    "4. CoxPH meta TEST prediction stats"
)

# %%%%%% CELL 11

### STEP 2A -- EXPRESSION-DEVIATION REPRESENTATIONS + EXACT sPCC CHECK

In [ ]:
"""
STEP 2A -- EXPRESSION-DEVIATION REPRESENTATIONS + EXACT sPCC CHECK
==================================================================

What this script does (no survival information is used anywhere):

  Part 1. Loads the same files your Stage C / CGGA code uses:
            - GTEx reference (334 samples, log-CPM, MT-fixed)
            - TCGA 144 calibrated expression (full network gene space)
            - frozen 90,959-edge candidate universe and 18,192-edge reference
            - saved Stage D M2 / M3 matrices (144 x 3,669), for patient and
              gene order
  Part 2. Shows numerically that each patient-specific sPCC value equals a
          closed-form function of only (z_i, z_j, reference r), where z is the
          tumour's deviation from the GTEx mean in GTEx-SD units.
  Part 3. Measures how much of each patient's retained 18,192-edge network is
          explained by deviation alone (overlap with the top-|z_i * z_j| edges).
  Part 4. Builds three new 144 x 3,669 matrices, in EXACTLY the format of
          STAGE_D_M2_EXPRESSION_RAW.csv, for the representation ladder:
            R1  = |z_g|                     absolute deviation of the gene itself
            R2  = mean_{n in N(g)} |z_n|     mean absolute deviation of its STRING
                                             neighbours (candidate universe)
            R2b = |z_g| * mean |z_n|         gene x neighbourhood deviation
                                             (the quantity that drives |sPCC|)
          NOTE: a plain z-score is NOT included on purpose. z is a per-gene
          linear rescaling of expression, and univariate C-index ranking, RSF,
          GBSA and standardized CoxNet are all invariant to it, so it would
          reproduce M2 exactly.
  Part 5. Gene-wise correlation (115 development patients only) between eNESH
          and expression, R1, R2, R2b -- extends the current Figure 4.

How to run: paste into a new notebook cell (same environment as Stage D) and
run. Then check everything printed below "PART 2".
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

# =============================================================================
# 0. PATHS (same folders your notebooks use)
# =============================================================================
DOWNLOADS = BASE
STAGE_C_REW = DOWNLOADS / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
STAGE_C_CAL = DOWNLOADS / "TCGA_GBM_STAGE_C_144_CALIBRATED"
STAGE_D = DOWNLOADS / "TCGA_GBM_STAGE_D_SURVIVAL"

GTEX_FILE = STAGE_C_REW / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
EDGE_FILE = STAGE_C_REW / "STAGE_C_FINAL_FROZEN_EDGE_UNIVERSE.csv"
REF_FILE = STAGE_C_REW / "STAGE_C_FINAL_REFERENCE_EDGES_20PCT.csv"
SPLIT_FILE = STAGE_D / "STAGE_D_LOCKED_SPLIT.csv"
M2_FILE = STAGE_D / "STAGE_D_M2_EXPRESSION_RAW.csv"
M3_FILE = STAGE_D / "STAGE_D_M3_REWIRING_RAW.csv"

# Full calibrated TCGA matrix (all network genes x 144 patients).
# Leave as None to let the script find it; or paste the exact path here.
TCGA_FULL_FILE = None

OUT_DIR = STAGE_D / "REVISION_STEP2_DEVIATION_LADDER"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_REF = 334
N_KEEP = 18192
SEED = 42


def checkpoint(cond, msg):
    print(f"[CHECKPOINT {'PASS' if cond else 'FAIL'}] {msg}")
    if not cond:
        raise AssertionError(msg)


def strip_version(idx):
    return pd.Index(idx).astype(str).str.split(".").str[0].str.strip()


def section(title):
    print("\n" + "=" * 80 + f"\n{title}\n" + "=" * 80)


# =============================================================================
# PART 1. LOAD
# =============================================================================
section("PART 1 -- LOAD INPUTS")

for f in [GTEX_FILE, EDGE_FILE, REF_FILE, SPLIT_FILE, M2_FILE, M3_FILE]:
    checkpoint(f.exists(), f"exists: {f.name}")

gtex = pd.read_csv(GTEX_FILE, index_col=0, low_memory=False)
gtex.index = strip_version(gtex.index)
checkpoint(gtex.shape[1] == N_REF, f"GTEx has {N_REF} samples (genes x samples)")

edges = pd.read_csv(EDGE_FILE, low_memory=False)
ref = pd.read_csv(REF_FILE, low_memory=False)
for df in (edges, ref):
    for c in ("gene1", "gene2"):
        df[c] = strip_version(df[c])
checkpoint(len(edges) == 90959, "candidate universe = 90,959 edges")
checkpoint(len(ref) == N_KEEP, "reference network = 18,192 edges")

m2 = pd.read_csv(M2_FILE, index_col=0)
m3 = pd.read_csv(M3_FILE, index_col=0)
m2.index, m3.index = m2.index.astype(str), m3.index.astype(str)
m2.columns, m3.columns = strip_version(m2.columns), strip_version(m3.columns)
checkpoint(m2.shape == (144, 3669) and m3.shape == (144, 3669),
           "saved M2/M3 matrices are 144 x 3,669")
checkpoint(m2.index.equals(m3.index) and m2.columns.equals(m3.columns),
           "M2 and M3 share patient and gene order")
patients, genes_3669 = m2.index, m2.columns

split = pd.read_csv(SPLIT_FILE)
split["patient_id"] = split["patient_id"].astype(str)
dev_ids = split.loc[split["split"].str.lower().str.startswith("dev"),
                    "patient_id"].tolist()
checkpoint(len(dev_ids) == 115, "115 development patients")
checkpoint(set(dev_ids).issubset(set(patients)), "dev patients present in M2")


# --- locate the full calibrated TCGA matrix ---------------------------------
def looks_like_full_matrix(path):
    try:
        head = pd.read_csv(path, index_col=0, nrows=5)
    except Exception:
        return None
    first_col = pd.read_csv(path, usecols=[0]).iloc[:, 0].astype(str)
    n_ens_rows = first_col.str.startswith("ENSG").sum()
    if n_ens_rows >= 8000 and head.shape[1] == 144:
        return "genes_x_patients"
    n_ens_cols = pd.Index(head.columns).astype(str).str.startswith("ENSG").sum()
    if n_ens_cols >= 8000 and len(first_col) == 144:
        return "patients_x_genes"
    return None


if TCGA_FULL_FILE is None:
    print("\nSearching for the full calibrated TCGA matrix ...")
    found = []
    for folder in (STAGE_C_CAL, STAGE_C_REW):
        if folder.exists():
            for p in folder.rglob("*.csv"):
                orient = looks_like_full_matrix(p)
                if orient:
                    found.append((p, orient))
                    print(f"  candidate: {p}  ({orient})")
    checkpoint(len(found) >= 1,
               "found a calibrated TCGA matrix with >= 8,000 genes and 144 patients"
               " (if this fails, set TCGA_FULL_FILE at the top)")
    if len(found) > 1:
        print("  More than one candidate; using the first. If wrong, set "
              "TCGA_FULL_FILE explicitly.")
    TCGA_FULL_FILE, orient = found[0]
else:
    TCGA_FULL_FILE = Path(TCGA_FULL_FILE)
    orient = looks_like_full_matrix(TCGA_FULL_FILE)
    checkpoint(orient is not None, "TCGA_FULL_FILE has the expected shape")

tcga = pd.read_csv(TCGA_FULL_FILE, index_col=0, low_memory=False)
if orient == "patients_x_genes":
    tcga = tcga.T
tcga.index = strip_version(tcga.index)
tcga.columns = tcga.columns.astype(str)
print(f"\nUsing TCGA matrix: {TCGA_FULL_FILE}\n  shape (genes x patients) = {tcga.shape}")
checkpoint(set(patients).issubset(set(tcga.columns)),
           "all 144 Stage D patients found in the full TCGA matrix")
tcga = tcga[list(patients)]

# Sanity: the full matrix must reproduce the saved M2 values
common = [g for g in genes_3669 if g in tcga.index]
checkpoint(len(common) == 3669, "all 3,669 modelling genes present in TCGA matrix")
max_diff = np.nanmax(np.abs(tcga.loc[common].T.values - m2[common].values))
print(f"max |full TCGA matrix - saved M2| = {max_diff:.3e}")
checkpoint(max_diff < 1e-4,
           "full TCGA matrix reproduces saved M2 values (same scale as GTEx)"
           " -- if this fails, the file found is not the calibrated matrix used"
           " for Stage C; set TCGA_FULL_FILE explicitly")

# =============================================================================
# Gene space and GTEx statistics (population SD, exactly as in your sPCC code)
# =============================================================================
net_genes = pd.Index(sorted(set(edges["gene1"]) | set(edges["gene2"])))
checkpoint(net_genes.isin(gtex.index).all(), "all network genes in GTEx")
checkpoint(net_genes.isin(tcga.index).all(), "all network genes in TCGA matrix")

G = gtex.loc[net_genes].to_numpy(np.float64)          # genes x 334
T = tcga.loc[net_genes].to_numpy(np.float64)          # genes x 144
mu = G.mean(axis=1)
SS = ((G - mu[:, None]) ** 2).sum(axis=1)             # centred sum of squares
sd_pop = np.sqrt(SS / N_REF)
checkpoint(np.all(sd_pop > 0), "all network genes have non-zero GTEx variance")
Z = (T - mu[:, None]) / sd_pop[:, None]               # genes x 144

gpos = pd.Series(np.arange(len(net_genes)), index=net_genes)
ei = gpos[edges["gene1"]].to_numpy()
ej = gpos[edges["gene2"]].to_numpy()

# reference correlation, recomputed from GTEx (and compared with the file if present)
Gc = G - mu[:, None]
r_ref = np.einsum("ij,ij->i", Gc[ei], Gc[ej]) / np.sqrt(SS[ei] * SS[ej])
if "reference_PCC" in edges.columns:
    d = np.max(np.abs(r_ref - edges["reference_PCC"].to_numpy(float)))
    print(f"max |recomputed r_ref - frozen reference_PCC| = {d:.3e}")
    checkpoint(d < 1e-6, "recomputed reference correlations match frozen file")

# =============================================================================
# PART 2. EXACT CLOSED FORM
# =============================================================================
section("PART 2 -- sPCC IS AN EXACT FUNCTION OF (z_i, z_j, r_ref)")

N = float(N_REF)
a = N / (N + 1.0)


def spcc_augmented_sums(x_col):
    """Your implementation (validation.ipynb, step 4A), for one patient."""
    dx, dy = x_col[ei] - mu[ei], x_col[ej] - mu[ej]
    sxx = SS[ei] + a * dx * dx
    syy = SS[ej] + a * dy * dy
    sxy = r_ref * np.sqrt(SS[ei] * SS[ej]) + a * dx * dy
    r_aug = np.clip(sxy / np.sqrt(sxx * syy), -1, 1)
    return (N + 1.0) * r_aug - N * r_ref


def spcc_closed_form(z_col):
    """Same quantity written only with z_i, z_j and r_ref."""
    zi, zj = z_col[ei], z_col[ej]
    r_aug = (N * r_ref + a * zi * zj) / np.sqrt((N + a * zi ** 2) * (N + a * zj ** 2))
    return (N + 1.0) * np.clip(r_aug, -1, 1) - N * r_ref


max_err = 0.0
for p in range(T.shape[1]):
    s1 = spcc_augmented_sums(T[:, p])
    s2 = spcc_closed_form(Z[:, p])
    max_err = max(max_err, float(np.max(np.abs(s1 - s2))))
print(f"max |sPCC(implementation) - sPCC(closed form)| over 144 patients x "
      f"90,959 edges = {max_err:.3e}")
checkpoint(max_err < 1e-8, "sPCC depends only on (z_i, z_j, r_ref)")

# =============================================================================
# PART 3. HOW MUCH OF EACH PATIENT NETWORK IS DEVIATION ALONE?
# =============================================================================
section("PART 3 -- RETAINED EDGES vs TOP |z_i * z_j| EDGES")

overlap = []
for p in range(T.shape[1]):
    s = spcc_augmented_sums(T[:, p])
    top_spcc = set(np.argsort(-np.abs(s), kind="mergesort")[:N_KEEP])
    prod = np.abs(Z[ei, p] * Z[ej, p])
    top_prod = set(np.argsort(-prod, kind="mergesort")[:N_KEEP])
    overlap.append(len(top_spcc & top_prod) / N_KEEP)
overlap = np.array(overlap)
print(f"Per-patient overlap between retained edges and top-|z_i z_j| edges:\n"
      f"  median = {np.median(overlap):.3f}, IQR = {np.percentile(overlap, 25):.3f}"
      f"-{np.percentile(overlap, 75):.3f}, min = {overlap.min():.3f}, "
      f"max = {overlap.max():.3f}")
print(f"  (random expectation = {N_KEEP / len(ei):.3f})")
pd.DataFrame({"patient_id": patients, "edge_overlap_top_abs_zz": overlap}).to_csv(
    OUT_DIR / "STEP2A_PATIENT_EDGE_OVERLAP.csv", index=False)

# =============================================================================
# PART 4. BUILD R1, R2, R2b  (144 x 3,669, same format as M2)
# =============================================================================
section("PART 4 -- BUILD DEVIATION REPRESENTATIONS")

absZ = np.abs(Z)                                       # genes x 144
nbrs = {g: [] for g in range(len(net_genes))}
for i, j in zip(ei, ej):
    nbrs[i].append(j)
    nbrs[j].append(i)

anchor_pos = gpos[genes_3669].to_numpy()
n_nbrs = np.array([len(nbrs[g]) for g in anchor_pos])
checkpoint(np.all(n_nbrs > 0), "every modelling anchor has >= 1 candidate neighbour")
print(f"neighbours per anchor: median {np.median(n_nbrs):.0f}, "
      f"range {n_nbrs.min()}-{n_nbrs.max()}")

R1 = absZ[anchor_pos].T                                                  # 144 x 3669
R2 = np.vstack([absZ[nbrs[g]].mean(axis=0) for g in anchor_pos]).T       # 144 x 3669
R2b = R1 * R2

ladder = {
    "R1_ABS_DEVIATION": R1,
    "R2_NEIGHBOUR_ABS_DEVIATION": R2,
    "R2B_GENE_X_NEIGHBOUR_DEVIATION": R2b,
}
for name, M in ladder.items():
    df = pd.DataFrame(M, index=patients, columns=genes_3669)
    df.index.name = m2.index.name
    checkpoint(np.isfinite(df.values).all(), f"{name}: all values finite")
    dev_const = int((df.loc[dev_ids].std(axis=0) <= 1e-12).sum())
    print(f"{name}: shape {df.shape}, constant in development = {dev_const}")
    path = OUT_DIR / f"STAGE_D_{name}_RAW.csv"
    df.to_csv(path)
    print(f"  saved -> {path}")

# =============================================================================
# PART 5. GENE-WISE CORRELATION WITH eNESH (development patients only)
# =============================================================================
section("PART 5 -- GENE-WISE CORRELATION OF eNESH WITH EACH REPRESENTATION")

e_dev = m3.loc[dev_ids]
compare = {"expression (M2)": m2.loc[dev_ids]}
for name, M in ladder.items():
    compare[name] = pd.DataFrame(M, index=patients, columns=genes_3669).loc[dev_ids]

rows = []
for label, X in compare.items():
    rs = []
    for g in genes_3669:
        x, y = X[g].to_numpy(float), e_dev[g].to_numpy(float)
        if np.std(x) > 1e-12 and np.std(y) > 1e-12:
            rs.append(pearsonr(x, y).statistic)
    rs = np.abs(np.array(rs))
    rows.append({"representation": label, "genes": len(rs),
                 "median_abs_r": np.median(rs),
                 "pct_abs_r_ge_0.5": 100 * np.mean(rs >= 0.5),
                 "pct_abs_r_lt_0.3": 100 * np.mean(rs < 0.3)})
corr = pd.DataFrame(rows)
print(corr.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
corr.to_csv(OUT_DIR / "STEP2A_ENESH_CORRELATION_BY_REPRESENTATION.csv", index=False)

section("DONE")
print("Check everything printed from PART 2 onwards.")
print("Next (Step 2B): run your Stage D script on each new matrix (see message).")

### STAGE D — PART 3

Deviation baselines: run this cell three times, with `REPRESENTATION = "R1"`, `"R2"` and `"R2B"`.

In [ ]:
"""
STAGE D — PART 3
V4-OFF-CORRECTED CONTROLLED EXPERIMENT
======================================

PURPOSE
-------
This script preserves the original V4 modeling design as closely as possible:

- same 115 development / 29 internal-test split
- same RANDOM_SEED
- same repeated 3 x 5 outer CV
- same 3-fold inner CV
- same TOP_K_VALUES = [10, 25, 50]
- same RSF / CoxNet / GBSA learner grid
- same univariate train-fold molecular ranking
- same M1-M6 definitions
- same final RSF 5-seed ensemble

ONLY IMPORTANT SCIENTIFIC CORRECTION:
-------------------------------------
The original V4 stacking trained the meta-model using base-model risk
predictions generated on the SAME patients used to fit the base model.

This version replaces those in-sample training risks with internal
out-of-fold (OOF) risks.

For strict leakage control, molecular feature selection is ALSO repeated
inside each OOF meta-training fold.

The 29-patient internal test must NOT be used to choose between versions.
Primary comparison = repeated nested development outer-CV.
"""

# =============================================================================
# STEP 2B -- RUN ONE BASELINE THROUGH THE IDENTICAL STAGE D PIPELINE
# Change ONLY the next line, run, then repeat for the other two choices:
#     "R1"   -> absolute deviation of each gene from healthy brain
#     "R2"   -> average absolute deviation of its network neighbours
#     "R2B"  -> gene deviation x neighbour deviation
REPRESENTATION = "R1"
# =============================================================================


from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sksurv.ensemble import (
    RandomSurvivalForest,
    GradientBoostingSurvivalAnalysis,
)
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored


warnings.filterwarnings("ignore")


# =============================================================================
# CONFIGURATION
# =============================================================================

DOWNLOADS = BASE
STAGE_D_DIR = DOWNLOADS / "TCGA_GBM_STAGE_D_SURVIVAL"

SPLIT_FILE = STAGE_D_DIR / "STAGE_D_LOCKED_SPLIT.csv"
COHORT_FILE = STAGE_D_DIR / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
M1_FILE = STAGE_D_DIR / "STAGE_D_M1_CLINICAL_RAW.csv"
M2_FILE = STAGE_D_DIR / "STAGE_D_M2_EXPRESSION_RAW.csv"
LADDER_DIR = STAGE_D_DIR / "REVISION_STEP2_DEVIATION_LADDER"
LADDER_FILES = {
    "R1": LADDER_DIR / "STAGE_D_R1_ABS_DEVIATION_RAW.csv",
    "R2": LADDER_DIR / "STAGE_D_R2_NEIGHBOUR_ABS_DEVIATION_RAW.csv",
    "R2B": LADDER_DIR / "STAGE_D_R2B_GENE_X_NEIGHBOUR_DEVIATION_RAW.csv",
}
# The baseline is fed through the standalone-molecular ("M3") path,
# so it gets exactly the same nested CV, Top-K grid and learners as M2/M3.
M3_FILE = LADDER_FILES[REPRESENTATION]

# IMPORTANT:
# Do NOT overwrite original V4 / V3 results.
# Separate output folder: original V4 results are never overwritten.
RESULT_DIR = STAGE_D_DIR / f"REVISION_STEP2B_{REPRESENTATION}"
RESULT_DIR.mkdir(parents=True, exist_ok=True)


RANDOM_SEED = 42

OUTER_FOLDS = 5
INNER_FOLDS = 3
N_OUTER_REPEATS = 3

# internal folds used ONLY to generate OOF stacking risks
META_OOF_FOLDS = 3

N_FINAL_SEEDS = 5

# Preserve original V4 Top-K grid
TOP_K_VALUES = [10, 25, 50]


MODELS = ["M3"]   # only the standalone molecular model is needed here

MODEL_DESCRIPTIONS = {
    "M1": "Clinical",
    "M2": "Expression",
    "M3": "Rewiring",
    "M4": "Clinical + Expression (OOF stacked)",
    "M5": "Clinical + Rewiring (OOF stacked)",
    "M6": "Clinical + Expression + Rewiring (OOF triple stacked)",
}


# Preserve original V4 learner grid exactly
LEARNER_GRID = [
    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },
    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
    },
    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 5,
        "max_features": 0.30,
    },
    {
        "learner": "rsf",
        "n_estimators": 300,
        "min_samples_leaf": 10,
        "max_features": 0.30,
    },
    {
        "learner": "coxnet",
        "l1_ratio": 0.5,
        "alpha_min_ratio": 0.01,
    },
    {
        "learner": "coxnet",
        "l1_ratio": 0.9,
        "alpha_min_ratio": 0.01,
    },
    {
        "learner": "coxnet",
        "l1_ratio": 0.5,
        "alpha_min_ratio": 0.1,
    },
    {
        "learner": "gbsa",
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 2,
        "subsample": 0.8,
    },
    {
        "learner": "gbsa",
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_depth": 3,
        "subsample": 0.8,
    },
]


# =============================================================================
# UTILITIES
# =============================================================================

def section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def checkpoint(cond, msg):
    print(f"[CHECKPOINT {'PASS' if cond else 'FAIL'}] {msg}")
    if not cond:
        raise RuntimeError(f"Checkpoint failed: {msg}")


def make_survival_y(df):
    events = pd.to_numeric(df["OS_event"], errors="raise").astype(int)
    times = pd.to_numeric(df["OS_months"], errors="raise").astype(float)

    if not events.isin([0, 1]).all():
        raise ValueError("OS_event must contain only 0/1.")

    if (times <= 0).any():
        raise ValueError("OS_months must be > 0.")

    return np.array(
        [(bool(e), float(t)) for e, t in zip(events, times)],
        dtype=[("event", "?"), ("time", "<f8")],
    )


def cindex(y, risk):
    risk = np.asarray(risk, dtype=float)

    if not np.isfinite(risk).all():
        return np.nan

    return float(
        concordance_index_censored(
            y["event"],
            y["time"],
            risk,
        )[0]
    )


# =============================================================================
# BASE LEARNERS
# =============================================================================

def fit_predict(params, X_train, y_train, X_valid):

    if params["learner"] == "rsf":

        model = RandomSurvivalForest(
            n_estimators=params["n_estimators"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            min_samples_split=10,
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)
        return model.predict(X_valid)

    elif params["learner"] == "coxnet":

        # Preserved exactly from V4 for controlled comparison.
        model = CoxnetSurvivalAnalysis(
            l1_ratio=params["l1_ratio"],
            alpha_min_ratio=params["alpha_min_ratio"],
        )

        model.fit(X_train, y_train)
        return model.predict(X_valid)

    elif params["learner"] == "gbsa":

        model = GradientBoostingSurvivalAnalysis(
            n_estimators=params["n_estimators"],
            learning_rate=params["learning_rate"],
            max_depth=params["max_depth"],
            subsample=params["subsample"],
            random_state=RANDOM_SEED,
        )

        model.fit(X_train, y_train)
        return model.predict(X_valid)

    else:
        raise ValueError(params["learner"])


def fit_predict_seed_ensemble(
    params,
    X_train,
    y_train,
    X_valid,
    n_seeds=N_FINAL_SEEDS,
):

    if params["learner"] != "rsf":
        return fit_predict(
            params,
            X_train,
            y_train,
            X_valid,
        )

    preds = []

    for seed_offset in range(n_seeds):

        model = RandomSurvivalForest(
            n_estimators=params["n_estimators"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            min_samples_split=10,
            random_state=RANDOM_SEED + seed_offset,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)
        preds.append(model.predict(X_valid))

    return np.mean(preds, axis=0)


# =============================================================================
# CLINICAL PREPROCESSING
# =============================================================================

def make_onehot():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def build_clinical_preprocessor(clinical_df):

    numeric_cols = [
        c
        for c in [
            "Diagnosis Age",
            "Karnofsky Performance Score",
        ]
        if c in clinical_df.columns
    ]

    categorical_cols = [
        c
        for c in [
            "Sex",
            "MGMT Status",
        ]
        if c in clinical_df.columns
    ]

    transformers = []

    if numeric_cols:
        transformers.append(
            (
                "numeric",
                SimpleImputer(strategy="median"),
                numeric_cols,
            )
        )

    if categorical_cols:

        cat_pipe = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "onehot",
                    make_onehot(),
                ),
            ]
        )

        transformers.append(
            (
                "categorical",
                cat_pipe,
                categorical_cols,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


# =============================================================================
# MOLECULAR FEATURE RANKING
# =============================================================================

def rank_molecular_features(X_train, y_train):

    feature_names = list(X_train.columns)

    values = X_train.to_numpy(
        dtype=np.float64
    )

    scores = np.full(
        len(feature_names),
        0.5,
        dtype=float,
    )

    for j in range(values.shape[1]):

        x = values[:, j]

        if np.nanstd(x) <= 1e-12:
            continue

        try:

            c = cindex(
                y_train,
                x,
            )

            scores[j] = (
                0.5
                if np.isnan(c)
                else max(c, 1.0 - c)
            )

        except Exception:
            pass

    ranking = pd.DataFrame(
        {
            "feature": feature_names,
            "univariate_c": scores,
        }
    )

    return ranking.sort_values(
        ["univariate_c", "feature"],
        ascending=[False, True],
    ).reset_index(drop=True)


_RANKING_CACHE = {}


def rank_molecular_features_cached(
    molecular,
    train_ids,
    y_train,
):

    key = (
        id(molecular),
        tuple(sorted(train_ids)),
    )

    cached = _RANKING_CACHE.get(key)

    if cached is None:

        cached = rank_molecular_features(
            molecular.loc[train_ids],
            y_train,
        )

        _RANKING_CACHE[key] = cached

    return cached


def select_molecular(
    molecular,
    train_ids,
    y_train,
    top_k,
):

    ranking = rank_molecular_features_cached(
        molecular,
        train_ids,
        y_train,
    )

    return (
        ranking["feature"]
        .head(top_k)
        .tolist()
    )


# =============================================================================
# BASIC BLOCK PREPARATION
# =============================================================================

def prepare_clinical_block(
    clinical,
    train_ids,
    valid_ids,
):

    clinical_train = clinical.loc[train_ids]
    clinical_valid = clinical.loc[valid_ids]

    processor = build_clinical_preprocessor(
        clinical_train
    )

    X_train = processor.fit_transform(
        clinical_train
    )

    X_valid = processor.transform(
        clinical_valid
    )

    return X_train, X_valid


def prepare_molecular_block(
    molecular,
    selected_cols,
    train_ids,
    valid_ids,
):

    X_train = molecular.loc[
        train_ids,
        selected_cols,
    ].to_numpy(
        dtype=np.float64
    )

    X_valid = molecular.loc[
        valid_ids,
        selected_cols,
    ].to_numpy(
        dtype=np.float64
    )

    return X_train, X_valid


# =============================================================================
# OOF STACKING
# =============================================================================

def generate_oof_base_risks(
    block_type,
    clinical,
    molecular,
    train_ids,
    y_train,
    survival,
    top_k,
    learner_params,
):

    """
    Generate leakage-safe OOF base-model risk predictions.

    block_type:
        "clinical"
        "expression"
        "rewiring"

    IMPORTANT:
    For molecular blocks, feature selection is repeated INSIDE each
    meta-OOF training fold.
    """

    train_ids = list(train_ids)

    train_events = (
        survival.loc[
            train_ids,
            "OS_event",
        ]
        .astype(int)
        .to_numpy()
    )

    meta_cv = StratifiedKFold(
        n_splits=META_OOF_FOLDS,
        shuffle=True,
        random_state=RANDOM_SEED + 777,
    )

    oof_risk = np.full(
        len(train_ids),
        np.nan,
        dtype=float,
    )

    for (
        meta_train_idx,
        meta_valid_idx,
    ) in meta_cv.split(
        train_ids,
        train_events,
    ):

        meta_train_ids = [
            train_ids[i]
            for i in meta_train_idx
        ]

        meta_valid_ids = [
            train_ids[i]
            for i in meta_valid_idx
        ]

        y_meta_train = make_survival_y(
            survival.loc[
                meta_train_ids
            ]
        )

        if block_type == "clinical":

            X_meta_train, X_meta_valid = (
                prepare_clinical_block(
                    clinical,
                    meta_train_ids,
                    meta_valid_ids,
                )
            )

        else:

            if block_type == "expression":
                block_df = molecular

            elif block_type == "rewiring":
                block_df = molecular

            else:
                raise ValueError(block_type)

            selected = select_molecular(
                block_df,
                meta_train_ids,
                y_meta_train,
                top_k,
            )

            (
                X_meta_train,
                X_meta_valid,
            ) = prepare_molecular_block(
                block_df,
                selected,
                meta_train_ids,
                meta_valid_ids,
            )

        preds = fit_predict(
            learner_params,
            X_meta_train,
            y_meta_train,
            X_meta_valid,
        )

        oof_risk[
            meta_valid_idx
        ] = preds

    if not np.isfinite(
        oof_risk
    ).all():

        raise RuntimeError(
            "Non-finite OOF risk generated."
        )

    return oof_risk


def fit_full_block_predict(
    block_type,
    clinical,
    molecular,
    train_ids,
    valid_ids,
    y_train,
    top_k,
    learner_params,
    use_seed_ensemble=False,
):

    """
    Fit one base block on ALL current training patients
    and predict the held-out current validation patients.

    Feature selection is performed using current training patients only.
    """

    predict_fn = (
        fit_predict_seed_ensemble
        if use_seed_ensemble
        else fit_predict
    )

    selected = ["clinical"]

    if block_type == "clinical":

        X_train, X_valid = (
            prepare_clinical_block(
                clinical,
                train_ids,
                valid_ids,
            )
        )

    else:

        selected = select_molecular(
            molecular,
            train_ids,
            y_train,
            top_k,
        )

        X_train, X_valid = (
            prepare_molecular_block(
                molecular,
                selected,
                train_ids,
                valid_ids,
            )
        )

    valid_risk = predict_fn(
        learner_params,
        X_train,
        y_train,
        X_valid,
    )

    return valid_risk, selected


def fit_oof_stacked_and_predict(
    model_name,
    clinical,
    expression,
    rewiring,
    survival,
    train_ids,
    valid_ids,
    y_train,
    top_k,
    learner_params,
    use_seed_ensemble=False,
):

    """
    Leakage-safe stacked prediction.

    Meta-model training features:
        OOF risks generated inside current training cohort.

    Meta-model validation features:
        risks from base models fitted on the entire current training cohort.

    No current validation outcome enters:
        feature selection
        base learner training
        OOF risk generation
        meta-model fitting
    """

    if model_name == "M4":

        block_specs = [
            (
                "clinical",
                None,
            ),
            (
                "expression",
                expression,
            ),
        ]

    elif model_name == "M5":

        block_specs = [
            (
                "clinical",
                None,
            ),
            (
                "rewiring",
                rewiring,
            ),
        ]

    elif model_name == "M6":

        block_specs = [
            (
                "clinical",
                None,
            ),
            (
                "expression",
                expression,
            ),
            (
                "rewiring",
                rewiring,
            ),
        ]

    else:
        raise ValueError(
            f"Not a stacked model: {model_name}"
        )

    train_risks = []
    valid_risks = []
    all_selected = []

    for (
        block_type,
        molecular_df,
    ) in block_specs:

        # -------------------------------------------------------------
        # OOF training risks for meta-model
        # -------------------------------------------------------------

        oof_risk = generate_oof_base_risks(
            block_type=block_type,
            clinical=clinical,
            molecular=molecular_df,
            train_ids=train_ids,
            y_train=y_train,
            survival=survival,
            top_k=top_k,
            learner_params=learner_params,
        )

        train_risks.append(
            oof_risk
        )

        # -------------------------------------------------------------
        # Full-training fit -> current held-out validation risk
        # -------------------------------------------------------------

        valid_risk, selected = (
            fit_full_block_predict(
                block_type=block_type,
                clinical=clinical,
                molecular=molecular_df,
                train_ids=train_ids,
                valid_ids=valid_ids,
                y_train=y_train,
                top_k=top_k,
                learner_params=learner_params,
                use_seed_ensemble=use_seed_ensemble,
            )
        )

        valid_risks.append(
            valid_risk
        )

        if block_type != "clinical":

            prefix = (
                "EXPR::"
                if block_type == "expression"
                else "REWIRE::"
            )

            all_selected.extend(
                [
                    prefix + f
                    for f in selected
                ]
            )

    meta_X_train = np.column_stack(
        train_risks
    )

    meta_X_valid = np.column_stack(
        valid_risks
    )

    # Preserve original V4 meta-model
    meta_model = CoxnetSurvivalAnalysis(
        l1_ratio=0.01,
        alpha_min_ratio=0.5,
    )

    meta_model.fit(
        meta_X_train,
        y_train,
    )

    meta_risk = meta_model.predict(
        meta_X_valid
    )

    return meta_risk, all_selected


# =============================================================================
# MODEL EVALUATION
# =============================================================================

def evaluate_model(
    model_name,
    train_ids,
    valid_ids,
    clinical,
    expression,
    rewiring,
    survival,
    top_k,
    learner_params,
    use_seed_ensemble=False,
):

    predict_fn = (
        fit_predict_seed_ensemble
        if use_seed_ensemble
        else fit_predict
    )

    y_train = make_survival_y(
        survival.loc[
            train_ids
        ]
    )

    # -------------------------------------------------------------------------
    # M1 CLINICAL
    # -------------------------------------------------------------------------

    if model_name == "M1":

        X_train, X_valid = (
            prepare_clinical_block(
                clinical,
                train_ids,
                valid_ids,
            )
        )

        risk = predict_fn(
            learner_params,
            X_train,
            y_train,
            X_valid,
        )

        return risk, ["clinical"]

    # -------------------------------------------------------------------------
    # STACKED M4 / M5 / M6
    # -------------------------------------------------------------------------

    if model_name in (
        "M4",
        "M5",
        "M6",
    ):

        return fit_oof_stacked_and_predict(
            model_name=model_name,
            clinical=clinical,
            expression=expression,
            rewiring=rewiring,
            survival=survival,
            train_ids=train_ids,
            valid_ids=valid_ids,
            y_train=y_train,
            top_k=top_k,
            learner_params=learner_params,
            use_seed_ensemble=use_seed_ensemble,
        )

    # -------------------------------------------------------------------------
    # STANDALONE MOLECULAR M2 / M3
    # -------------------------------------------------------------------------

    molecular = {
        "M2": expression,
        "M3": rewiring,
    }[model_name]

    selected = select_molecular(
        molecular,
        train_ids,
        y_train,
        top_k,
    )

    X_train, X_valid = (
        prepare_molecular_block(
            molecular,
            selected,
            train_ids,
            valid_ids,
        )
    )

    risk = predict_fn(
        learner_params,
        X_train,
        y_train,
        X_valid,
    )

    return risk, selected


# =============================================================================
# STEP 1 — LOAD DATA
# =============================================================================

section(
    "STEP 1 -- LOAD STAGE-D LOCKED DATA"
)

for path in [
    SPLIT_FILE,
    COHORT_FILE,
    M1_FILE,
    M2_FILE,
    M3_FILE,
]:
    checkpoint(
        path.exists(),
        f"{path.name} exists",
    )


split_df = pd.read_csv(
    SPLIT_FILE
)

survival = pd.read_csv(
    COHORT_FILE
).set_index(
    "patient_id"
)

clinical = pd.read_csv(
    M1_FILE,
    index_col=0,
)

expression = pd.read_csv(
    M2_FILE,
    index_col=0,
)

rewiring = pd.read_csv(
    M3_FILE,
    index_col=0,
)


development_ids = (
    split_df.loc[
        split_df["split"]
        == "development",
        "patient_id",
    ]
    .tolist()
)

test_ids = (
    split_df.loc[
        split_df["split"]
        == "locked_test",
        "patient_id",
    ]
    .tolist()
)


checkpoint(
    len(development_ids) == 115,
    "115 development patients",
)

checkpoint(
    len(test_ids) == 29,
    "29 internal-test patients",
)

checkpoint(
    not (
        set(development_ids)
        & set(test_ids)
    ),
    "development/test remain disjoint",
)


for name, df in [
    ("survival", survival),
    ("clinical", clinical),
    ("expression", expression),
    ("rewiring", rewiring),
]:

    checkpoint(
        df.index.is_unique,
        f"{name} patient IDs unique",
    )


all_ids = set(
    development_ids
    + test_ids
)

checkpoint(
    all_ids.issubset(
        survival.index
    ),
    "all split IDs present in survival",
)

checkpoint(
    all_ids.issubset(
        clinical.index
    ),
    "all split IDs present in clinical",
)

checkpoint(
    all_ids.issubset(
        expression.index
    ),
    "all split IDs present in expression",
)

checkpoint(
    all_ids.issubset(
        rewiring.index
    ),
    "all split IDs present in rewiring",
)


print(
    "\nModels:",
    [
        (
            m,
            MODEL_DESCRIPTIONS[m],
        )
        for m in MODELS
    ],
)

print(
    f"Base learner candidates: "
    f"{len(LEARNER_GRID)}"
)

print(
    f"Top-K values: "
    f"{TOP_K_VALUES}"
)

print(
    f"Outer CV: "
    f"{N_OUTER_REPEATS} x "
    f"{OUTER_FOLDS}"
)

print(
    f"Inner CV: "
    f"{INNER_FOLDS}"
)

print(
    f"Meta OOF folds: "
    f"{META_OOF_FOLDS}"
)


# =============================================================================
# STEP 2 — REPEATED NESTED DEVELOPMENT CV
# =============================================================================

section(
    "STEP 2 -- REPEATED NESTED DEVELOPMENT-ONLY CV"
)


development_survival = (
    survival.loc[
        development_ids
    ]
)

development_events = (
    development_survival[
        "OS_event"
    ]
    .astype(int)
    .to_numpy()
)


outer_results = []
outer_selected_features = []

pipeline_start = time.time()


for repeat in range(
    1,
    N_OUTER_REPEATS + 1,
):

    outer_cv = StratifiedKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_SEED
            + repeat * 100
        ),
    )

    print(
        "\n"
        + "#" * 70
    )

    print(
        f"REPEAT "
        f"{repeat}/"
        f"{N_OUTER_REPEATS}"
    )

    print(
        "#" * 70
    )


    for (
        outer_fold,
        (
            outer_train_idx,
            outer_valid_idx,
        ),
    ) in enumerate(
        outer_cv.split(
            development_ids,
            development_events,
        ),
        start=1,
    ):

        fold_start = time.time()

        print(
            "\n"
            + "=" * 70
        )

        print(
            f"REPEAT {repeat} "
            f"-- OUTER FOLD "
            f"{outer_fold}/"
            f"{OUTER_FOLDS}"
        )

        print(
            "=" * 70
        )


        outer_train_ids = [
            development_ids[i]
            for i
            in outer_train_idx
        ]

        outer_valid_ids = [
            development_ids[i]
            for i
            in outer_valid_idx
        ]

        y_outer_valid = make_survival_y(
            survival.loc[
                outer_valid_ids
            ]
        )

        inner_events = (
            survival.loc[
                outer_train_ids,
                "OS_event",
            ]
            .astype(int)
            .to_numpy()
        )


        for model_name in MODELS:

            print(
                f"\n  MODEL "
                f"{model_name} "
                f"("
                f"{MODEL_DESCRIPTIONS[model_name]}"
                f")"
            )


            candidate_k_values = (
                [None]
                if model_name == "M1"
                else TOP_K_VALUES
            )

            candidate_results = []


            for top_k in candidate_k_values:

                for params in LEARNER_GRID:

                    inner_cv = StratifiedKFold(
                        n_splits=INNER_FOLDS,
                        shuffle=True,
                        random_state=(
                            RANDOM_SEED
                            + repeat * 100
                            + outer_fold
                        ),
                    )

                    fold_scores = []


                    for (
                        inner_train_idx,
                        inner_valid_idx,
                    ) in inner_cv.split(
                        outer_train_ids,
                        inner_events,
                    ):

                        inner_train_ids = [
                            outer_train_ids[i]
                            for i
                            in inner_train_idx
                        ]

                        inner_valid_ids = [
                            outer_train_ids[i]
                            for i
                            in inner_valid_idx
                        ]

                        y_inner_valid = (
                            make_survival_y(
                                survival.loc[
                                    inner_valid_ids
                                ]
                            )
                        )

                        try:

                            risk, _ = evaluate_model(
                                model_name,
                                inner_train_ids,
                                inner_valid_ids,
                                clinical,
                                expression,
                                rewiring,
                                survival,
                                top_k,
                                params,
                                use_seed_ensemble=False,
                            )

                            score = cindex(
                                y_inner_valid,
                                risk,
                            )

                        except Exception as exc:

                            print(
                                "      "
                                "[candidate failure] "
                                f"{model_name}, "
                                f"k={top_k}, "
                                f"{params['learner']}: "
                                f"{type(exc).__name__}: "
                                f"{exc}"
                            )

                            score = np.nan


                        fold_scores.append(
                            score
                        )


                    valid_fold_scores = [
                        s
                        for s in fold_scores
                        if np.isfinite(s)
                    ]

                    mean_inner = (
                        float(
                            np.mean(
                                valid_fold_scores
                            )
                        )
                        if valid_fold_scores
                        else np.nan
                    )


                    candidate_results.append(
                        {
                            "top_k": top_k,
                            "params": params,
                            "mean_inner_c": mean_inner,
                        }
                    )


            valid_candidates = [
                x
                for x in candidate_results
                if np.isfinite(
                    x["mean_inner_c"]
                )
            ]

            if not valid_candidates:

                raise RuntimeError(
                    f"No valid candidates for "
                    f"{model_name} "
                    f"repeat={repeat} "
                    f"outer_fold={outer_fold}"
                )


            best = max(
                valid_candidates,
                key=lambda x:
                    x["mean_inner_c"],
            )


            print(
                f"    Best inner C-index: "
                f"{best['mean_inner_c']:.4f}"
            )

            print(
                f"    Learner: "
                f"{best['params']['learner']}"
            )

            print(
                f"    Top-K: "
                f"{best['top_k']}"
            )


            risk, selected = (
                evaluate_model(
                    model_name,
                    outer_train_ids,
                    outer_valid_ids,
                    clinical,
                    expression,
                    rewiring,
                    survival,
                    best["top_k"],
                    best["params"],
                    use_seed_ensemble=False,
                )
            )


            outer_c = cindex(
                y_outer_valid,
                risk,
            )


            print(
                f"    OUTER C-index: "
                f"{outer_c:.4f}"
            )


            outer_results.append(
                {
                    "repeat": repeat,
                    "outer_fold": outer_fold,
                    "model": model_name,
                    "description":
                        MODEL_DESCRIPTIONS[
                            model_name
                        ],
                    "top_k":
                        best["top_k"],
                    "learner":
                        best["params"][
                            "learner"
                        ],
                    "inner_cindex":
                        best[
                            "mean_inner_c"
                        ],
                    "outer_cindex":
                        outer_c,
                    "n_selected_molecular":
                        (
                            0
                            if model_name == "M1"
                            else len(selected)
                        ),
                }
            )


            if model_name != "M1":

                for (
                    rank,
                    feature,
                ) in enumerate(
                    selected,
                    start=1,
                ):

                    outer_selected_features.append(
                        {
                            "repeat":
                                repeat,
                            "outer_fold":
                                outer_fold,
                            "model":
                                model_name,
                            "rank":
                                rank,
                            "feature":
                                feature,
                        }
                    )


        pd.DataFrame(
            outer_results
        ).to_csv(
            RESULT_DIR
            / "STAGE_D_V4_OOF_OUTER_CV_RESULTS_partial.csv",
            index=False,
        )


        elapsed = (
            time.time()
            - fold_start
        )

        total_elapsed = (
            time.time()
            - pipeline_start
        )


        print(
            f"\n  [TIMING] "
            f"fold took "
            f"{elapsed / 60:.1f} min"
        )

        print(
            f"  Total elapsed: "
            f"{total_elapsed / 60:.1f} min"
        )

        print(
            f"  Ranking cache size: "
            f"{len(_RANKING_CACHE)}"
        )


# =============================================================================
# STEP 3 — DEVELOPMENT SUMMARY
# =============================================================================

section(
    "STEP 3 -- DEVELOPMENT CV SUMMARY"
)


outer_results_df = pd.DataFrame(
    outer_results
)


development_summary = (
    outer_results_df
    .groupby(
        "model"
    )
    .agg(
        mean_outer_cindex=(
            "outer_cindex",
            "mean",
        ),
        sd_outer_cindex=(
            "outer_cindex",
            "std",
        ),
        median_outer_cindex=(
            "outer_cindex",
            "median",
        ),
        min_outer_cindex=(
            "outer_cindex",
            "min",
        ),
        max_outer_cindex=(
            "outer_cindex",
            "max",
        ),
        n_folds=(
            "outer_cindex",
            "count",
        ),
    )
    .reset_index()
)


development_summary[
    "description"
] = development_summary[
    "model"
].map(
    MODEL_DESCRIPTIONS
)


print(
    "\nDEVELOPMENT PERFORMANCE:"
)

print(
    development_summary.to_string(
        index=False
    )
)


outer_results_df.to_csv(
    RESULT_DIR
    / "STAGE_D_V4_OOF_OUTER_CV_RESULTS.csv",
    index=False,
)

development_summary.to_csv(
    RESULT_DIR
    / "STAGE_D_V4_OOF_DEVELOPMENT_SUMMARY.csv",
    index=False,
)

pd.DataFrame(
    outer_selected_features
).to_csv(
    RESULT_DIR
    / "STAGE_D_V4_OOF_OUTER_SELECTED_FEATURES.csv",
    index=False,
)


# =============================================================================
# STEP 4 — FINAL TUNING ON ALL DEVELOPMENT PATIENTS
# =============================================================================

section(
    "STEP 4 -- FINAL TUNING ON ALL 115 DEVELOPMENT PATIENTS"
)


final_specs = {}


for model_name in MODELS:

    print(
        f"\nFinal tuning: "
        f"{model_name}"
    )


    candidate_k_values = (
        [None]
        if model_name == "M1"
        else TOP_K_VALUES
    )


    candidate_results = []


    final_inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=(
            RANDOM_SEED
            + 1000
        ),
    )


    for top_k in candidate_k_values:

        for params in LEARNER_GRID:

            scores = []


            for (
                train_idx,
                valid_idx,
            ) in final_inner_cv.split(
                development_ids,
                development_events,
            ):

                train_ids = [
                    development_ids[i]
                    for i
                    in train_idx
                ]

                valid_ids = [
                    development_ids[i]
                    for i
                    in valid_idx
                ]

                y_valid = make_survival_y(
                    survival.loc[
                        valid_ids
                    ]
                )


                try:

                    risk, _ = evaluate_model(
                        model_name,
                        train_ids,
                        valid_ids,
                        clinical,
                        expression,
                        rewiring,
                        survival,
                        top_k,
                        params,
                        use_seed_ensemble=False,
                    )

                    scores.append(
                        cindex(
                            y_valid,
                            risk,
                        )
                    )

                except Exception as exc:

                    print(
                        "    "
                        "[final-tuning failure] "
                        f"{model_name}, "
                        f"k={top_k}, "
                        f"{params['learner']}: "
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    )

                    scores.append(
                        np.nan
                    )


            valid_scores = [
                s
                for s in scores
                if np.isfinite(s)
            ]

            mean_c = (
                float(
                    np.mean(
                        valid_scores
                    )
                )
                if valid_scores
                else np.nan
            )


            candidate_results.append(
                {
                    "top_k":
                        top_k,
                    "params":
                        params,
                    "mean_cindex":
                        mean_c,
                }
            )


    valid_candidates = [
        x
        for x in candidate_results
        if np.isfinite(
            x["mean_cindex"]
        )
    ]


    if not valid_candidates:

        raise RuntimeError(
            f"No valid final tuning "
            f"candidate for "
            f"{model_name}"
        )


    best = max(
        valid_candidates,
        key=lambda x:
            x["mean_cindex"],
    )


    final_specs[
        model_name
    ] = best


    print(
        "  Best specification:"
    )

    print(
        f"  {best}"
    )


# Save specification BEFORE test
with open(
    RESULT_DIR
    / "STAGE_D_V4_OOF_FINAL_SPECS.json",
    "w",
) as f:

    json.dump(
        final_specs,
        f,
        indent=2,
    )


print(
    "\nFinal specifications "
    "saved BEFORE internal-test evaluation."
)


# =============================================================================
# STEP 5 — INTERNAL TEST
# =============================================================================

section(
    "STEP 5 -- INTERNAL 29-PATIENT TEST"
)


locked_results = []
final_selected_features = []

y_test = make_survival_y(
    survival.loc[
        test_ids
    ]
)


for model_name in MODELS:

    spec = final_specs[
        model_name
    ]


    risk, selected = evaluate_model(
        model_name,
        development_ids,
        test_ids,
        clinical,
        expression,
        rewiring,
        survival,
        spec["top_k"],
        spec["params"],
        use_seed_ensemble=True,
    )


    test_c = cindex(
        y_test,
        risk,
    )


    print(
        f"\nFINAL "
        f"{model_name} "
        f"("
        f"{MODEL_DESCRIPTIONS[model_name]}"
        f")"
    )

    print(
        f"Internal-test C-index: "
        f"{test_c:.4f}"
    )

    print(
        f"Learner: "
        f"{spec['params']['learner']}"
    )

    print(
        f"Top-K: "
        f"{spec['top_k']}"
    )


    locked_results.append(
        {
            "model":
                model_name,
            "description":
                MODEL_DESCRIPTIONS[
                    model_name
                ],
            "development_inner_cindex":
                spec[
                    "mean_cindex"
                ],
            "internal_test_cindex":
                test_c,
            "top_k":
                spec[
                    "top_k"
                ],
            "learner":
                spec[
                    "params"
                ][
                    "learner"
                ],
            "n_selected_molecular_features":
                (
                    0
                    if model_name == "M1"
                    else len(selected)
                ),
        }
    )


    if model_name != "M1":

        for (
            rank,
            feature,
        ) in enumerate(
            selected,
            start=1,
        ):

            final_selected_features.append(
                {
                    "model":
                        model_name,
                    "rank":
                        rank,
                    "feature":
                        feature,
                }
            )


locked_results_df = pd.DataFrame(
    locked_results
)


# =============================================================================
# STEP 6 — RESULTS
# =============================================================================

section(
    "STEP 6 -- FINAL M1-M6 RESULTS"
)


print(
    locked_results_df.to_string(
        index=False
    )
)


locked_results_df.to_csv(
    RESULT_DIR
    / "STAGE_D_V4_OOF_INTERNAL_TEST_RESULTS.csv",
    index=False,
)


pd.DataFrame(
    final_selected_features
).to_csv(
    RESULT_DIR
    / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv",
    index=False,
)


# =============================================================================
# AUDIT
# =============================================================================

audit = {

    "version":
        "V4_OOF_CORRECTED",

    "outer_repeats":
        N_OUTER_REPEATS,

    "outer_folds":
        OUTER_FOLDS,

    "inner_folds":
        INNER_FOLDS,

    "meta_oof_folds":
        META_OOF_FOLDS,

    "top_k_values":
        TOP_K_VALUES,

    "final_seed_ensemble_size":
        N_FINAL_SEEDS,

    "learners":
        [
            "rsf",
            "coxnet",
            "gbsa",
        ],

    "models":
        MODEL_DESCRIPTIONS,

    "stacking_training_risks":
        "internal out-of-fold predictions",

    "molecular_selection_inside_meta_oof":
        True,

    "test_set_used_for_tuning":
        False,

    "test_set_used_for_feature_selection":
        False,

    "note":
        (
            "Internal 29-patient test has already "
            "been observed in previous project "
            "iterations. Use repeated development "
            "outer CV as the primary comparison "
            "between V4 and V4-OOF-corrected. "
            "External validation remains required."
        ),
}


with open(
    RESULT_DIR
    / "STAGE_D_V4_OOF_AUDIT.json",
    "w",
) as f:

    json.dump(
        audit,
        f,
        indent=2,
    )


section(
    "V4 OOF-CORRECTED RUN COMPLETE"
)


print(
    f"Total wall time: "
    f"{(time.time() - pipeline_start) / 60:.1f} min"
)


print(
    "\nResults saved in:"
)

print(
    RESULT_DIR
)


print(
    "\nPRIMARY FILE TO EXAMINE FIRST:"
)

print(
    RESULT_DIR
    / "STAGE_D_V4_OOF_DEVELOPMENT_SUMMARY.csv"
)


print(
    "\nIMPORTANT:"
)

print(
    "Compare original V4 versus this corrected "
    "version using DEVELOPMENT OUTER CV first."
)

print(
    "Do NOT choose the version based on the "
    "29-patient internal-test C-index."
)

# %%%%%% CELL 10

### FINAL RUN -- COHORT CHARACTERISTICS TABLE + ADDITIONAL PERFORMANCE METRICS

In [ ]:
"""
FINAL RUN -- COHORT CHARACTERISTICS TABLE + ADDITIONAL PERFORMANCE METRICS
=========================================================================

PART A  Baseline characteristics (new Table 1):
        TCGA development (115), TCGA internal test (29), CGGA external (104)
        deaths, censored, median OS, median follow-up (reverse Kaplan-Meier),
        age, sex, MGMT, KPS, radiotherapy / TMZ (CGGA only), missingness.

PART B  Performance metrics for every model on the internal test and CGGA:
          Harrell C + 2,000-replicate bootstrap 95% CI
          Uno's C (IPCW, tau = 24 months)
          time-dependent AUC at 12 and 24 months
          integrated Brier score (IBS, 3-24 months) next to a Kaplan-Meier
          reference IBS (a model is only useful if its IBS is LOWER than KM's)
        Models refit from the FROZEN final specifications (nothing re-tuned):
          M1 clinical, M2 expression, M3 eNESH, R1, R2, R2B
        CGGA M1 and M4-M6 use your saved CGGA risk predictions (risk-based
        metrics only; M4, M5 and M6 are identical, so one row is shown).

Before computing new metrics, every refit model must reproduce its saved
internal Harrell C (checkpoint) -- so the new numbers belong to the same models.

How to run: paste into a new notebook cell (Stage D environment) and run once.
Check everything printed from "PART A" onwards.
"""

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import (concordance_index_censored, concordance_index_ipcw,
                            cumulative_dynamic_auc, integrated_brier_score)
from sksurv.nonparametric import kaplan_meier_estimator

warnings.filterwarnings("ignore")

# =============================================================================
# 0. PATHS AND SETTINGS
# =============================================================================
DOWNLOADS = BASE
STAGE_C_REW = DOWNLOADS / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
STAGE_D = DOWNLOADS / "TCGA_GBM_STAGE_D_SURVIVAL"
V4_DIR = STAGE_D / "STAGE_D_PART3_V4_OOF_CORRECTED"
LADDER_DIR = STAGE_D / "REVISION_STEP2_DEVIATION_LADDER"
CGGA_DIR = DOWNLOADS / "CGGA_GBM_EXTERNAL_REWIRING"

SPLIT_FILE = STAGE_D / "STAGE_D_LOCKED_SPLIT.csv"
SURVIVAL_FILE = STAGE_D / "STAGE_D_SURVIVAL_MODELING_COHORT.csv"
M1_FILE = STAGE_D / "STAGE_D_M1_CLINICAL_RAW.csv"
GTEX_FILE = STAGE_C_REW / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv"
EDGE_FILE = STAGE_C_REW / "STAGE_C_FINAL_FROZEN_EDGE_UNIVERSE.csv"
CGGA_EXPR_FILE = CGGA_DIR / "CGGA_104_HARMONIZED_TO_GTEX.csv"
CGGA_ENESH_FILE = CGGA_DIR / "CGGA_104_EXTERNAL_ENESH_3944.csv"
CGGA_M1M6_PRED = CGGA_DIR / "CGGA_104_FINAL_M1_M4_M5_M6_PREDICTIONS.csv"

OUT_DIR = STAGE_D / "REVISION_FINAL_COHORT_AND_METRICS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_FINAL_SEEDS = 5
N_BOOT = 2000
N_REF = 334
DAYS_PER_MONTH = 30.4375
TAU = 24.0                      # months, Uno's C
AUC_TIMES = [12.0, 24.0]        # months
IBS_START, IBS_END = 3.0, 24.0  # months
EVENT_COL = "Censor (alive=0; dead=1)"


def checkpoint(cond, msg):
    print(f"[CHECKPOINT {'PASS' if cond else 'FAIL'}] {msg}")
    if not cond:
        raise AssertionError(msg)


def section(t):
    print("\n" + "=" * 80 + f"\n{t}\n" + "=" * 80)


def strip_version(idx):
    return pd.Index(idx).astype(str).str.split(".").str[0].str.strip()


def make_y(event, time):
    return np.array([(bool(e), float(t)) for e, t in zip(event, time)],
                    dtype=[("event", "?"), ("time", "<f8")])


def harrell(y, risk):
    return float(concordance_index_censored(y["event"], y["time"],
                                            np.asarray(risk, float))[0])


# =============================================================================
# Kaplan-Meier helpers
# =============================================================================
def km_median(event, time):
    t, s = kaplan_meier_estimator(np.asarray(event, bool), np.asarray(time, float))
    below = t[s <= 0.5]
    return float(below[0]) if len(below) else np.nan


def km_survival_at(event, time, grid):
    t, s = kaplan_meier_estimator(np.asarray(event, bool), np.asarray(time, float))
    idx = np.searchsorted(t, grid, side="right") - 1
    return np.where(idx >= 0, s[np.clip(idx, 0, None)], 1.0)


# =============================================================================
# Model fitting -- identical to your Stage D fit_predict / seed ensemble,
# with survival functions enabled (risk predictions are unchanged)
# =============================================================================
def fit_models(params, X, y):
    if params["learner"] == "rsf":
        models = []
        for k in range(N_FINAL_SEEDS):
            m = RandomSurvivalForest(
                n_estimators=params["n_estimators"],
                min_samples_leaf=params["min_samples_leaf"],
                max_features=params["max_features"], min_samples_split=10,
                random_state=RANDOM_SEED + k, n_jobs=-1)
            models.append(m.fit(X, y))
        return models
    if params["learner"] == "coxnet":
        m = CoxnetSurvivalAnalysis(l1_ratio=params["l1_ratio"],
                                   alpha_min_ratio=params["alpha_min_ratio"],
                                   fit_baseline_model=True)
        return [m.fit(X, y)]
    if params["learner"] == "gbsa":
        m = GradientBoostingSurvivalAnalysis(
            n_estimators=params["n_estimators"], learning_rate=params["learning_rate"],
            max_depth=params["max_depth"], subsample=params["subsample"],
            random_state=RANDOM_SEED)
        return [m.fit(X, y)]
    raise ValueError(params["learner"])


def predict_risk(models, X):
    return np.mean([m.predict(X) for m in models], axis=0)


def predict_surv(models, X, grid):
    mats = []
    for m in models:
        fns = m.predict_survival_function(X)
        mats.append(np.vstack([fn(np.clip(grid, fn.domain[0], fn.domain[1]))
                               for fn in fns]))
    return np.mean(mats, axis=0)


# =============================================================================
# Metric helpers (each returns NaN with a printed reason instead of crashing)
# =============================================================================
def admin_censor(y, cap):
    ev = y["event"] & (y["time"] <= cap)
    return make_y(ev, np.minimum(y["time"], cap))


def boot_ci(y, risk, rng_seed=RANDOM_SEED):
    rng = np.random.default_rng(rng_seed)
    n, vals = len(y), []
    for _ in range(N_BOOT):
        idx = rng.integers(0, n, size=n)
        try:
            vals.append(harrell(y[idx], risk[idx]))
        except Exception:
            pass
    vals = np.array(vals)
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5), len(vals)


def safe(fn, label):
    try:
        return fn()
    except Exception as e:
        print(f"    ({label} not estimable: {str(e)[:90]})")
        return np.nan


def evaluate(name, cohort, y_train, y_eval, risk, surv_fn=None):
    """surv_fn(grid) -> survival matrix, or None for risk-only models."""
    cap = min(36.0, 0.98 * float(y_train["time"].max()))
    y_cap = admin_censor(y_eval, cap)
    c = harrell(y_eval, risk)
    lo, hi, nb = boot_ci(y_eval, risk)
    row = {"model": name, "cohort": cohort, "n": len(y_eval),
           "events": int(y_eval["event"].sum()), "harrell_C": c,
           "harrell_CI_low": lo, "harrell_CI_high": hi}
    row["uno_C_tau24"] = safe(lambda: concordance_index_ipcw(
        y_train, y_cap, risk, tau=TAU)[0], f"{name} Uno C")
    for t in AUC_TIMES:
        row[f"AUC_{int(t)}m"] = safe(lambda t=t: cumulative_dynamic_auc(
            y_train, y_cap, risk, [t])[0][0], f"{name} AUC {int(t)}m")
    end = min(IBS_END, 0.95 * float(y_cap["time"].max()))
    grid = np.linspace(IBS_START, end, 20)
    km = np.tile(km_survival_at(y_train["event"], y_train["time"], grid),
                 (len(y_eval), 1))
    row["IBS_KM_reference"] = safe(lambda: integrated_brier_score(
        y_train, y_cap, km, grid), f"{name} KM IBS")
    row["IBS"] = (safe(lambda: integrated_brier_score(
        y_train, y_cap, surv_fn(grid), grid), f"{name} IBS")
        if surv_fn is not None else np.nan)
    return row


# =============================================================================
# 1. LOAD TCGA
# =============================================================================
section("LOAD DATA")
split = pd.read_csv(SPLIT_FILE)
split["patient_id"] = split["patient_id"].astype(str)
dev_ids = split.loc[split["split"] == "development", "patient_id"].tolist()
test_ids = split.loc[split["split"] == "locked_test", "patient_id"].tolist()
checkpoint(len(dev_ids) == 115 and len(test_ids) == 29, "115 development / 29 internal test")

surv = pd.read_csv(SURVIVAL_FILE)
surv["patient_id"] = surv["patient_id"].astype(str)
surv = surv.set_index("patient_id")
y_dev = make_y(surv.loc[dev_ids, "OS_event"], surv.loc[dev_ids, "OS_months"])
y_test = make_y(surv.loc[test_ids, "OS_event"], surv.loc[test_ids, "OS_months"])

clin = pd.read_csv(M1_FILE, index_col=0)
clin.index = clin.index.astype(str)
print("TCGA clinical columns:", list(clin.columns))

# CGGA clinical cohort (same eligibility rule as your M2 / M3 cells)
cands = [p for p in DOWNLOADS.rglob("CGGA.mRNAseq_693_clinical.20200506.txt") if p.is_file()]
checkpoint(len(cands) >= 1, "CGGA clinical file found")
cc = pd.read_csv(cands[0], sep="\t", low_memory=False)
for c in ("Age", "OS", EVENT_COL):
    cc[c] = pd.to_numeric(cc[c], errors="coerce")
elig = cc.loc[(cc["PRS_type"] == "Primary") & (cc["Histology"] == "GBM")
              & (cc["Grade"] == "WHO IV") & (cc["IDH_mutation_status"] == "Wildtype")
              & (cc["Age"] >= 18) & (cc["OS"] > 0) & cc[EVENT_COL].isin([0, 1])].copy()
elig["CGGA_ID"] = elig["CGGA_ID"].astype(str)
checkpoint(len(elig) == 104 and int(elig[EVENT_COL].sum()) == 92, "CGGA cohort = 104, 92 events")
order = elig["CGGA_ID"].tolist()
elig = elig.set_index("CGGA_ID").loc[order]
y_cgga = make_y(elig[EVENT_COL].to_numpy(bool), elig["OS"].to_numpy(float) / DAYS_PER_MONTH)
print("CGGA OS converted from days to months (/30.4375) so all cohorts use months.")

# =============================================================================
# PART A. BASELINE CHARACTERISTICS
# =============================================================================
section("PART A -- BASELINE CHARACTERISTICS (new Table 1)")


def find_col(df, *keys):
    for c in df.columns:
        if all(k.lower() in c.lower() for k in keys):
            return c
    return None


def median_iqr(x):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna()
    if len(x) == 0:
        return "not available"
    return f"{x.median():.1f} ({x.quantile(.25):.1f}-{x.quantile(.75):.1f})"


def counts(x):
    s = pd.Series(x).astype("object").where(pd.Series(x).notna(), "missing")
    n = len(s)
    return "; ".join(f"{k}: {v} ({100 * v / n:.0f}%)" for k, v in s.value_counts().items())


def fmt_med(v):
    return "not reached" if not np.isfinite(v) else f"{v:.1f}"


def cohort_column(event, time, age, sex, mgmt, kps, rt=None, tmz=None):
    event, time = np.asarray(event, bool), np.asarray(time, float)
    n = len(event)
    return {
        "N": str(n),
        "Deaths, n (%)": f"{event.sum()} ({100 * event.mean():.0f}%)",
        "Censored, n": str(n - event.sum()),
        "Median OS, months (KM)": fmt_med(km_median(event, time)),
        "Median follow-up, months (reverse KM)": fmt_med(km_median(~event, time)),
        "Age, years, median (IQR)": median_iqr(age),
        "Sex": counts(sex) if sex is not None else "not available",
        "MGMT promoter status": counts(mgmt) if mgmt is not None else "not available",
        "KPS, median (IQR)": median_iqr(kps) if kps is not None else "not available",
        "KPS missing, n": (str(int(pd.to_numeric(pd.Series(kps), errors='coerce').isna().sum()))
                           if kps is not None else "not available"),
        "Radiotherapy": counts(rt) if rt is not None else "not available",
        "Temozolomide": counts(tmz) if tmz is not None else "not available",
    }


age_c = find_col(clin, "age")
sex_c = find_col(clin, "sex")
mgmt_c = find_col(clin, "mgmt")
kps_c = find_col(clin, "karnofsky")
table = {}
for label, ids in [("TCGA development", dev_ids), ("TCGA internal test", test_ids)]:
    sub = clin.loc[ids]
    table[label] = cohort_column(
        surv.loc[ids, "OS_event"], surv.loc[ids, "OS_months"],
        sub[age_c] if age_c else [], sub[sex_c] if sex_c else None,
        sub[mgmt_c] if mgmt_c else None, sub[kps_c] if kps_c else None)
rt_c = find_col(elig, "radio")
tmz_c = find_col(elig, "chemo")
table["CGGA external"] = cohort_column(
    y_cgga["event"], y_cgga["time"], elig["Age"],
    elig[find_col(elig, "gender")] if find_col(elig, "gender") else None,
    elig[find_col(elig, "mgmt")] if find_col(elig, "mgmt") else None, None,
    elig[rt_c] if rt_c else None, elig[tmz_c] if tmz_c else None)
table1 = pd.DataFrame(table)
with pd.option_context("display.max_colwidth", 70, "display.width", 250):
    print(table1.to_string())
table1.to_csv(OUT_DIR / "TABLE1_BASELINE_CHARACTERISTICS.csv")

# =============================================================================
# PART B. MODELS
# =============================================================================
section("PART B -- REFIT FROZEN MODELS AND CHECK THEY REPRODUCE")

v4_specs = json.load(open(V4_DIR / "STAGE_D_V4_OOF_FINAL_SPECS.json"))
v4_feats = pd.read_csv(V4_DIR / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv")
v4_res = pd.read_csv(V4_DIR / "STAGE_D_V4_OOF_INTERNAL_TEST_RESULTS.csv")


def frozen_features(sel_df, model):
    f = (sel_df.loc[sel_df["model"] == model].sort_values("rank")["feature"].astype(str)
         .str.replace("EXPR::", "", regex=False).str.replace("REWIRE::", "", regex=False))
    return list(strip_version(f))


# --- CGGA molecular inputs ---------------------------------------------------
gtex = pd.read_csv(GTEX_FILE, index_col=0, low_memory=False)
gtex.index = strip_version(gtex.index)
cg_expr = pd.read_csv(CGGA_EXPR_FILE, index_col=0)
cg_expr.index = strip_version(cg_expr.index)
cg_expr.columns = cg_expr.columns.astype(str)
cg_expr = cg_expr[order]
G = gtex.loc[cg_expr.index].to_numpy(np.float64)
mu = G.mean(axis=1)
sd = np.sqrt(((G - mu[:, None]) ** 2).sum(axis=1) / N_REF)
absZ = pd.DataFrame(np.abs((cg_expr.to_numpy(np.float64) - mu[:, None]) / sd[:, None]),
                    index=cg_expr.index, columns=order)
edges = pd.read_csv(EDGE_FILE, low_memory=False)
for c in ("gene1", "gene2"):
    edges[c] = strip_version(edges[c])
avail = set(cg_expr.index)
ec = edges[edges["gene1"].isin(avail) & edges["gene2"].isin(avail)]
nbrs = {}
for a, b in zip(ec["gene1"], ec["gene2"]):
    nbrs.setdefault(a, []).append(b)
    nbrs.setdefault(b, []).append(a)

cg_enesh = pd.read_csv(CGGA_ENESH_FILE, index_col=0)
cg_enesh.index = cg_enesh.index.astype(str)
cg_enesh.columns = strip_version(cg_enesh.columns)
checkpoint(set(order).issubset(cg_enesh.index), "CGGA eNESH matrix has the 104 patients (patients x genes)")
cg_enesh = cg_enesh.loc[order]


def cgga_value(kind, g):
    if kind == "M2":
        return cg_expr.loc[g].to_numpy() if g in avail else None
    if kind == "M3":
        return cg_enesh[g].to_numpy() if g in cg_enesh.columns else None
    if g not in avail:
        return None
    if kind == "R1":
        return absZ.loc[g].to_numpy()
    nb = nbrs.get(g, [])
    if not nb:
        return None
    nm = absZ.loc[nb].mean(axis=0).to_numpy()
    return nm if kind == "R2" else absZ.loc[g].to_numpy() * nm


MOLECULAR = [
    ("M2 expression", "M2", STAGE_D / "STAGE_D_M2_EXPRESSION_RAW.csv", v4_specs["M2"],
     frozen_features(v4_feats, "M2"),
     float(v4_res.query("model == 'M2'")["internal_test_cindex"].iloc[0])),
    ("M3 eNESH", "M3", STAGE_D / "STAGE_D_M3_REWIRING_RAW.csv", v4_specs["M3"],
     frozen_features(v4_feats, "M3"),
     float(v4_res.query("model == 'M3'")["internal_test_cindex"].iloc[0])),
]
for kind, fname in [("R1", "STAGE_D_R1_ABS_DEVIATION_RAW.csv"),
                    ("R2", "STAGE_D_R2_NEIGHBOUR_ABS_DEVIATION_RAW.csv"),
                    ("R2B", "STAGE_D_R2B_GENE_X_NEIGHBOUR_DEVIATION_RAW.csv")]:
    rdir = STAGE_D / f"REVISION_STEP2B_{kind}"
    sel = pd.read_csv(rdir / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv")
    res = pd.read_csv(rdir / "STAGE_D_V4_OOF_INTERNAL_TEST_RESULTS.csv")
    MOLECULAR.append((f"{kind} deviation", kind, LADDER_DIR / fname,
                      json.load(open(rdir / "STAGE_D_V4_OOF_FINAL_SPECS.json"))["M3"],
                      frozen_features(sel, "M3"),
                      float(res.query("model == 'M3'")["internal_test_cindex"].iloc[0])))

rows = []

# --- M1 clinical (internal; refit with your exact preprocessing) --------------
def build_clinical_preprocessor(df):
    num = [c for c in ["Diagnosis Age", "Karnofsky Performance Score"] if c in df.columns]
    cat = [c for c in ["Sex", "MGMT Status"] if c in df.columns]
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)
    tr = []
    if num:
        tr.append(("numeric", SimpleImputer(strategy="median"), num))
    if cat:
        tr.append(("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                            ("onehot", oh)]), cat))
    return ColumnTransformer(transformers=tr, remainder="drop")


print("\nM1 clinical")
proc = build_clinical_preprocessor(clin.loc[dev_ids])
Xd1 = proc.fit_transform(clin.loc[dev_ids])
Xt1 = proc.transform(clin.loc[test_ids])
m1_models = fit_models(v4_specs["M1"]["params"], Xd1, y_dev)
m1_risk = predict_risk(m1_models, Xt1)
m1_saved = float(v4_res.query("model == 'M1'")["internal_test_cindex"].iloc[0])
print(f"  reconstructed internal C = {harrell(y_test, m1_risk):.6f} | saved = {m1_saved:.6f}")
checkpoint(abs(harrell(y_test, m1_risk) - m1_saved) < 1e-4, "M1 reproduced")
rows.append(evaluate("M1 clinical", "TCGA internal test", y_dev, y_test, m1_risk,
                     lambda g: predict_surv(m1_models, Xt1, g)))

# --- molecular models ---------------------------------------------------------
for name, kind, path, spec, feats, saved in MOLECULAR:
    print(f"\n{name}")
    X = pd.read_csv(path, index_col=0)
    X.index = X.index.astype(str)
    X.columns = strip_version(X.columns)
    params = spec["params"]
    Xd, Xt = X.loc[dev_ids, feats].to_numpy(float), X.loc[test_ids, feats].to_numpy(float)
    models = fit_models(params, Xd, y_dev)
    risk = predict_risk(models, Xt)
    print(f"  reconstructed internal C = {harrell(y_test, risk):.6f} | saved = {saved:.6f}")
    checkpoint(abs(harrell(y_test, risk) - saved) < 1e-4, f"{name} reproduced")
    rows.append(evaluate(name, "TCGA internal test", y_dev, y_test, risk,
                         lambda g, m=models, x=Xt: predict_surv(m, x, g)))

    keep, cols = [], []
    for f in feats:
        v = cgga_value(kind, f)
        if v is not None:
            keep.append(f)
            cols.append(v)
    dropped = [f for f in feats if f not in keep]
    print(f"  CGGA features available {len(keep)}/{len(feats)}"
          + (f" (dropped {dropped})" if dropped else ""))
    Xe = np.column_stack(cols)
    models_a = models if not dropped else fit_models(
        params, X.loc[dev_ids, keep].to_numpy(float), y_dev)
    risk_e = predict_risk(models_a, Xe)
    print(f"  CGGA Harrell C = {harrell(y_cgga, risk_e):.4f}")
    rows.append(evaluate(name, "CGGA external", y_dev, y_cgga, risk_e,
                         lambda g, m=models_a, x=Xe: predict_surv(m, x, g)))

# --- CGGA M1 and M4-M6 from your saved predictions (risk-based metrics) -------
if CGGA_M1M6_PRED.exists():
    p = pd.read_csv(CGGA_M1M6_PRED)
    idc = p.columns[0]
    p[idc] = p[idc].astype(str)
    p = p.set_index(idc)
    if set(order).issubset(p.index):
        p = p.loc[order]
        for col, label in [("M1_adapted_risk", "M1 clinical (adapted, no KPS)"),
                           ("M4_adapted_risk", "M4-M6 stacked (identical)")]:
            if col in p.columns:
                r = p[col].to_numpy(float)
                print(f"\n{label}: CGGA Harrell C = {harrell(y_cgga, r):.4f} (saved predictions)")
                rows.append(evaluate(label, "CGGA external", y_dev, y_cgga, r, None))
else:
    print(f"\n(note) {CGGA_M1M6_PRED.name} not found; CGGA M1/M4-M6 rows skipped")

# =============================================================================
# SUMMARY
# =============================================================================
section("PART B -- SUMMARY (all times in months)")
res = pd.DataFrame(rows)
order_cols = ["cohort", "model", "n", "events", "harrell_C", "harrell_CI_low", "harrell_CI_high",
              "uno_C_tau24", "AUC_12m", "AUC_24m", "IBS", "IBS_KM_reference"]
res = res[order_cols].sort_values(["cohort", "model"], ascending=[False, True])
with pd.option_context("display.width", 250):
    print(res.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
res.to_csv(OUT_DIR / "TABLE2_ALL_METRICS.csv", index=False)
print("\nIBS: lower is better; a model adds information only if IBS < IBS_KM_reference.")
print(f"Saved to: {OUT_DIR}")
print("Check everything printed from PART A onwards (and any CHECKPOINT FAIL).")

### check_cohort_flow.py  -  answers two Methods questions, reads files only, changes nothing.

In [ ]:
"""
check_cohort_flow.py  -  answers two Methods questions, reads files only, changes nothing.

Q1. Why do 213 confirmed IDH-WT patients become 144?
Q2. What is the IDH status of the 60 Stage-A discovery / bridge tumours?

Run:  python check_cohort_flow.py      (check everything it prints)
"""
from pathlib import Path
import pandas as pd

D = BASE
A = D / "TCGA_GBM_STAGE_A_DISCOVERY"

FILES = {
    "idhwt213":  D / "TCGA_GBM_IDHWT_RNA_cohort.tsv",
    "real144":   D / "TCGA_GBM_REAL_IDHWT_RNA_cohort.tsv",
    "overlap259": D / "TCGA_GBM_RNA_clinical_overlap.tsv",
    "idh_audit": D / "TCGA_GBM_stage1E_IDH_source_audit.tsv",
    "real_rna":  D / "TCGA_GBM_REAL_RNA_primary_metadata.csv",
    "old_rna":   D / "Combined_RNA_correct_primary_mapping.csv",
    "disc60":    A / "STAGE_A_TCGA_GBM_INDEPENDENT_TUMORS.csv",
    "bridge":    A / "BRIDGE_CHECK_GDC_AVAILABILITY.csv",
}


def load(key):
    p = FILES[key]
    if not p.exists():
        print(f"[MISSING] {p}")
        return None
    sep = "\t" if p.suffix == ".tsv" else ","
    df = pd.read_csv(p, sep=sep, low_memory=False)
    print(f"[OK] {p.name}: {df.shape}")
    return df


def pid_set(df):
    col = next((c for c in df.columns if c.strip().lower() in
                ("patient id", "patient_id", "case_submitter_id", "submitter_id")), None)
    if col is None:
        col = next(c for c in df.columns if df[c].astype(str).str.match(r"TCGA-\w\w-\w{4}").mean() > 0.5)
    return set(df[col].astype(str).str.strip().str.upper().str[:12]), col


dfs = {k: load(k) for k in FILES}
print()

# ---------------------------------------------------------------- Q1
print("=" * 70, "\nQ1  213 -> 144\n" + "=" * 70)
s213, _ = pid_set(dfs["idhwt213"])
s144, _ = pid_set(dfs["real144"])
print("213 file:", len(s213), "| 144 file:", len(s144))
print("144 subset of 213:", s144 <= s213, "| in 144 but not 213:", sorted(s144 - s213)[:10])
dropped = s213 - s144
print("Dropped IDH-WT patients:", len(dropped))

for key in ("real_rna", "old_rna"):
    if dfs[key] is not None:
        s, _ = pid_set(dfs[key])
        print(f"  dropped patients present in {FILES[key].name}: {len(dropped & s)}")
if dfs["real_rna"] is not None:
    rr = dfs["real_rna"]
    print("\nColumns of REAL RNA metadata:", list(rr.columns))
    for c in rr.columns:
        if any(k in c.lower() for k in ("workflow", "data type", "data_type", "strategy",
                                        "experimental", "platform", "format", "sample type")):
            print(f"\n{c}:\n{rr[c].value_counts(dropna=False).head(10).to_string()}")
if dfs["old_rna"] is not None:
    orr = dfs["old_rna"]
    s_old, col = pid_set(orr)
    sub = orr[orr[col].astype(str).str.upper().str[:12].isin(dropped)]
    print("\nOLD RNA metadata rows for dropped patients:", len(sub))
    for c in orr.columns:
        if any(k in c.lower() for k in ("workflow", "data type", "data_type", "strategy",
                                        "experimental", "platform", "file name", "format")):
            print(f"\n{c} (dropped patients):\n{sub[c].astype(str).str[-45:].value_counts().head(10).to_string()}")

# clinical comparison dropped vs kept
d213 = dfs["idhwt213"]
_, c213 = pid_set(d213)
d213 = d213.assign(_pid=d213[c213].astype(str).str.upper().str[:12])
d213["_group"] = d213["_pid"].isin(s144).map({True: "kept_144", False: "dropped"})
for c in ["Overall Survival (Months)", "Diagnosis Age", "Sex", "Overall Survival Status"]:
    if c in d213.columns:
        if pd.api.types.is_numeric_dtype(d213[c]):
            print(f"\n{c}:\n{d213.groupby('_group')[c].describe()[['count','50%']].to_string()}")
        else:
            print(f"\n{c}:\n{pd.crosstab(d213['_group'], d213[c]).to_string()}")
pd.DataFrame({"Patient ID": sorted(dropped)}).to_csv(D / "CHECK_213_minus_144_dropped.csv", index=False)

# ---------------------------------------------------------------- Q2
print("\n" + "=" * 70, "\nQ2  IDH status of the 60 discovery/bridge tumours\n" + "=" * 70)
s60, _ = pid_set(dfs["disc60"])
print("Discovery tumours:", len(s60), "| overlap with 144:", len(s60 & s144),
      "| overlap with dropped IDH-WT:", len(s60 & dropped))

status = {}
for key in ("idh_audit", "overlap259"):
    df = dfs[key]
    if df is None:
        continue
    _, col = pid_set(df)
    stat_col = next((c for c in df.columns if c.lower() in ("final_idh_status", "idh_status")), None)
    if stat_col is None:
        print(f"No IDH status column in {FILES[key].name}: {list(df.columns)}")
        continue
    for p, s in zip(df[col].astype(str).str.upper().str[:12], df[stat_col]):
        status.setdefault(p, s)

st60 = pd.Series({p: status.get(p, "NOT_IN_CLINICAL_541") for p in s60})
print("\nIDH status of the 60:\n" + st60.value_counts().to_string())
st60.rename("IDH_status").rename_axis("Patient ID").to_csv(D / "CHECK_discovery60_IDH_status.csv")
print("\nSaved: CHECK_213_minus_144_dropped.csv, CHECK_discovery60_IDH_status.csv")

### step_stageA_idh_sensitivity.py

In [ ]:
"""
step_stageA_idh_sensitivity.py
Is the Stage-A gene discovery (and therefore the network) driven by the
21 discovery tumours that are NOT confirmed IDH-wildtype?

Re-runs the SAME PyDESeq2 analysis (design ~condition, refit_cooks=True,
FDR < 0.05, |log2FC| >= 1.5, 334 GTEx controls) on:
    S1: the 39 confirmed IDH-WT tumours only          (main sensitivity)
    S2: all tumours except the 8 confirmed IDH-mutant (52 tumours)
and compares each with the original 60-tumour result.

Reads only; writes to Downloads/TCGA_GBM_STAGE_A_DISCOVERY/IDH_SENSITIVITY.
Runtime: a few minutes per run. Check everything it prints.
"""
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

try:
    from pydeseq2.dds import DeseqDataSet
    from pydeseq2.ds import DeseqStats
except ImportError:
    raise ImportError("Run:  pip install pydeseq2")

D = BASE
A = D / "TCGA_GBM_STAGE_A_DISCOVERY"
DE = A / "DE_RESULTS"
OUT = A / "IDH_SENSITIVITY"
OUT.mkdir(parents=True, exist_ok=True)

COUNTS = DE / "STAGE_A_FILTERED_READ_COUNTS.csv.gz"   # 29,741 genes x 394 samples
FULL = DE / "STAGE_A_DE_FULL_RESULTS.csv"             # original 60-tumour result
TUMORS = A / "STAGE_A_TCGA_GBM_INDEPENDENT_TUMORS.csv"
IDH60 = D / "CHECK_discovery60_IDH_status.csv"
NODES = D / "TCGA_GBM_STAGE_B_NETWORK" / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
HUBS = D / "TCGA_GBM_STAGE_B_NETWORK" / "STAGE_B_TOP50_HUB_DEG_ANCHORS.csv"

FDR, LFC, MIN_COUNT, FRAC = 0.05, 1.5, 10, 0.20


def strip(ix):
    return pd.Index(ix).astype(str).str.replace(r"\.\d+$", "", regex=True)


def ensg_set(path):
    """Ensembl IDs from whichever column holds them."""
    if not path.exists():
        print(f"[MISSING] {path}")
        return None
    df = pd.read_csv(path, low_memory=False)
    for c in df.columns:
        s = df[c].astype(str)
        if s.str.startswith("ENSG").mean() > 0.9:
            return set(s.str.replace(r"\.\d+$", "", regex=True))
    print(f"[WARN] no Ensembl column in {path.name}: {list(df.columns)[:10]}")
    return None


def deg_set(res):
    sig = res[(res["FDR"] < FDR) & (res["log2FC"].abs() >= LFC)]
    return set(strip(sig.index)), sig


# ------------------------------------------------------------------ load
counts = pd.read_csv(COUNTS, index_col=0)
print("Filtered counts:", counts.shape)
tum = pd.read_csv(TUMORS)
idh = pd.read_csv(IDH60)
tum = tum.merge(idh, on="Patient ID", how="left")
print(tum["IDH_status"].value_counts(dropna=False).to_string())

tumor_cols = [c for c in tum["Recount sample ID"].astype(str) if c in counts.columns]
normal_cols = [c for c in counts.columns if c not in set(tum["Recount sample ID"].astype(str))]
print("Tumour columns found:", len(tumor_cols), "| normal columns:", len(normal_cols))
assert len(tumor_cols) == 60 and len(normal_cols) == 334, "sample ID mismatch - check this output"

orig = pd.read_csv(FULL, index_col=0)
orig_deg, orig_sig = deg_set(orig)
print("\nOriginal DEGs (60 tumours):", len(orig_deg))
anchors = ensg_set(NODES)
hubs = ensg_set(HUBS)
if anchors:
    print("Original network anchors:", len(anchors))


# ------------------------------------------------------------------ rerun
def run(label, keep_status):
    keep = tum[tum["IDH_status"].isin(keep_status)]["Recount sample ID"].astype(str)
    tcols = [c for c in tumor_cols if c in set(keep)]
    cols = tcols + normal_cols
    X = counts[cols]
    # same low-expression rule, recomputed for the smaller tumour group
    min_s = math.ceil(min(len(tcols), len(normal_cols)) * FRAC)
    X = X[(X >= MIN_COUNT).sum(axis=1) >= min_s]
    meta = pd.DataFrame({"condition": pd.Categorical(
        ["tumor"] * len(tcols) + ["normal"] * len(normal_cols),
        categories=["normal", "tumor"])}, index=cols)
    Xt = X.T.round().astype(int)
    try:
        dds = DeseqDataSet(counts=Xt, metadata=meta, design="~condition", refit_cooks=True)
    except TypeError:
        dds = DeseqDataSet(counts=Xt, metadata=meta, design_factors="condition", refit_cooks=True)
    dds.deseq2()
    st = DeseqStats(dds, contrast=["condition", "tumor", "normal"])
    st.summary()
    res = st.results_df.rename(columns={"log2FoldChange": "log2FC", "padj": "FDR"})
    res.to_csv(OUT / f"{label}_DE_FULL_RESULTS.csv")

    new_deg, new_sig = deg_set(res)
    common = orig.index.intersection(res.index)
    rho = spearmanr(orig.loc[common, "log2FC"], res.loc[common, "log2FC"], nan_policy="omit")[0]
    both = orig_deg & new_deg
    o = orig_sig.set_axis(strip(orig_sig.index))
    n = new_sig.set_axis(strip(new_sig.index))
    same_dir = (np.sign(o.loc[list(both), "log2FC"]) == np.sign(n.loc[list(both), "log2FC"])).mean()
    row = {
        "analysis": label,
        "n_tumours": len(tcols),
        "genes_tested": len(res),
        "DEGs": len(new_deg),
        "DEGs_up": int((new_sig["log2FC"] > 0).sum()),
        "DEGs_down": int((new_sig["log2FC"] < 0).sum()),
        "overlap_with_original": len(both),
        "pct_original_DEGs_recovered": 100 * len(both) / len(orig_deg),
        "jaccard": len(both) / len(orig_deg | new_deg),
        "direction_agreement_pct": 100 * same_dir,
        "spearman_log2FC_all_genes": rho,
    }
    if anchors:
        row["pct_anchors_still_DEG"] = 100 * len(anchors & new_deg) / len(anchors)
    if hubs:
        row["top50_hubs_still_DEG"] = f"{len(hubs & new_deg)}/{len(hubs)}"
    return row


rows = [
    run("S1_IDHWT_only", ["IDH_WT_CONFIRMED"]),
    run("S2_exclude_IDH_mutant", ["IDH_WT_CONFIRMED", "IDH_UNRESOLVED", "NOT_IN_CLINICAL_541"]),
]
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "STAGE_A_IDH_SENSITIVITY_SUMMARY.csv", index=False)
pd.set_option("display.width", 250)
print("\n" + "=" * 80 + "\nSTAGE A IDH SENSITIVITY\n" + "=" * 80)
print(summary.T.to_string(header=False))
print("\nSaved ->", OUT)

### Clinical positive control and additional analyses

In [ ]:
"""
Clinical positive control and additional analyses
=================================================
Reads the Stage D / CGGA output files only. Writes CSVs to
Downloads/TCGA_GBM_STAGE_D_SURVIVAL/REVISION_REVIEW_ANALYSES.

PART A1  Positive control: does known clinical prognosis (age, MGMT, KPS)
         work in development CV, the internal partition and CGGA?
         + sanity checks of CGGA survival / MGMT coding.
PART A3  How much of eNESH is explained by deviation from the reference?
         Gene-wise R^2 of eNESH on (local deviation) and (+ patient's global
         deviation), compared with eNESH on expression. Count of binary
         (0/1) reference-isolated features overall and in the final M3 model.
PART A4  How much of each gene's z-score is a constant TCGA-vs-GTEx offset
         (same in every patient)?
PART A5  Small numbers for the text: KM median OS of all 144; CGGA mapping
         counts (12,338 + 553 vs 12,888) if the mapping file is found.

Run in your Stage D environment. Check everything printed from PART A1.
"""
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.compare import compare_survival

warnings.filterwarnings("ignore")

DOWNLOADS = BASE
STAGE_C_REW = DOWNLOADS / "TCGA_GBM_STAGE_C_PERSONALIZED_REWIRING"
STAGE_D = DOWNLOADS / "TCGA_GBM_STAGE_D_SURVIVAL"
V4_DIR = STAGE_D / "STAGE_D_PART3_V4_OOF_CORRECTED"
LADDER_DIR = STAGE_D / "REVISION_STEP2_DEVIATION_LADDER"
CGGA_DIR = DOWNLOADS / "CGGA_GBM_EXTERNAL_REWIRING"
OUT = STAGE_D / "REVISION_REVIEW_ANALYSES"
OUT.mkdir(parents=True, exist_ok=True)

SEED, N_BOOT, N_REF = 42, 2000, 334
DAYS_PER_MONTH = 30.4375
EVENT_COL = "Censor (alive=0; dead=1)"


def section(t):
    print("\n" + "=" * 80 + f"\n{t}\n" + "=" * 80)


def checkpoint(cond, msg):
    print(f"[CHECKPOINT {'PASS' if cond else 'FAIL'}] {msg}")
    if not cond:
        raise AssertionError(msg)


def strip_version(idx):
    return pd.Index(idx).astype(str).str.split(".").str[0].str.strip()


def make_y(event, time):
    return np.array([(bool(e), float(t)) for e, t in zip(event, time)],
                    dtype=[("event", "?"), ("time", "<f8")])


def harrell(y, risk):
    return float(concordance_index_censored(y["event"], y["time"], np.asarray(risk, float))[0])


def boot_ci(y, risk):
    rng = np.random.default_rng(SEED)
    vals = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        try:
            vals.append(harrell(y[i], risk[i]))
        except Exception:
            pass
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)


def km_median(event, time):
    t, s = kaplan_meier_estimator(np.asarray(event, bool), np.asarray(time, float))
    b = t[s <= 0.5]
    return float(b[0]) if len(b) else np.nan


def find_col(df, *keys):
    for c in df.columns:
        if all(k.lower() in str(c).lower() for k in keys):
            return c
    return None


def mgmt_code(series):
    """1 = methylated, 0 = unmethylated, NaN = missing/other."""
    s = series.astype(str).str.strip().str.lower()
    out = pd.Series(np.nan, index=series.index)
    unm = s.str.contains("un") & s.str.contains("methyl")
    met = s.str.contains("methyl") & ~unm
    out[met] = 1.0
    out[unm] = 0.0
    return out


# =============================================================================
# LOAD
# =============================================================================
section("LOAD")
split = pd.read_csv(STAGE_D / "STAGE_D_LOCKED_SPLIT.csv")
split["patient_id"] = split["patient_id"].astype(str)
dev = split.loc[split["split"] == "development", "patient_id"].tolist()
test = split.loc[split["split"] == "locked_test", "patient_id"].tolist()
checkpoint(len(dev) == 115 and len(test) == 29, "115 / 29 split")

surv = pd.read_csv(STAGE_D / "STAGE_D_SURVIVAL_MODELING_COHORT.csv")
surv["patient_id"] = surv["patient_id"].astype(str)
surv = surv.set_index("patient_id")
y_dev = make_y(surv.loc[dev, "OS_event"], surv.loc[dev, "OS_months"])
y_test = make_y(surv.loc[test, "OS_event"], surv.loc[test, "OS_months"])

clin = pd.read_csv(STAGE_D / "STAGE_D_M1_CLINICAL_RAW.csv", index_col=0)
clin.index = clin.index.astype(str)
age_c, mgmt_c, kps_c = find_col(clin, "age"), find_col(clin, "mgmt"), find_col(clin, "karnofsky")
print("TCGA columns used:", age_c, "|", mgmt_c, "|", kps_c)
print("TCGA MGMT raw values:\n" + clin[mgmt_c].astype(str).value_counts(dropna=False).to_string())

cands = [p for p in DOWNLOADS.rglob("CGGA.mRNAseq_693_clinical.20200506.txt") if p.is_file()]
checkpoint(len(cands) >= 1, "CGGA clinical file found")
cc = pd.read_csv(cands[0], sep="\t", low_memory=False)
for c in ("Age", "OS", EVENT_COL):
    cc[c] = pd.to_numeric(cc[c], errors="coerce")
elig = cc.loc[(cc["PRS_type"] == "Primary") & (cc["Histology"] == "GBM")
              & (cc["Grade"] == "WHO IV") & (cc["IDH_mutation_status"] == "Wildtype")
              & (cc["Age"] >= 18) & (cc["OS"] > 0) & cc[EVENT_COL].isin([0, 1])].copy()
checkpoint(len(elig) == 104, "CGGA cohort = 104")
elig["CGGA_ID"] = elig["CGGA_ID"].astype(str)
elig = elig.set_index("CGGA_ID")
y_cg = make_y(elig[EVENT_COL].to_numpy(bool), elig["OS"].to_numpy(float) / DAYS_PER_MONTH)
cg_mgmt_c = find_col(elig, "mgmt")
print("CGGA MGMT column:", cg_mgmt_c)
print("CGGA MGMT raw values:\n" + elig[cg_mgmt_c].astype(str).value_counts(dropna=False).to_string())


# =============================================================================
# PART A1 -- POSITIVE CONTROL
# =============================================================================
section("PART A1 -- CLINICAL POSITIVE CONTROL (Cox models)")


def design(age, mgmt, kps=None, ref=None):
    """Age (per 10 y), MGMT methylated (missing -> separate indicator), optional KPS (per 10)."""
    X = pd.DataFrame(index=age.index)
    X["age_per10"] = pd.to_numeric(age, errors="coerce") / 10.0
    m = mgmt_code(mgmt)
    X["mgmt_methylated"] = m.fillna(0.0)
    X["mgmt_missing"] = m.isna().astype(float)
    if kps is not None:
        X["kps_per10"] = pd.to_numeric(kps, errors="coerce") / 10.0
    med = (ref if ref is not None else X).median()
    return X.fillna(med), med


def fit_cox(X, y, cols):
    return CoxPHSurvivalAnalysis(alpha=1e-4).fit(X[cols].to_numpy(float), y)


SPECS = {
    "age only": ["age_per10"],
    "MGMT only": ["mgmt_methylated", "mgmt_missing"],
    "age + MGMT": ["age_per10", "mgmt_methylated", "mgmt_missing"],
    "age + MGMT + KPS": ["age_per10", "mgmt_methylated", "mgmt_missing", "kps_per10"],
}

# development: same 3 x 5 outer CV layout as Stage D (stratified by event, seed 42)
rows = []
Xall, _ = design(clin.loc[dev, age_c], clin.loc[dev, mgmt_c], clin.loc[dev, kps_c])
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
for name, cols in SPECS.items():
    cs = []
    for tr, va in rskf.split(np.zeros(len(dev)), y_dev["event"]):
        ids_tr, ids_va = [dev[i] for i in tr], [dev[i] for i in va]
        Xtr, med = design(clin.loc[ids_tr, age_c], clin.loc[ids_tr, mgmt_c], clin.loc[ids_tr, kps_c])
        Xva, _ = design(clin.loc[ids_va, age_c], clin.loc[ids_va, mgmt_c], clin.loc[ids_va, kps_c], ref=med)
        m = fit_cox(Xtr, y_dev[tr], cols)
        cs.append(harrell(y_dev[va], m.predict(Xva[cols].to_numpy(float))))
    rows.append({"model": name, "cohort": "TCGA development CV (15 folds)",
                 "C": np.mean(cs), "CI_low": np.mean(cs) - np.std(cs), "CI_high": np.mean(cs) + np.std(cs),
                 "note": "mean +/- SD"})

# internal partition and CGGA: fit on all 115 development patients
Xd, med = design(clin.loc[dev, age_c], clin.loc[dev, mgmt_c], clin.loc[dev, kps_c])
Xt, _ = design(clin.loc[test, age_c], clin.loc[test, mgmt_c], clin.loc[test, kps_c], ref=med)
Xc, _ = design(elig["Age"], elig[cg_mgmt_c], None, ref=med)
coef_rows = []
for name, cols in SPECS.items():
    m = fit_cox(Xd, y_dev, cols)
    coef_rows.append({"model": name, **{c: round(float(b), 3) for c, b in zip(cols, m.coef_)}})
    r = m.predict(Xt[cols].to_numpy(float))
    lo, hi = boot_ci(y_test, r)
    rows.append({"model": name, "cohort": "TCGA internal (n=29)", "C": harrell(y_test, r),
                 "CI_low": lo, "CI_high": hi, "note": "bootstrap 95% CI"})
    if "kps_per10" not in cols:
        r = m.predict(Xc[cols].to_numpy(float))
        lo, hi = boot_ci(y_cg, r)
        rows.append({"model": name, "cohort": "CGGA (n=104)", "C": harrell(y_cg, r),
                     "CI_low": lo, "CI_high": hi, "note": "frozen TCGA fit; bootstrap 95% CI"})
    # CGGA refit (is the signal present at all in CGGA, independent of transfer?)
    if "kps_per10" not in cols:
        mc = fit_cox(Xc, y_cg, cols)
        rows.append({"model": name, "cohort": "CGGA refit in CGGA (apparent)",
                     "C": harrell(y_cg, mc.predict(Xc[cols].to_numpy(float))),
                     "CI_low": np.nan, "CI_high": np.nan, "note": "optimistic; sanity only"})

pc = pd.DataFrame(rows)
pd.set_option("display.width", 250)
print(pc.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nCox log-HR fitted on TCGA development (age per 10 y; MGMT methylated vs not):")
print(pd.DataFrame(coef_rows).to_string(index=False))
pc.to_csv(OUT / "A1_POSITIVE_CONTROL.csv", index=False)

print("\nCGGA sanity checks:")
print(f"  OS days -> months; KM median = {km_median(y_cg['event'], y_cg['time']):.1f} months; "
      f"events = {int(y_cg['event'].sum())}/104")
mc = mgmt_code(elig[cg_mgmt_c])
for lab, mask in [("methylated", mc == 1), ("unmethylated", mc == 0), ("missing", mc.isna())]:
    mask = mask.to_numpy()
    if mask.sum():
        print(f"  MGMT {lab:12s}: n={mask.sum():3d}, KM median OS = "
              f"{km_median(y_cg['event'][mask], y_cg['time'][mask]):.1f} months")
known = mc.notna().to_numpy()
if known.sum() > 10:
    chi, p = compare_survival(y_cg[known], mc[known].astype(int).to_numpy())
    print(f"  log-rank methylated vs unmethylated in CGGA: chi2 = {chi:.2f}, p = {p:.3g}")
mt = mgmt_code(clin.loc[dev + test, mgmt_c])
y_all = make_y(surv.loc[dev + test, "OS_event"], surv.loc[dev + test, "OS_months"])
k = mt.notna().to_numpy()
chi, p = compare_survival(y_all[k], mt[k].astype(int).to_numpy())
print(f"  log-rank methylated vs unmethylated in TCGA (144): chi2 = {chi:.2f}, p = {p:.3g}")
age_c_cg = pd.to_numeric(elig["Age"], errors="coerce").to_numpy()
print(f"  CGGA age vs OS: Harrell C (older = higher risk) = {harrell(y_cg, age_c_cg):.3f}")


# =============================================================================
# PART A3 -- VARIANCE OF eNESH EXPLAINED BY DEVIATION
# =============================================================================
section("PART A3 -- eNESH explained by deviation from the reference (115 development patients)")


def load_matrix(p):
    X = pd.read_csv(p, index_col=0)
    X.index = X.index.astype(str)
    X.columns = strip_version(X.columns)
    return X


M3 = load_matrix(STAGE_D / "STAGE_D_M3_REWIRING_RAW.csv")
M2 = load_matrix(STAGE_D / "STAGE_D_M2_EXPRESSION_RAW.csv")
R1 = load_matrix(LADDER_DIR / "STAGE_D_R1_ABS_DEVIATION_RAW.csv")
R2 = load_matrix(LADDER_DIR / "STAGE_D_R2_NEIGHBOUR_ABS_DEVIATION_RAW.csv")
R2B = load_matrix(LADDER_DIR / "STAGE_D_R2B_GENE_X_NEIGHBOUR_DEVIATION_RAW.csv")
genes = [g for g in M3.columns if g in M2.columns and g in R1.columns and g in R2.columns and g in R2B.columns]
print("Genes available in all matrices:", len(genes))
E, X2, A, B, AB = (M.loc[dev, genes].to_numpy(float) for M in (M3, M2, R1, R2, R2B))
glob = R1.loc[dev].to_numpy(float).mean(axis=1)          # patient's overall deviation (mean |z|)
glob = (glob - glob.mean()) / glob.std()


def adj_r2(y, Xcols):
    Xd_ = np.column_stack([np.ones(len(y))] + Xcols)
    ok = np.isfinite(Xd_).all(1) & np.isfinite(y)
    y_, Xd_ = y[ok], Xd_[ok]
    if np.var(y_) < 1e-12:
        return np.nan
    beta, *_ = np.linalg.lstsq(Xd_, y_, rcond=None)
    rss = ((y_ - Xd_ @ beta) ** 2).sum()
    tss = ((y_ - y_.mean()) ** 2).sum()
    n, p = Xd_.shape
    return 1 - (rss / (n - p)) / (tss / (n - 1))


res = {k: [] for k in ["expression", "local (R1,R2,R2b)", "local + global",
                       "local + global, flexible"]}
binary = []
for j in range(len(genes)):
    y = E[:, j]
    binary.append(len(np.unique(y[np.isfinite(y)])) <= 2)
    res["expression"].append(adj_r2(y, [X2[:, j]]))
    res["local (R1,R2,R2b)"].append(adj_r2(y, [A[:, j], B[:, j], AB[:, j]]))
    res["local + global"].append(adj_r2(y, [A[:, j], B[:, j], AB[:, j], glob]))
    res["local + global, flexible"].append(adj_r2(y, [A[:, j], B[:, j], AB[:, j], glob,
                                                      A[:, j] ** 2, B[:, j] ** 2, glob ** 2]))
binary = np.array(binary)
summ = []
for k, v in res.items():
    v = np.array(v, float)
    for subset, mask in [("all genes", np.ones(len(v), bool)),
                         ("non-binary genes", ~binary), ("binary (0/1) genes", binary)]:
        w = v[mask & np.isfinite(v)]
        if len(w):
            summ.append({"predictors": k, "genes": subset, "n": len(w),
                         "median_adj_R2": np.median(w), "IQR_low": np.percentile(w, 25),
                         "IQR_high": np.percentile(w, 75), "pct_R2_ge_0.5": 100 * np.mean(w >= 0.5)})
summ = pd.DataFrame(summ)
print(summ.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
summ.to_csv(OUT / "A3_ENESH_VARIANCE_EXPLAINED.csv", index=False)

print(f"\nBinary (two-valued) eNESH features among {len(genes)}: {binary.sum()}")
sel = pd.read_csv(V4_DIR / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv")
m3f = list(strip_version(sel.loc[sel["model"] == "M3", "feature"].astype(str)
                         .str.replace("REWIRE::", "", regex=False)))
nb = sum(len(np.unique(M3.loc[dev, g].dropna())) <= 2 for g in m3f if g in M3.columns)
print(f"Binary features among the {len(m3f)} final M3 features: {nb}")


# =============================================================================
# PART A4 -- CONSTANT TCGA-vs-GTEx OFFSET INSIDE z
# =============================================================================
section("PART A4 -- share of z-scores that is a constant TCGA-vs-GTEx offset")
gtex = pd.read_csv(STAGE_C_REW / "STAGE_C_GTEX_334_REFERENCE_LOG_CPM_MTFIXED.csv", index_col=0, low_memory=False)
gtex.index = strip_version(gtex.index)
g2 = [g for g in genes if g in gtex.index]
G = gtex.loc[g2].to_numpy(float)
mu, sd = G.mean(1), np.sqrt(((G - G.mean(1, keepdims=True)) ** 2).sum(1) / N_REF)
ok = sd > 0
Z = (M2.loc[dev, g2].to_numpy(float).T - mu[:, None]) / sd[:, None]       # genes x patients
Z = Z[ok]
gmean = Z.mean(1)
share = Z.shape[1] * (gmean ** 2).sum() / (Z ** 2).sum()
print(f"Genes: {Z.shape[0]}; patients: {Z.shape[1]}")
print(f"Share of total sum of squares of z that is a per-gene constant offset: {100 * share:.1f}%")
print(f"Median |per-gene mean z| = {np.median(np.abs(gmean)):.2f}; "
      f"genes with |mean z| > 2: {100 * np.mean(np.abs(gmean) > 2):.1f}%")
within = Z.std(1)
print(f"Median within-gene SD of z across patients = {np.median(within):.2f}")
pd.DataFrame({"gene": np.array(g2)[ok], "mean_z": gmean, "sd_z_across_patients": within}) \
    .to_csv(OUT / "A4_Z_OFFSET_PER_GENE.csv", index=False)


# =============================================================================
# PART A5 -- small numbers for the text
# =============================================================================
section("PART A5 -- numbers for text fixes")
y144 = make_y(surv.loc[dev + test, "OS_event"], surv.loc[dev + test, "OS_months"])
print(f"KM median OS, all 144 retained patients = {km_median(y144['event'], y144['time']):.1f} months")
print(f"Crude median of observed OS times (144) = {np.median(y144['time']):.1f} months")
maps = [p for p in CGGA_DIR.rglob("*.csv") if "map" in p.name.lower()]
for p in maps[:5]:
    try:
        d = pd.read_csv(p, nrows=200000)
        tcol = find_col(d, "mapping") or find_col(d, "type")
        print(f"\n{p.name}: {len(d)} rows" + (f"\n{d[tcol].value_counts().to_string()}" if tcol else ""))
    except Exception as e:
        print(p.name, "unreadable:", e)
if not maps:
    print("No CGGA mapping CSV found under", CGGA_DIR)

print("\nSaved to:", OUT)
print("Check everything printed from PART A1 onwards.")

### build_supplement.py -- builds the Supplementary Tables workbook

In [ ]:
"""
build_supplement.py -- builds the Supplementary Tables workbook
================================================================
Reads your existing files only. Writes one Excel workbook (plus CSV copies) to
Downloads/TCGA_GBM_STAGE_D_SURVIVAL/SUPPLEMENTARY_TABLES.

  S1  Retained (n=144) vs excluded (n=69) IDH-wildtype TCGA patients
  S2  The 752 reference-isolated DEG anchors (Stage B audit)
  S3  Every CGGA transfer variant scored against CGGA outcomes
      (recomputed with the frozen M1/M2 specifications)
  S4  Time-dependent and calibration metrics for all models
  ID lists: 144 TCGA (with split), 60 discovery/bridge (with IDH status),
            334 GTEx, 104 CGGA, 20 CGGA controls
  Final feature lists of M2, M3, R1, R2, R2b with gene symbols

Run it in your Stage D environment; check everything it prints.
"""
from pathlib import Path
import json
import shutil
import warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.nonparametric import kaplan_meier_estimator

warnings.filterwarnings("ignore")

D = BASE
STAGE_A = D / "TCGA_GBM_STAGE_A_DISCOVERY"
STAGE_B = D / "TCGA_GBM_STAGE_B_NETWORK"
STAGE_D = D / "TCGA_GBM_STAGE_D_SURVIVAL"
V4 = STAGE_D / "STAGE_D_PART3_V4_OOF_CORRECTED"
CGGA = D / "CGGA_GBM_EXTERNAL_REWIRING"
OUT = STAGE_D / "SUPPLEMENTARY_TABLES"
OUT.mkdir(parents=True, exist_ok=True)
SEED, N_SEEDS, DPM = 42, 5, 30.4375
EVENT_COL = "Censor (alive=0; dead=1)"
sheets = {}


def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78)


def sv(idx):
    return pd.Index(idx).astype(str).str.split(".").str[0].str.strip()


def make_y(e, t):
    return np.array([(bool(a), float(b)) for a, b in zip(e, t)], dtype=[("event", "?"), ("time", "<f8")])


def harrell(y, r):
    return float(concordance_index_censored(y["event"], y["time"], np.asarray(r, float))[0])


def km_median(e, t):
    tt, s = kaplan_meier_estimator(np.asarray(e, bool), np.asarray(t, float))
    b = tt[s <= 0.5]
    return float(b[0]) if len(b) else np.nan


def find_col(df, *keys):
    for c in df.columns:
        if all(k.lower() in str(c).lower() for k in keys):
            return c
    return None


def pid_col(df):
    return next(c for c in df.columns if df[c].astype(str).str.match(r"TCGA-\w\w-\w{4}").mean() > 0.5)


# ---------------------------------------------------------------- S1
section("S1 -- retained (144) vs excluded (69) IDH-wildtype patients")
p213, p144 = D / "TCGA_GBM_IDHWT_RNA_cohort.tsv", D / "TCGA_GBM_REAL_IDHWT_RNA_cohort.tsv"
if p213.exists() and p144.exists():
    a = pd.read_csv(p213, sep="\t", low_memory=False)
    b = pd.read_csv(p144, sep="\t", low_memory=False)
    ca, cb = pid_col(a), pid_col(b)
    keep = set(b[cb].astype(str).str[:12])
    a["group"] = np.where(a[ca].astype(str).str[:12].isin(keep), "Retained (survival cohort)", "Excluded")
    os_c = find_col(a, "overall survival", "month")
    st_c = find_col(a, "overall survival", "status")
    age_c = find_col(a, "diagnosis age") or find_col(a, "age")
    sex_c = find_col(a, "sex")
    rows = {}
    for g, sub in a.groupby("group"):
        ev = sub[st_c].astype(str).str.startswith("1").to_numpy()
        t = pd.to_numeric(sub[os_c], errors="coerce").to_numpy()
        ok = np.isfinite(t)
        age = pd.to_numeric(sub[age_c], errors="coerce")
        rows[g] = {
            "Patients, n": len(sub),
            "Deaths, n (%)": f"{ev.sum()} ({100 * ev.mean():.0f}%)",
            "Age, median (IQR)": f"{age.median():.0f} ({age.quantile(.25):.0f}-{age.quantile(.75):.0f})",
            "Male, n (%)": f"{(sub[sex_c].astype(str).str.lower() == 'male').sum()} "
                           f"({100 * (sub[sex_c].astype(str).str.lower() == 'male').mean():.0f}%)",
            "Observed OS time, crude median (months)": f"{np.nanmedian(t):.1f}",
            "OS, Kaplan-Meier median (months)": f"{km_median(ev[ok], t[ok]):.1f}",
        }
    s1 = pd.DataFrame(rows)
    print(s1.to_string())
    sheets["S1_retained_vs_excluded"] = s1.reset_index().rename(columns={"index": "Characteristic"})
else:
    print("[MISSING] 213 or 144 cohort file")

# ---------------------------------------------------------------- S2
section("S2 -- reference-isolated DEG anchors")
p = STAGE_B / "STAGE_B_ISOLATED_DEG_ANCHORS_AUDIT.csv"
if p.exists():
    s2 = pd.read_csv(p)
    print("rows:", len(s2), "| columns:", list(s2.columns)[:8])
    sheets["S2_isolated_anchors"] = s2
else:
    print("[MISSING]", p)

# ---------------------------------------------------------------- S3
section("S3 -- CGGA transfer variants scored against CGGA outcomes (frozen M1/M2)")
split = pd.read_csv(STAGE_D / "STAGE_D_LOCKED_SPLIT.csv")
split["patient_id"] = split["patient_id"].astype(str)
dev = split.loc[split["split"] == "development", "patient_id"].tolist()
surv = pd.read_csv(STAGE_D / "STAGE_D_SURVIVAL_MODELING_COHORT.csv")
surv["patient_id"] = surv["patient_id"].astype(str)
surv = surv.set_index("patient_id")
y_dev = make_y(surv.loc[dev, "OS_event"], surv.loc[dev, "OS_months"])

cands = [q for q in D.rglob("CGGA.mRNAseq_693_clinical.20200506.txt") if q.is_file()]
cc = pd.read_csv(cands[0], sep="\t", low_memory=False)
for c in ("Age", "OS", EVENT_COL):
    cc[c] = pd.to_numeric(cc[c], errors="coerce")
el = cc.loc[(cc["PRS_type"] == "Primary") & (cc["Histology"] == "GBM") & (cc["Grade"] == "WHO IV")
            & (cc["IDH_mutation_status"] == "Wildtype") & (cc["Age"] >= 18) & (cc["OS"] > 0)
            & cc[EVENT_COL].isin([0, 1])].copy()
el["CGGA_ID"] = el["CGGA_ID"].astype(str)
el = el.set_index("CGGA_ID")
order = el.index.tolist()
y_cg = make_y(el[EVENT_COL].to_numpy(bool), el["OS"].to_numpy(float) / DPM)

specs = json.load(open(V4 / "STAGE_D_V4_OOF_FINAL_SPECS.json"))
sel = pd.read_csv(V4 / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv")
m2_feats = list(sv(sel.loc[sel["model"] == "M2"].sort_values("rank")["feature"].astype(str)
                   .str.replace("EXPR::", "", regex=False)))
prm = specs["M2"]["params"]
print("M2 frozen spec:", prm, "| features:", len(m2_feats))


def fit_rsf(X, y):
    return [RandomSurvivalForest(n_estimators=prm["n_estimators"], min_samples_leaf=prm["min_samples_leaf"],
                                 max_features=prm["max_features"], min_samples_split=10,
                                 random_state=SEED + k, n_jobs=-1).fit(X, y) for k in range(N_SEEDS)]


def pred(ms, X):
    return np.mean([m.predict(X) for m in ms], axis=0)


M2 = pd.read_csv(STAGE_D / "STAGE_D_M2_EXPRESSION_RAW.csv", index_col=0)
M2.index = M2.index.astype(str)
M2.columns = sv(M2.columns)
Xd = M2.loc[dev, m2_feats].to_numpy(float)

raw = pd.read_csv(CGGA / "CGGA_104_LOG2CPM_FROZEN_GENES.csv", index_col=0)
har = pd.read_csv(CGGA / "CGGA_104_HARMONIZED_TO_GTEX.csv", index_col=0)
for m in (raw, har):
    m.index = sv(m.index)
    m.columns = m.columns.astype(str)
feats_ok = [g for g in m2_feats if g in raw.index]
print("M2 features measurable in CGGA:", len(feats_ok), "/", len(m2_feats))


def zcols(X):
    return (X - X.mean(0)) / (X.std(0) + 1e-12)


def rcols(X):
    return np.column_stack([(rankdata(X[:, j]) - 0.5) / X.shape[0] for j in range(X.shape[1])])


Xc_raw = raw.loc[m2_feats, order].T.to_numpy(float)
Xc_har = har.loc[m2_feats, order].T.to_numpy(float)
ms_raw = fit_rsf(Xd, y_dev)
s3 = [
    {"model": "M2 (25 genes)", "CGGA preprocessing": "log2(CPM+1), no cross-cohort correction",
     "used for reported results": "no", "CGGA C-index": harrell(y_cg, pred(ms_raw, Xc_raw))},
    {"model": "M2 (25 genes)", "CGGA preprocessing": "within-cohort z-score per gene (TCGA and CGGA separately)",
     "used for reported results": "no", "CGGA C-index": harrell(y_cg, pred(fit_rsf(zcols(Xd), y_dev), zcols(Xc_raw)))},
    {"model": "M2 (25 genes)", "CGGA preprocessing": "within-cohort rank (quantile) normalization per gene",
     "used for reported results": "no", "CGGA C-index": harrell(y_cg, pred(fit_rsf(rcols(Xd), y_dev), rcols(Xc_raw)))},
    {"model": "M2 (25 genes)", "CGGA preprocessing": "control-anchored location shift to GTEx (final)",
     "used for reported results": "yes", "CGGA C-index": harrell(y_cg, pred(ms_raw, Xc_har))},
]
s3 = pd.DataFrame(s3)
print(s3.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
old = STAGE_D / "CGGA_EXTERNAL_VALIDATION" / "CGGA_M1_M2_VALIDATION_RESULTS.csv"
if old.exists():
    print("\nEarlier saved exploratory results (as originally run):")
    print(pd.read_csv(old).to_string(index=False))
else:
    print("\n(no earlier exploratory results file found at", old, ")")
print("NOTE: an earlier 50-gene stability-selected M2 variant (v5) was also scored in CGGA; "
      "if you saved its output, send it to me.")
sheets["S3_CGGA_transfer_variants"] = s3

# ---------------------------------------------------------------- S4
section("S4 -- time-dependent and calibration metrics")
p = STAGE_D / "REVISION_FINAL_COHORT_AND_METRICS" / "TABLE2_ALL_METRICS.csv"
if p.exists():
    s4 = pd.read_csv(p)
    print(s4.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    sheets["S4_time_dependent_metrics"] = s4
else:
    print("[MISSING]", p)

# ---------------------------------------------------------------- IDs and features
section("ID lists and final feature lists")
sheets["IDs_TCGA_144"] = split.rename(columns={"split": "assignment"})
p = D / "CHECK_discovery60_IDH_status.csv"
if p.exists():
    sheets["IDs_TCGA_discovery_60"] = pd.read_csv(p)
p = STAGE_A / "STAGE_A_GTEX_CORTEX_FINAL_QC.csv"
if p.exists():
    g = pd.read_csv(p, low_memory=False)
    cols = [c for c in ["Recount sample ID", "GTEx donor ID", "SMTSD", "SMTSISCH", "SMRIN", "DTHHRDY"] if c in g.columns]
    sheets["IDs_GTEx_334"] = g[cols]
sheets["IDs_CGGA_104"] = pd.DataFrame({"CGGA_ID": order})
p = CGGA / "CGGA_NORMAL20_LOG2CPM_FROZEN_GENES.csv"
if p.exists():
    sheets["IDs_CGGA_controls_20"] = pd.DataFrame({"CGGA_control_ID": pd.read_csv(p, nrows=1, index_col=0).columns})

nodes = STAGE_B / "STAGE_B_ALL_ANCHORED_NETWORK_METRICS.csv"
sym = {}
if nodes.exists():
    n = pd.read_csv(nodes)
    namec = "gene_name" if "gene_name" in n.columns else "preferred_name"
    sym = dict(zip(sv(n["ensembl_id"].astype(str)), n[namec]))
feat_rows = []
for model, folder in [("M2", V4), ("M3", V4), ("R1", STAGE_D / "REVISION_STEP2B_R1"),
                      ("R2", STAGE_D / "REVISION_STEP2B_R2"), ("R2B", STAGE_D / "REVISION_STEP2B_R2B")]:
    f = folder / "STAGE_D_V4_OOF_FINAL_SELECTED_FEATURES.csv"
    if not f.exists():
        continue
    s = pd.read_csv(f)
    key = "M2" if model == "M2" else "M3"
    s = s.loc[s["model"] == key].sort_values("rank")
    for r, feat in zip(s["rank"], s["feature"].astype(str)):
        e = sv([feat.split("::")[-1]])[0]
        feat_rows.append({"model": model, "rank": r, "ensembl_id": e, "symbol": sym.get(e, "")})
fl = pd.DataFrame(feat_rows)
print(fl.groupby("model").size().to_string())
print("features without symbol:", (fl["symbol"] == "").sum() if len(fl) else 0)
sheets["Final_feature_lists"] = fl

# ---------------------------------------------------------------- write
xlsx = OUT / "Supplementary_Tables.xlsx"
try:
    with pd.ExcelWriter(xlsx) as w:
        for name, df in sheets.items():
            df.to_excel(w, sheet_name=name[:31], index=False)
    print("\nWorkbook:", xlsx)
except Exception as e:
    print("\n(Excel writer unavailable:", e, ") -- CSVs only")
for name, df in sheets.items():
    df.to_csv(OUT / f"{name}.csv", index=False)
print("CSV copies in:", OUT)
print("Sheets:", list(sheets))